# Experiment: Offline local-Qwen validation under spatiotemporal shift

This notebook benchmarks **validation-strategy selection** for the CSIRO Image2Biomass task; it does not propose a new biomass estimator. It combines frozen DINOv2 image embeddings, a leakage-safe DINOv2 + Ridge specialist, deterministic baselines, and local Qwen3-4B selectors with and without allowlisted validation tools.

The primary outcome is `abs(validation weighted R2 - deployment weighted R2)`. Scientific decisions are serialized and cryptographically frozen before pseudo-deployment labels are accessed, allowing the evaluator to reveal deployment performance and construct the oracle only after the blind phase.


## 1. Research scope and provenance

The original executed Kaggle artifact is preserved unchanged at `../original/original_kaggle_notebook.ipynb`. This repository notebook changes presentation and filesystem configuration only; seeds, model settings, prompts, splits, evaluation logic, thresholds, and integrity checks are retained.

The benchmark tests the falsifiable hypothesis that random cross-validation may be optimistic under distribution shift, and whether local tool use improves validation selection relative to fixed protocols, the same local Qwen model without tools, or a deterministic heuristic.


## 2. Setup

Import the scientific stack and resolve the repository root. Heavy model dependencies remain optional during definition so that the synthetic smoke checks can report missing components clearly.


In [ ]:
# Imports: optional heavy dependencies are checked only when their functionality is requested.
import os
import sys

os.environ["USE_TF"] = "0"
os.environ["PROTOCOL_BUFFERS_PYTHON_IMPLEMENTATION"] = "python"
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"

import gc
import gzip
import hashlib
import json
import math
import platform
import random
import re
import tempfile
import time
import warnings
import copy
from collections import Counter, defaultdict
from dataclasses import asdict, dataclass, field
from datetime import datetime, timezone
from importlib import metadata
from pathlib import Path
from typing import Any, Dict, Iterable, List, Mapping, Optional, Sequence, Tuple


# Support kernels started from either the repository root or notebooks/.
PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "src").is_dir():
    parent_candidate = PROJECT_ROOT.parent
    if (parent_candidate / "src").is_dir():
        PROJECT_ROOT = parent_candidate
    else:
        raise RuntimeError("Start Jupyter from the repository root or notebooks/ directory.")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))


import numpy as np
import pandas as pd
try:
    from scipy.stats import pearsonr, spearmanr, wilcoxon
    SCIPY_AVAILABLE = True
except ImportError:
    SCIPY_AVAILABLE = False
    wilcoxon = None
    def pearsonr(x, y):
        return float(np.corrcoef(np.asarray(x, float), np.asarray(y, float))[0, 1]), np.nan
    def spearmanr(x, y):
        return float(np.corrcoef(pd.Series(x).rank().to_numpy(), pd.Series(y).rank().to_numpy())[0, 1]), np.nan
try:
    from sklearn.base import clone
    from sklearn.feature_selection import VarianceThreshold
    from sklearn.linear_model import Ridge
    from sklearn.metrics import mean_absolute_error, mean_squared_error
    from sklearn.model_selection import GroupKFold, KFold
    from sklearn.neighbors import NearestNeighbors
    from sklearn.preprocessing import RobustScaler
    SKLEARN_AVAILABLE = True
except ImportError:
    # Minimal deterministic smoke-test fallbacks. Final mode still requires scikit-learn.
    SKLEARN_AVAILABLE = False
    clone = copy.deepcopy
    class VarianceThreshold:
        def __init__(self, threshold=0.0): self.threshold = threshold
        def fit(self, X):
            self.mask_ = np.var(np.asarray(X, float), axis=0) > self.threshold
            if not self.mask_.any(): raise ValueError("No feature meets the variance threshold.")
            return self
        def transform(self, X): return np.asarray(X, float)[:, self.mask_]
        def fit_transform(self, X): return self.fit(X).transform(X)
    class RobustScaler:
        def fit(self, X):
            X = np.asarray(X, float)
            self.center_ = np.median(X, axis=0)
            self.scale_ = np.quantile(X, 0.75, axis=0) - np.quantile(X, 0.25, axis=0)
            self.scale_[self.scale_ < 1e-12] = 1.0
            return self
        def transform(self, X): return (np.asarray(X, float) - self.center_) / self.scale_
    class Ridge:
        def __init__(self, alpha=1.0, random_state=None): self.alpha = float(alpha)
        def fit(self, X, y):
            X, y = np.asarray(X, float), np.asarray(y, float)
            self.x_mean_, self.y_mean_ = X.mean(0), y.mean(0)
            centered_X, centered_y = X - self.x_mean_, y - self.y_mean_
            gram = centered_X.T @ centered_X + self.alpha * np.eye(X.shape[1])
            self.coef_matrix_ = np.linalg.solve(gram, centered_X.T @ centered_y)
            self.coef_ = self.coef_matrix_.T if self.coef_matrix_.ndim == 2 else self.coef_matrix_
            self.intercept_ = self.y_mean_ - self.x_mean_ @ self.coef_matrix_
            return self
        def predict(self, X): return np.asarray(X, float) @ self.coef_matrix_ + self.intercept_
    def mean_absolute_error(y_true, y_pred, multioutput="uniform_average"):
        raw = np.mean(np.abs(np.asarray(y_true) - np.asarray(y_pred)), axis=0)
        return raw if multioutput == "raw_values" else float(np.mean(raw))
    def mean_squared_error(y_true, y_pred, multioutput="uniform_average"):
        raw = np.mean((np.asarray(y_true) - np.asarray(y_pred)) ** 2, axis=0)
        return raw if multioutput == "raw_values" else float(np.mean(raw))
    class KFold:
        def __init__(self, n_splits=5, shuffle=False, random_state=None):
            self.n_splits, self.shuffle, self.random_state = int(n_splits), bool(shuffle), random_state
        def split(self, X, y=None, groups=None):
            indices = np.arange(len(X))
            if self.shuffle: np.random.default_rng(self.random_state).shuffle(indices)
            for validation in np.array_split(indices, self.n_splits):
                yield np.setdiff1d(indices, validation, assume_unique=True), np.sort(validation)
    class GroupKFold:
        def __init__(self, n_splits=5): self.n_splits = int(n_splits)
        def split(self, X, y=None, groups=None):
            groups = np.asarray(groups)
            unique, counts = np.unique(groups, return_counts=True)
            assignments, fold_sizes = {}, np.zeros(self.n_splits, dtype=int)
            for group, count in sorted(zip(unique, counts), key=lambda item: (-item[1], str(item[0]))):
                fold = int(np.argmin(fold_sizes)); assignments[group] = fold; fold_sizes[fold] += count
            for fold in range(self.n_splits):
                validation = np.flatnonzero(np.array([assignments[group] == fold for group in groups]))
                training = np.flatnonzero(np.array([assignments[group] != fold for group in groups]))
                yield training, validation
    class NearestNeighbors:
        def __init__(self, n_neighbors=1): self.n_neighbors = n_neighbors
        def fit(self, X): self.X_ = np.asarray(X, float); return self
        def kneighbors(self, X, return_distance=True):
            distances = np.sqrt(((np.asarray(X, float)[:, None, :] - self.X_[None, :, :]) ** 2).sum(2))
            nearest = np.argsort(distances, axis=1)[:, :self.n_neighbors]
            nearest_distances = np.take_along_axis(distances, nearest, axis=1)
            return (nearest_distances, nearest) if return_distance else nearest

try:
    import matplotlib.pyplot as plt
    import seaborn as sns
except ImportError:
    plt = None
    sns = None

try:
    import psutil
except ImportError:
    psutil = None

try:
    import torch
    from PIL import Image
    from torch.utils.data import DataLoader, Dataset
    import torchvision.transforms as T
    import transformers
    from transformers import AutoModel, AutoModelForCausalLM, AutoTokenizer
    TRANSFORMERS_VERSION = transformers.__version__
    TORCH_STACK_AVAILABLE = True
    TORCH_IMPORT_ERROR = None
except Exception as error:
    torch = None
    Image = None
    DataLoader = None
    T = None
    AutoModel = None
    AutoModelForCausalLM = None
    AutoTokenizer = None
    transformers = None
    TRANSFORMERS_VERSION = None
    TORCH_STACK_AVAILABLE = False
    TORCH_IMPORT_ERROR = repr(error)
    class Dataset:  # Allows smoke-test-safe class definition without the image stack.
        pass

try:
    import lightgbm as lgb
except ImportError:
    lgb = None

try:
    import xgboost as xgb
except ImportError:
    xgb = None

warnings.filterwarnings("ignore", category=FutureWarning)

## 3. Configuration

All scientific parameters below are retained from the source notebook. Filesystem locations are repository-relative by default and may be overridden with the environment variables documented in the root README.


In [ ]:
# Offline experiment configuration. Review local resources before running the final cell.
RUN_MODE = "final"
MASTER_SEED = 158
KAGGLE_RUNTIME = bool(os.environ.get("KAGGLE_KERNEL_RUN_TYPE"))
DATA_PATH = Path(os.environ.get("CSIRO_DATA_PATH", str(PROJECT_ROOT / "data"))).expanduser().resolve()
ARTIFACT_DIR = Path(os.environ.get("AGENTICLS_OUTPUT_DIR", str(PROJECT_ROOT / "outputs"))).expanduser().resolve()
INPUT_SEARCH_ROOT = Path(os.environ.get("AGENTICLS_INPUT_ROOT", str(PROJECT_ROOT))).expanduser().resolve()
FEATURE_CACHE_READ_DIR = (
    Path(os.environ["FEATURE_CACHE_READ_DIR"]).expanduser().resolve()
    if os.environ.get("FEATURE_CACHE_READ_DIR") else None
)

PRIMARY_SPECIALIST_MODE = "fast"
ROBUSTNESS_SPECIALIST_MODE = "nested_stacking"
SPECIALIST_MODE = PRIMARY_SPECIALIST_MODE  # Backward-compatible alias used by core functions.
RUN_SPECIALIST_ROBUSTNESS = False
ROBUSTNESS_EPISODE_COUNT = 4

N_OUTER_FOLDS = 4
N_INNER_FOLDS = 4
MIN_TRAIN_SAMPLES = 120
MIN_DEPLOY_SAMPLES = 20
MIN_VALIDATION_SAMPLES = 10
MAX_EPISODES = 24
USE_RECONCILIATION = True
FORCE_RECOMPUTE_FEATURES = False
FORCE_RECOMPUTE_PROTOCOLS = False
FORCE_RECOMPUTE_LLM = False

# Primary local open-weight LLM. No network/API is used in final mode.
EXECUTE_LLM_CALLS = True
LOCAL_LLM_BACKEND = "transformers"
LOCAL_LLM_MODEL_NAME = "Qwen3-4B-Instruct-2507"
LOCAL_LLM_PATH = (
    Path(os.environ["LOCAL_LLM_PATH"]).expanduser().resolve()
    if os.environ.get("LOCAL_LLM_PATH") else None
)
LOCAL_LLM_ENABLE_THINKING = False
LOCAL_LLM_DO_SAMPLE = False
LOCAL_LLM_MAX_NEW_TOKENS = 384
LOCAL_LLM_ALLOW_4BIT_FALLBACK = True
LOCAL_LLM_TEMPERATURE = 0.0
LOCAL_LLM_MIN_TRANSFORMERS_VERSION = "4.51.0"
MAX_LLM_RETRIES = 2
AGENT_REPLICATES = 1
ALLOW_VOLUNTARY_ABSTAIN = False
MAX_TOOL_CALLS = 6
MIN_LLM_SUCCESS_RATE = 0.90
RUN_AGENT_STABILITY_ANALYSIS = False

BOOTSTRAP_RESAMPLES = 10_000
ORACLE_TOLERANCE = 0.02
CATASTROPHIC_THRESHOLD = 0.15
RUN_ABLATIONS = False
RUN_COMPETITION_SUBMISSION = False

# Frozen image representation. Supply a local model directory through DINO_MODEL_PATH.
DINO_MODEL_ID = "facebook/dinov2-giant"
DINO_MODEL_PATH = (
    Path(os.environ["DINO_MODEL_PATH"]).expanduser().resolve()
    if os.environ.get("DINO_MODEL_PATH") else None
)
DINO_LOCAL_PATHS = []
IMAGE_SIZE = 518
FEATURE_BATCH_SIZE = 2
FEATURE_NUM_WORKERS = 2
USE_FP16 = True
TTA_COUNT = 3
POOLING_STRATEGY = "cls+mean_patch+max_patch"
FEATURE_CACHE_VERSION = "agenticls-dinov2-v2"
FEATURE_CACHE_FILENAME = "dinov2_features.npz"
N_TEMPORAL_BLOCKS = 4
N_ID_CONTROLS = 4
ID_DEPLOY_FRACTION = 0.20
USE_GPU_TREES = False
NOTEBOOK_VERSION = "agenticls-3.0.0"
CODE_CONFIG_VERSION = "offline-blind-phase-v3"
AGENT_PROMPT_VERSION = "v2.1-local-qwen-semantic-repair"

# Optional legacy aliases. The final validator requires LOCAL_LLM_BACKEND="transformers".
LLM_BACKEND = LOCAL_LLM_BACKEND
LLM_MODEL = LOCAL_LLM_MODEL_NAME
LLM_TEMPERATURE = LOCAL_LLM_TEMPERATURE

TARGET_COLS = ["Dry_Green_g", "Dry_Dead_g", "Dry_Clover_g", "GDM_g", "Dry_Total_g"]
TARGET_WEIGHTS = [0.1, 0.1, 0.1, 0.2, 0.5]
PROTOCOLS = [
    "random_kfold", "date_group_kfold", "state_group_kfold",
    "date_state_group_kfold", "temporal_block_holdout",
]
REASON_CODES = {
    "future_temporal_shift", "past_temporal_shift", "unseen_state",
    "state_distribution_shift", "date_group_concentration", "embedding_shift_high",
    "embedding_shift_low", "small_sample_warning", "random_cv_optimism_risk",
    "grouped_cv_supported", "temporal_holdout_supported", "multiple_protocols_similar",
    "insufficient_evidence",
}


class CFG:
    """Compatibility facade retained from the original Kaggle biomass notebook."""
    DATA_PATH = str(DATA_PATH)
    train_path = str(Path(DATA_PATH) / "train.csv")
    test_path = str(Path(DATA_PATH) / "test.csv")
    dinov2_paths = [str(DINO_MODEL_PATH)] if DINO_MODEL_PATH else list(DINO_LOCAL_PATHS)
    img_size = IMAGE_SIZE
    batch_size = FEATURE_BATCH_SIZE
    use_fp16 = USE_FP16
    seed = MASTER_SEED
    n_fold = N_OUTER_FOLDS
    target_cols = TARGET_COLS
    target_weights = TARGET_WEIGHTS
    device = (torch.device("cuda" if torch.cuda.is_available() else "cpu")
              if TORCH_STACK_AVAILABLE else "cpu")

## 4. Experimental design and phase boundary

Pseudo-deployment episodes are created from the labeled CSIRO training set, but Phase I receives training labels only for candidate validation. Deployment observations contain metadata and frozen DINO features, never deployment targets, evaluator scores, oracle choices, or hindsight gaps.

All fixed, heuristic, LLM-only, and tool-agent decisions are written, audited, and hashed before Phase II accesses deployment biomass labels. DINOv2 and Qwen never coexist on the GPU; both use local files only and are explicitly unloaded with CUDA diagnostics.


In [ ]:
# Reproducibility, metrics, reconciliation, hashing, serialization, and runtime utilities.
def seed_everything(seed: int) -> None:
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    if TORCH_STACK_AVAILABLE:
        torch.manual_seed(seed)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
        try:
            torch.use_deterministic_algorithms(True, warn_only=True)
        except (AttributeError, RuntimeError):
            pass


def stable_seed(*parts: Any, base: int = MASTER_SEED) -> int:
    payload = "|".join(map(str, (base, *parts))).encode("utf-8")
    return int(hashlib.sha256(payload).hexdigest()[:8], 16) % (2**31 - 1)


def canonical_json(value: Any) -> str:
    def default(obj):
        if isinstance(obj, (np.integer,)): return int(obj)
        if isinstance(obj, (np.floating,)): return float(obj)
        if isinstance(obj, (np.ndarray,)): return obj.tolist()
        if isinstance(obj, (pd.Timestamp, datetime)): return obj.isoformat()
        if isinstance(obj, Path): return str(obj)
        if isinstance(obj, set): return sorted(obj)
        if pd.isna(obj): return None
        raise TypeError(f"Cannot serialize {type(obj).__name__}")
    return json.dumps(value, sort_keys=True, separators=(",", ":"), default=default, allow_nan=False)


def object_hash(value: Any) -> str:
    return hashlib.sha256(canonical_json(value).encode("utf-8")).hexdigest()


def file_sha256(path: Path, chunk_size: int = 2**20) -> str:
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for chunk in iter(lambda: handle.read(chunk_size), b""):
            digest.update(chunk)
    return digest.hexdigest()


def array_sha256(array: np.ndarray) -> str:
    contiguous = np.ascontiguousarray(array)
    digest = hashlib.sha256()
    digest.update(str(contiguous.dtype).encode("utf-8"))
    digest.update(canonical_json(list(contiguous.shape)).encode("utf-8"))
    digest.update(contiguous.view(np.uint8).tobytes())
    return digest.hexdigest()


def atomic_json_dump(value: Any, path: Path) -> None:
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + ".tmp")
    temporary.write_text(json.dumps(value, indent=2, default=str), encoding="utf-8")
    temporary.replace(path)


def append_jsonl(path: Path, records: Iterable[Mapping[str, Any]]) -> None:
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("a", encoding="utf-8") as handle:
        for record in records:
            handle.write(canonical_json(dict(record)) + "\n")


from src.metrics import (
    calculate_regression_metrics,
    post_process_biomass,
    reconcile_array,
    weighted_r2_metric,
)


def synchronize_device() -> None:
    if TORCH_STACK_AVAILABLE and torch.cuda.is_available():
        torch.cuda.synchronize()


def timed_call(function, *args, **kwargs):
    synchronize_device()
    started = time.perf_counter()
    result = function(*args, **kwargs)
    synchronize_device()
    return result, float(time.perf_counter() - started)


class RuntimeTracker:
    def __init__(self): self.records = []
    def add(self, stage, seconds, n_samples=None, category=None, **extra):
        seconds = float(seconds)
        row = {"Category": category, "Stage": stage, "Time_s": seconds,
               "Time_min": seconds / 60.0, "N_samples": n_samples,
               "Time_per_image_ms": (seconds / n_samples * 1000 if n_samples else np.nan),
               "Throughput_images_s": (n_samples / seconds if n_samples and seconds > 0 else np.nan)}
        row.update(extra)
        self.records.append(row)
    def get_seconds(self, stage):
        matches = [row["Time_s"] for row in self.records if row["Stage"] == stage]
        if not matches: raise KeyError(stage)
        return matches[-1]
    def to_dataframe(self): return pd.DataFrame(self.records)


def get_package_version(name: str):
    try: return metadata.version(name)
    except metadata.PackageNotFoundError: return None


def get_cpu_model() -> str:
    try:
        cpuinfo = Path("/proc/cpuinfo")
        if cpuinfo.exists():
            for line in cpuinfo.read_text(encoding="utf-8").splitlines():
                if line.lower().startswith("model name"):
                    return line.split(":", 1)[1].strip()
    except OSError:
        pass
    return platform.processor() or platform.uname().processor or "Unknown"


seed_everything(MASTER_SEED)

# Local path discovery, offline safety, and GPU lifecycle diagnostics.
def discover_kaggle_input(keyword_candidates, required_files=(), root=INPUT_SEARCH_ROOT):
    """Return one unambiguous local input directory; never choose among multiple matches."""
    root = Path(root)
    if not root.exists():
        return None
    keywords = [str(keyword).casefold() for keyword in keyword_candidates]
    candidates = []
    for directory in [root, *[path for path in root.rglob("*") if path.is_dir()]]:
        relative = str(directory.relative_to(root)).casefold() if directory != root else ""
        if not any(keyword in relative for keyword in keywords):
            continue
        if all((directory / filename).exists() for filename in required_files):
            candidates.append(directory)
    # Prefer the shallowest matching model root, but ambiguity at equal depth is explicit.
    if not candidates:
        return None
    minimum_depth = min(len(path.relative_to(root).parts) for path in candidates)
    candidates = sorted({path for path in candidates if len(path.relative_to(root).parts) == minimum_depth})
    if len(candidates) > 1:
        print("Ambiguous local inputs; set the path explicitly:")
        for path in candidates:
            print("  ", path)
        return None
    return candidates[0]


def version_tuple(version_string):
    match = re.match(r"^(\d+)\.(\d+)\.(\d+)", str(version_string or ""))
    if not match:
        raise RuntimeError(f"Cannot parse Transformers version: {version_string!r}")
    return tuple(map(int, match.groups()))


def require_transformers_version(current=None, minimum=LOCAL_LLM_MIN_TRANSFORMERS_VERSION):
    current = current or TRANSFORMERS_VERSION
    if current is None or version_tuple(current) < version_tuple(minimum):
        raise RuntimeError(
            f"Qwen3 requires transformers>={minimum}; found {current!r}. Configure the required "
            "version in the active environment before final execution. "
            "This notebook will not install packages from the internet in FINAL mode."
        )
    return current


def resolve_local_model_path(explicit_path, keywords, required_files, label):
    if explicit_path is not None:
        path = Path(explicit_path)
        if not path.exists():
            raise FileNotFoundError(f"{label} path does not exist: {path}")
        missing = [name for name in required_files if not (path / name).exists()]
        if missing:
            raise FileNotFoundError(f"{label} directory {path} is missing required files: {missing}")
        return path
    discovered = discover_kaggle_input(keywords, required_files)
    if discovered is None:
        raise FileNotFoundError(
            f"No unambiguous local {label} input was found under {INPUT_SEARCH_ROOT}. Set its path explicitly."
        )
    return discovered


def resolve_data_path(explicit_path=DATA_PATH):
    path = Path(explicit_path)
    if (path / "train.csv").exists(): return path
    discovered = discover_kaggle_input(["csiro", "biomass"], ["train.csv"])
    if discovered is None:
        raise FileNotFoundError(
            f"CSIRO competition data were not found at {path}. Attach the competition input and set DATA_PATH explicitly."
        )
    print(f"Discovered CSIRO competition data: {discovered}")
    return discovered


def cuda_memory_snapshot(label):
    if not TORCH_STACK_AVAILABLE or not torch.cuda.is_available():
        return {"label": label, "cuda_available": False, "allocated_bytes": 0, "reserved_bytes": 0,
                "peak_allocated_bytes": 0}
    torch.cuda.synchronize()
    return {"label": label, "cuda_available": True,
            "allocated_bytes": int(torch.cuda.memory_allocated()),
            "reserved_bytes": int(torch.cuda.memory_reserved()),
            "peak_allocated_bytes": int(torch.cuda.max_memory_allocated()),
            "device": torch.cuda.get_device_name(0)}


def release_cuda_memory(label, before=None):
    gc.collect()
    if TORCH_STACK_AVAILABLE and torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.synchronize()
    after = cuda_memory_snapshot(label)
    before = before or after
    released = max(0, int(before.get("allocated_bytes", 0)) - int(after.get("allocated_bytes", 0)))
    after["released_allocated_bytes"] = released
    after["substantial_release"] = (before.get("allocated_bytes", 0) == 0 or
                                     after.get("allocated_bytes", 0) <= 0.25 * before.get("allocated_bytes", 0))
    print(f"GPU lifecycle {label}: allocated={after['allocated_bytes']} reserved={after['reserved_bytes']}")
    return after


def assert_final_offline_safety(local_llm_path=None, dino_path=None):
    if LOCAL_LLM_BACKEND != "transformers":
        raise AssertionError("FINAL mode requires LOCAL_LLM_BACKEND='transformers'; external APIs are disabled.")
    for label, path in (("Qwen", local_llm_path), ("DINOv2", dino_path)):
        if path is not None and not Path(path).exists():
            raise AssertionError(f"{label} must resolve to a local filesystem resource in FINAL mode: {path}")
    return True

## 5. Data loading

The loader prints the raw CSV schema before resolving aliases. Date and location columns are discovered from the data: dates are mandatory for temporal shift, while spatial protocols are explicitly disabled when location metadata are absent. Metadata are reduced to one consistency-checked row per image, targets are pivoted separately, and the two are merged one-to-one.


In [ ]:
DATE_ALIASES = ["Sampling_Date", "sampling_date", "Acquisition_Date", "acquisition_date", "Date", "date"]
STATE_ALIASES = ["State", "state", "Region", "region", "Location", "location", "Site", "site"]
SPECIES_ALIASES = ["Species", "species", "Cultivar", "cultivar", "Pasture_Species", "pasture_species"]
IMAGE_ID_ALIASES = ["image_id", "Image_ID", "imageid"]
IMAGE_PATH_ALIASES = ["image_path", "Image_Path", "path", "filepath"]


def _resolve_alias(columns: Sequence[str], aliases: Sequence[str]) -> Optional[str]:
    exact = {str(column): str(column) for column in columns}
    lower = {str(column).casefold(): str(column) for column in columns}
    for alias in aliases:
        if alias in exact: return exact[alias]
    for alias in aliases:
        if alias.casefold() in lower: return lower[alias.casefold()]
    return None


def discover_metadata_columns(raw: pd.DataFrame, require_date: bool = True) -> Dict[str, Any]:
    columns = list(map(str, raw.columns))
    print("Raw train.csv columns:")
    for index, column in enumerate(columns, 1):
        print(f"  {index:02d}. {column}")
    date_column = _resolve_alias(columns, DATE_ALIASES)
    state_column = _resolve_alias(columns, STATE_ALIASES)
    species_columns = [column for column in columns
                       if column.casefold() in {alias.casefold() for alias in SPECIES_ALIASES}
                       or any(token in column.casefold() for token in ("species", "cultivar"))]
    protected = {"sample_id", "target_name", "target", "target_type", "image_id", "image_path"}
    acquisition_columns = [column for column in columns if column not in protected and column not in species_columns]
    if require_date and date_column is None:
        raise ValueError("No acquisition/sampling date field was found. Available columns: " + ", ".join(columns))
    if state_column is None:
        warnings.warn("No state/region/location field was found. Spatial episodes and protocols will be disabled explicitly.")
    discovery = {
        "date_column": date_column,
        "state_column": state_column,
        "species_columns": species_columns,
        "other_acquisition_columns": acquisition_columns,
    }
    print("Metadata discovery:", json.dumps(discovery, indent=2))
    return discovery


def _ensure_image_id(frame: pd.DataFrame) -> pd.DataFrame:
    frame = frame.copy()
    image_id_column = _resolve_alias(frame.columns, IMAGE_ID_ALIASES)
    if image_id_column and image_id_column != "image_id":
        frame = frame.rename(columns={image_id_column: "image_id"})
    if "image_id" not in frame.columns:
        if "sample_id" not in frame.columns:
            raise ValueError("Neither image_id nor sample_id is present.")
        split = frame["sample_id"].astype(str).str.rsplit("__", n=1, expand=True)
        if split.shape[1] != 2:
            raise ValueError("sample_id must follow image_id__target format.")
        frame["image_id"] = split[0]
        frame["target_type"] = split[1]
    path_column = _resolve_alias(frame.columns, IMAGE_PATH_ALIASES)
    if path_column and path_column != "image_path":
        frame = frame.rename(columns={path_column: "image_path"})
    if "image_path" not in frame.columns:
        raise ValueError("No image path column was found.")
    return frame


def _consistent_image_metadata(raw: pd.DataFrame, metadata_columns: Sequence[str]) -> pd.DataFrame:
    for column in metadata_columns:
        inconsistent = raw.groupby("image_id", dropna=False)[column].nunique(dropna=False)
        if (inconsistent > 1).any():
            examples = inconsistent[inconsistent > 1].index.astype(str).tolist()[:5]
            raise ValueError(f"Metadata column {column!r} varies within image IDs; examples: {examples}")
    return raw[["image_id", *metadata_columns]].drop_duplicates("image_id", keep="first")


def _prepare_wide_training(raw_train: pd.DataFrame, discovery: Mapping[str, Any]) -> pd.DataFrame:
    train = _ensure_image_id(raw_train)
    target_name_column = "target_name" if "target_name" in train.columns else "target_type"
    if target_name_column not in train.columns or "target" not in train.columns:
        raise ValueError("Training CSV must contain target_name/target_type and target columns.")
    date_column = discovery["date_column"]
    state_column = discovery["state_column"]
    excluded = {"sample_id", "target_name", "target_type", "target"}
    metadata_columns = [column for column in train.columns if column not in excluded and column != "image_id"]
    metadata = _consistent_image_metadata(train, metadata_columns)
    targets = train.pivot(index="image_id", columns=target_name_column, values="target").reset_index()
    targets.columns.name = None
    missing = [column for column in TARGET_COLS if column not in targets.columns]
    if missing: raise ValueError(f"Missing required biomass targets after pivot: {missing}")
    wide = metadata.merge(targets[["image_id", *TARGET_COLS]], on="image_id", how="inner", validate="one_to_one")
    if not wide["image_id"].is_unique: raise AssertionError("image_id is not unique after pivoting.")
    parsed = pd.to_datetime(wide[date_column], errors="coerce", utc=True)
    failure_rate = float(parsed.isna().mean())
    if failure_rate > 0.10:
        raise ValueError(f"Catastrophic date parsing failure for {date_column!r}: {failure_rate:.1%} missing/unparseable.")
    wide["sampling_date_dt"] = parsed.dt.tz_convert(None)
    if state_column is not None:
        wide["state_norm"] = wide[state_column].astype("string").str.strip().str.casefold()
        wide.loc[wide[state_column].isna(), "state_norm"] = pd.NA
        if wide["state_norm"].notna().sum() == 0:
            warnings.warn("The discovered location column is entirely missing; spatial experiments are disabled.")
            discovery["state_column"] = None
            wide = wide.drop(columns=["state_norm"])
        else:
            assert state_column in wide.columns, "State metadata was lost during pivoting."
    wide.attrs["metadata_discovery"] = dict(discovery)
    wide.attrs["date_parse_failure_rate"] = failure_rate
    return wide.reset_index(drop=True)


def load_data(data_path: Path = DATA_PATH, load_test: bool = False):
    data_path = Path(data_path)
    train_path = data_path / "train.csv"
    if not train_path.exists():
        raise FileNotFoundError(f"Required training file not found: {train_path}")
    raw_train = pd.read_csv(train_path)
    discovery = discover_metadata_columns(raw_train, require_date=True)
    train_wide = _prepare_wide_training(raw_train, discovery)
    test_wide = None
    if load_test:
        test_path = data_path / "test.csv"
        if not test_path.exists(): raise FileNotFoundError(f"Competition test file not found: {test_path}")
        test_raw = _ensure_image_id(pd.read_csv(test_path))
        test_wide = test_raw.drop_duplicates("image_id", keep="first").reset_index(drop=True)
        if not test_wide["image_id"].is_unique: raise AssertionError("Test image IDs are not unique.")
    print(f"Prepared {len(train_wide)} unique labeled images; spatial metadata available={discovery['state_column'] is not None}.")
    return train_wide, test_wide, discovery

## 6. Frozen DINOv2 representation

The three deterministic views and CLS/mean-patch/max-patch pooling are retained. DINOv2 is loaded from local files, frozen, and released immediately after feature extraction. A compatible cache may be supplied through `FEATURE_CACHE_READ_DIR`; it is validated and ID-realigned but never overwritten. New caches are written beneath `outputs/cache/` by default.


In [ ]:
class BiomassDataset(Dataset):
    def __init__(self, df: pd.DataFrame, transform=None, data_path: Path = DATA_PATH):
        if not TORCH_STACK_AVAILABLE: raise ImportError("PyTorch, torchvision, Pillow, and transformers are required.")
        self.image_paths = df["image_path"].astype(str).tolist()
        self.transform = transform
        self.data_path = Path(data_path)
    def __len__(self): return len(self.image_paths)
    def __getitem__(self, idx):
        path = self.data_path / self.image_paths[idx]
        if not path.exists(): raise FileNotFoundError(f"Image not found: {path}")
        image = Image.open(path).convert("RGB")
        return self.transform(image) if self.transform else image


def _dinov2_views():
    mean, std = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]
    common = [T.Resize(IMAGE_SIZE), T.CenterCrop(IMAGE_SIZE)]
    return [
        T.Compose([*common, T.ToTensor(), T.Normalize(mean, std)]),
        T.Compose([*common, T.RandomHorizontalFlip(p=1.0), T.ToTensor(), T.Normalize(mean, std)]),
        T.Compose([*common, T.RandomVerticalFlip(p=1.0), T.ToTensor(), T.Normalize(mean, std)]),
    ]


def extract_features(df: pd.DataFrame, model, data_path: Path = DATA_PATH) -> pd.DataFrame:
    """Frozen DINOv2 features with CLS + mean patch + max patch pooling and three TTA views."""
    if not df["image_id"].is_unique: raise ValueError("Feature extraction requires unique image IDs.")
    all_views = []
    for view_index, transform in enumerate(_dinov2_views()):
        dataset = BiomassDataset(df, transform=transform, data_path=data_path)
        loader = DataLoader(dataset, batch_size=FEATURE_BATCH_SIZE, shuffle=False,
                            num_workers=FEATURE_NUM_WORKERS, pin_memory=torch.cuda.is_available())
        batches = []
        with torch.inference_mode():
            for images in loader:
                images = images.to(CFG.device, non_blocking=True)
                autocast_enabled = bool(USE_FP16 and torch.cuda.is_available())
                with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=autocast_enabled):
                    outputs = model(images)
                    if hasattr(outputs, "last_hidden_state"):
                        tokens = outputs.last_hidden_state
                        cls_token, patches = tokens[:, 0], tokens[:, 1:]
                    elif isinstance(outputs, dict) and "x_norm_clstoken" in outputs:
                        cls_token, patches = outputs["x_norm_clstoken"], outputs["x_norm_patchtokens"]
                    else:
                        tokens = model.forward_features(images) if hasattr(model, "forward_features") else outputs
                        if isinstance(tokens, dict):
                            cls_token, patches = tokens["x_norm_clstoken"], tokens["x_norm_patchtokens"]
                        elif getattr(tokens, "ndim", 0) == 3:
                            cls_token, patches = tokens[:, 0], tokens[:, 1:]
                        else:
                            raise TypeError("Unsupported DINOv2 output; token embeddings are required for triple pooling.")
                    embedding = torch.cat([cls_token, patches.mean(1), patches.max(1).values], dim=1)
                batches.append(embedding.float().cpu().numpy())
        if not batches: raise ValueError("No images were available for feature extraction.")
        all_views.append(np.concatenate(batches, axis=0))
        print(f"DINOv2 view {view_index + 1}/{TTA_COUNT} complete.")
    features = np.mean(np.stack(all_views, axis=0), axis=0).astype(np.float32)
    return pd.DataFrame(features, index=df["image_id"].astype(str),
                        columns=[f"feat_{i}" for i in range(features.shape[1])])

# Local-only DINO loading and optional read-only feature cache.
def resolved_dino_model_path():
    explicit = DINO_MODEL_PATH
    if explicit is None:
        existing = [Path(path) for path in DINO_LOCAL_PATHS if Path(path).exists()]
        if len(existing) == 1:
            return existing[0]
        if len(existing) > 1:
            print("Multiple DINOv2 paths exist; set DINO_MODEL_PATH explicitly:")
            for path in existing: print("  ", path)
            raise RuntimeError("Ambiguous local DINOv2 input.")
    return resolve_local_model_path(explicit, ["dinov2", "dino-v2"], ["config.json"], "DINOv2 model")


def feature_configuration():
    return {
        "cache_version": FEATURE_CACHE_VERSION, "model_id": DINO_MODEL_ID,
        "model_source_identifier": DINO_MODEL_ID, "image_resolution": IMAGE_SIZE,
        "tta_count": TTA_COUNT, "pooling_strategy": POOLING_STRATEGY,
        "preprocessing": {"resize": IMAGE_SIZE, "center_crop": IMAGE_SIZE,
                          "mean": [0.485, 0.456, 0.406], "std": [0.229, 0.224, 0.225],
                          "views": ["identity", "horizontal_flip", "vertical_flip"]},
        "code_config_version": CODE_CONFIG_VERSION,
    }


def get_dinov2_model():
    if not TORCH_STACK_AVAILABLE:
        raise ImportError(f"PyTorch/Transformers image stack unavailable: {TORCH_IMPORT_ERROR}")
    path = resolved_dino_model_path()
    print(f"Loading frozen DINOv2 locally from: {path}")
    model = AutoModel.from_pretrained(str(path), local_files_only=True, trust_remote_code=True)
    model.agenticls_model_source = str(path)
    model = model.to(CFG.device).eval()
    for parameter in model.parameters(): parameter.requires_grad_(False)
    if any(parameter.requires_grad for parameter in model.parameters()):
        raise AssertionError("DINOv2 must remain frozen.")
    return model


def _load_validated_feature_cache(cache_path, manifest_path, requested_ids, expected):
    if not cache_path.exists() or not manifest_path.exists():
        return None
    manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
    with np.load(cache_path, allow_pickle=False) as payload:
        cached_ids = payload["image_ids"].astype(str)
        features = payload["features"].astype(np.float32)
    if manifest.get("configuration") != expected: raise ValueError("feature configuration changed")
    if len(cached_ids) != len(set(cached_ids)) or features.shape[0] != len(cached_ids):
        raise ValueError("feature cache IDs/rows are invalid")
    if set(cached_ids) != set(requested_ids): raise ValueError("cached/requested image-ID sets differ")
    if manifest.get("image_ids_hash") != object_hash(sorted(requested_ids)):
        raise ValueError("feature image-ID hash mismatch")
    lookup = {image_id: index for index, image_id in enumerate(cached_ids)}
    aligned = features[[lookup[image_id] for image_id in requested_ids]]
    if manifest.get("feature_dim") != aligned.shape[1]: raise ValueError("feature dimension mismatch")
    manifest = {**manifest, "loaded_from": str(cache_path), "read_only_source": bool(FEATURE_CACHE_READ_DIR is not None and Path(cache_path).parent.resolve() == Path(FEATURE_CACHE_READ_DIR).resolve())}
    return aligned, manifest, True


def load_or_extract_dinov2_features(df, data_path=DATA_PATH, cache_name="dinov2_features"):
    requested_ids = df["image_id"].astype(str).tolist()
    if len(requested_ids) != len(set(requested_ids)): raise ValueError("Duplicate image IDs cannot be cached.")
    expected = feature_configuration()
    writable_dir = ARTIFACT_DIR / "cache"; writable_dir.mkdir(parents=True, exist_ok=True)
    filename = FEATURE_CACHE_FILENAME if cache_name == "dinov2_features" else f"{cache_name}.npz"
    manifest_name = "feature_cache_manifest.json" if cache_name == "dinov2_features" else f"{cache_name}_manifest.json"
    candidate_directories = []
    if FEATURE_CACHE_READ_DIR is not None:
        candidate_directories.append(Path(FEATURE_CACHE_READ_DIR))
    candidate_directories.append(writable_dir)
    if not FORCE_RECOMPUTE_FEATURES:
        for directory in candidate_directories:
            try:
                loaded = _load_validated_feature_cache(directory / filename, directory / manifest_name,
                                                       requested_ids, expected)
                if loaded is not None:
                    print(f"Loaded compatible ID-aligned DINO cache: {directory / filename}")
                    return loaded
            except Exception as error:
                print(f"Feature cache rejected at {directory}: {error}")
    before_load = cuda_memory_snapshot("before_dinov2_load")
    model = get_dinov2_model()
    after_load = cuda_memory_snapshot("after_dinov2_load")
    source = getattr(model, "agenticls_model_source", str(resolved_dino_model_path()))
    feature_frame, elapsed = timed_call(extract_features, df, model, data_path)
    after_extract = cuda_memory_snapshot("after_dinov2_extract")
    del model
    release = release_cuda_memory("after_dinov2_unload", after_extract)
    if after_extract.get("allocated_bytes", 0) > 0 and not release["substantial_release"]:
        raise RuntimeError("DINOv2 GPU memory was not substantially released; refusing to load Qwen later.")
    features = feature_frame.to_numpy(np.float32)
    cache_path, manifest_path = writable_dir / filename, writable_dir / manifest_name
    temporary = cache_path.with_suffix(".tmp.npz")
    np.savez_compressed(temporary, image_ids=np.asarray(requested_ids, dtype=str), features=features,
                        configuration_json=np.asarray(canonical_json(expected)))
    temporary.replace(cache_path)
    manifest = {
        "configuration": expected, "image_count": len(requested_ids), "feature_dim": int(features.shape[1]),
        "image_ids_hash": object_hash(sorted(requested_ids)), "cache_sha256": file_sha256(cache_path),
        "selected_model_source": source, "extraction_seconds": elapsed,
        "gpu_lifecycle": {"before_load": before_load, "after_load": after_load,
                          "after_extract": after_extract, "after_unload": release},
        "created_at": datetime.now(timezone.utc).isoformat(), "read_only_source": False,
    }
    atomic_json_dump(manifest, manifest_path)
    return features, manifest, False

## 7. Leakage-safe specialists

The primary experiment uses the computationally tractable DINOv2 + Ridge specialist (`PRIMARY_SPECIALIST_MODE="fast"`) for every episode and protocol. Variance filtering and robust scaling are fit inside each training partition; targets use `log1p`, predictions are nonnegative, and agronomic reconciliation is applied consistently.

The retained LightGBM/XGBoost/Ridge/meta-Ridge nested specialist is an optional robustness analysis on four episodes selected by evaluator category before performance is observed. Its inner out-of-fold preprocessors are fold-local.


In [ ]:
class LeakageSafePreprocessor:
    """Variance filtering and robust scaling with an explicit sample-ID audit."""
    def __init__(self):
        self.selector = VarianceThreshold(threshold=0.0)
        self.scaler = RobustScaler()
        self.fit_ids = None
        self.n_input_features = None
        self.n_output_features = None
    def fit(self, X, sample_ids, forbidden_ids=()):
        X = np.asarray(X, dtype=np.float64)
        sample_ids = list(map(str, sample_ids))
        forbidden = set(map(str, forbidden_ids))
        overlap = forbidden.intersection(sample_ids)
        if overlap: raise AssertionError(f"Preprocessing leakage: {len(overlap)} held-out IDs were supplied to fit().")
        if X.shape[0] != len(sample_ids): raise ValueError("Preprocessing IDs do not align with X rows.")
        self.n_input_features = int(X.shape[1])
        selected = self.selector.fit_transform(X)
        if selected.shape[1] == 0: raise ValueError("VarianceThreshold removed every embedding feature.")
        self.scaler.fit(selected)
        self.fit_ids = tuple(sample_ids)
        self.n_output_features = int(selected.shape[1])
        return self
    def transform(self, X):
        if self.fit_ids is None: raise RuntimeError("Preprocessor must be fit before transform.")
        return self.scaler.transform(self.selector.transform(np.asarray(X, dtype=np.float64)))
    def fit_transform(self, X, sample_ids, forbidden_ids=()):
        return self.fit(X, sample_ids, forbidden_ids).transform(X)


class StackingTrainer:
    """Refactored level-1 trainer: all split pairs are supplied by its caller."""
    def __init__(self, seed=MASTER_SEED, lgbm_params=None, xgb_params=None):
        self.seed = int(seed)
        self.lgbm_params = lgbm_params
        self.xgb_params = xgb_params
        self.runtime = defaultdict(float)
    def get_base_models(self):
        if lgb is None or xgb is None:
            missing = [name for name, module in (("lightgbm", lgb), ("xgboost", xgb)) if module is None]
            raise ImportError("nested_stacking requires: " + ", ".join(missing))
        lgb_params = self.lgbm_params or {
            "n_estimators": 300, "learning_rate": 0.05, "num_leaves": 31,
            "random_state": self.seed, "n_jobs": -1, "verbose": -1,
        }
        if USE_GPU_TREES: lgb_params = {**lgb_params, "device": "gpu"}
        xgb_params = self.xgb_params or {
            "n_estimators": 300, "learning_rate": 0.05, "max_depth": 6,
            "subsample": 0.8, "colsample_bytree": 0.8,
            "objective": "reg:squarederror", "n_jobs": -1,
            "random_state": self.seed, "tree_method": "hist",
            "device": "cuda" if USE_GPU_TREES else "cpu",
        }
        # Each target is modeled separately to preserve the original design.
        return {
            "lgbm": [lgb.LGBMRegressor(**lgb_params) for _ in TARGET_COLS],
            "xgb": [xgb.XGBRegressor(**xgb_params) for _ in TARGET_COLS],
            "ridge": [Ridge(alpha=1.0) for _ in TARGET_COLS],
        }
    @staticmethod
    def _fit_predict_models(models, X_train, y_train, X_eval):
        predictions = np.zeros((len(X_eval), y_train.shape[1]), dtype=float)
        fitted = []
        for target_index, model in enumerate(models):
            fitted_model = clone(model).fit(X_train, y_train[:, target_index])
            predictions[:, target_index] = fitted_model.predict(X_eval)
            fitted.append(fitted_model)
        return fitted, predictions
    def generate_inner_oof(self, X, y_log, sample_ids, split_pairs, forbidden_ids=()):
        factories = self.get_base_models()
        oof = {name: np.full_like(y_log, np.nan, dtype=float) for name in factories}
        audits = []
        for fold_index, (fit_idx, val_idx) in enumerate(split_pairs):
            fold_forbidden = set(map(str, forbidden_ids)).union(np.asarray(sample_ids)[val_idx].astype(str))
            preprocessor = LeakageSafePreprocessor()
            X_fit = preprocessor.fit_transform(X[fit_idx], np.asarray(sample_ids)[fit_idx], fold_forbidden)
            X_val = preprocessor.transform(X[val_idx])
            audits.append({
                "scope": "inner", "fold": fold_index, "fit_n": len(fit_idx), "heldout_n": len(val_idx),
                "fit_ids_hash": object_hash(sorted(preprocessor.fit_ids)), "forbidden_overlap": 0,
            })
            for name, model_list in factories.items():
                started = time.perf_counter()
                _, prediction = self._fit_predict_models(model_list, X_fit, y_log[fit_idx], X_val)
                self.runtime[f"{name}_inner_fit_predict_s"] += time.perf_counter() - started
                oof[name][val_idx] = prediction
        if any(np.isnan(values).any() for values in oof.values()):
            raise AssertionError("Inner stacking OOF predictions do not cover every training sample.")
        return oof, audits


class SpecialistBiomassRegressor:
    def __init__(self, mode=SPECIALIST_MODE, seed=MASTER_SEED,
                 n_inner_folds=N_INNER_FOLDS, use_reconciliation=USE_RECONCILIATION):
        if mode not in {"fast", "nested_stacking"}: raise ValueError(f"Unknown specialist mode: {mode}")
        self.mode = mode
        self.seed = int(seed)
        self.n_inner_folds = int(n_inner_folds)
        self.use_reconciliation = bool(use_reconciliation)
        self.preprocessing_audit = []
        self.raw_predictions_ = None
        self.reconciled_predictions_ = None
        self.runtime = defaultdict(float)
        self.fitted = False
    def fit(self, X_train, y_train, sample_ids=None, forbidden_ids=()):
        X_train = np.asarray(X_train, dtype=np.float64)
        y_train = np.asarray(y_train, dtype=np.float64)
        if X_train.shape[0] != y_train.shape[0] or y_train.shape[1] != len(TARGET_COLS):
            raise ValueError("Specialist training arrays are misaligned.")
        if np.any(y_train < 0): raise ValueError("log1p target transform requires non-negative biomass targets.")
        sample_ids = np.asarray(sample_ids if sample_ids is not None else [f"row_{i}" for i in range(len(X_train))], dtype=str)
        forbidden = set(map(str, forbidden_ids))
        if forbidden.intersection(sample_ids): raise AssertionError("Held-out sample IDs reached specialist fit().")
        y_log = np.log1p(y_train)
        self.full_preprocessor = LeakageSafePreprocessor()
        X_full = self.full_preprocessor.fit_transform(X_train, sample_ids, forbidden)
        self.preprocessing_audit.append({
            "scope": "full_supplied_training_partition", "fit_n": len(X_train),
            "fit_ids_hash": object_hash(sorted(self.full_preprocessor.fit_ids)), "forbidden_overlap": 0,
        })
        if self.mode == "fast":
            self.fast_model = Ridge(alpha=10.0).fit(X_full, y_log)
        else:
            n_splits = min(self.n_inner_folds, len(X_train))
            if n_splits < 2: raise ValueError("At least two training samples are required for inner stacking.")
            inner = KFold(n_splits=n_splits, shuffle=True, random_state=self.seed)
            split_pairs = list(inner.split(X_train))
            self.trainer = StackingTrainer(seed=self.seed)
            oof, inner_audit = self.trainer.generate_inner_oof(
                X_train, y_log, sample_ids, split_pairs, forbidden_ids=forbidden)
            self.preprocessing_audit.extend(inner_audit)
            names = ["lgbm", "xgb", "ridge"]
            self.meta_models = []
            for target_index in range(y_log.shape[1]):
                meta_X = np.column_stack([oof[name][:, target_index] for name in names])
                self.meta_models.append(Ridge(alpha=10.0).fit(meta_X, y_log[:, target_index]))
            factories = self.trainer.get_base_models()
            self.base_models = {}
            for name, model_list in factories.items():
                fitted, _ = self.trainer._fit_predict_models(model_list, X_full, y_log, X_full[:1])
                self.base_models[name] = fitted
        self.fitted = True
        return self
    def predict(self, X_eval, return_both=False):
        if not self.fitted: raise RuntimeError("Specialist must be fit before prediction.")
        X_eval = self.full_preprocessor.transform(np.asarray(X_eval, dtype=np.float64))
        if self.mode == "fast":
            prediction_log = self.fast_model.predict(X_eval)
        else:
            base = {}
            for name, model_list in self.base_models.items():
                values = np.column_stack([model.predict(X_eval) for model in model_list])
                base[name] = values
            prediction_log = np.zeros((len(X_eval), len(TARGET_COLS)), dtype=float)
            for target_index, meta_model in enumerate(self.meta_models):
                meta_X = np.column_stack([base[name][:, target_index] for name in ("lgbm", "xgb", "ridge")])
                prediction_log[:, target_index] = meta_model.predict(meta_X)
        raw = np.clip(np.expm1(prediction_log), 0, None)
        reconciled = reconcile_array(raw)
        self.raw_predictions_, self.reconciled_predictions_ = raw, reconciled
        selected = reconciled if self.use_reconciliation else raw
        return {"raw": raw, "reconciled": reconciled, "selected": selected} if return_both else selected
    def fit_predict(self, X_train, y_train, X_eval, sample_ids=None, forbidden_ids=(), return_both=False):
        return self.fit(X_train, y_train, sample_ids, forbidden_ids).predict(X_eval, return_both=return_both)
    @property
    def preprocessing_train_only_verified(self):
        return bool(self.preprocessing_audit) and all(row["forbidden_overlap"] == 0 for row in self.preprocessing_audit)

## 8. Candidate validation protocols

Split generation is independent of the predictor. Random K-fold is the conventional baseline; date, state, and date-state GroupKFold prevent acquisition groups from crossing folds. Temporal-block validation uses rolling-origin evaluation, where earlier contiguous date blocks predict the next block. Impossible protocols return an explicit unavailable result rather than being silently substituted.


In [ ]:
def _unavailable(protocol: str, reason: str) -> Dict[str, Any]:
    return {"protocol": protocol, "available": False, "reason_unavailable": reason,
            "splits": [], "n_folds": 0, "coverage": 0.0}


def _validate_split_pairs(frame: pd.DataFrame, protocol: str, split_pairs, min_validation_samples: int):
    valid = []
    for fit_idx, val_idx in split_pairs:
        fit_idx, val_idx = np.asarray(fit_idx, dtype=int), np.asarray(val_idx, dtype=int)
        if len(fit_idx) == 0 or len(val_idx) < min_validation_samples: continue
        if np.intersect1d(fit_idx, val_idx).size: raise AssertionError(f"{protocol} has row overlap.")
        if protocol == "date_group_kfold":
            assert set(frame.iloc[fit_idx]["sampling_date_dt"]).isdisjoint(frame.iloc[val_idx]["sampling_date_dt"])
        if protocol == "state_group_kfold":
            assert set(frame.iloc[fit_idx]["state_norm"]).isdisjoint(frame.iloc[val_idx]["state_norm"])
        if protocol == "date_state_group_kfold":
            fit_groups = frame.iloc[fit_idx]["state_norm"].astype(str) + "__" + frame.iloc[fit_idx]["sampling_date_dt"].dt.strftime("%Y-%m-%d")
            val_groups = frame.iloc[val_idx]["state_norm"].astype(str) + "__" + frame.iloc[val_idx]["sampling_date_dt"].dt.strftime("%Y-%m-%d")
            assert set(fit_groups).isdisjoint(val_groups)
        valid.append((fit_idx, val_idx))
    return valid


def make_validation_splits(frame: pd.DataFrame, protocol: str, n_folds=N_OUTER_FOLDS,
                           min_validation_samples=MIN_VALIDATION_SAMPLES, seed=MASTER_SEED):
    frame = frame.reset_index(drop=True)
    n_samples = len(frame)
    if protocol not in PROTOCOLS: return _unavailable(protocol, "unknown protocol")
    if n_samples < 2 * min_validation_samples:
        return _unavailable(protocol, "training pool too small for two viable partitions")
    if protocol == "random_kfold":
        for folds in range(min(n_folds, n_samples), 1, -1):
            splits = list(KFold(folds, shuffle=True, random_state=seed).split(np.arange(n_samples)))
            splits = _validate_split_pairs(frame, protocol, splits, min_validation_samples)
            if len(splits) == folds:
                covered = len(np.unique(np.concatenate([val for _, val in splits]))) / n_samples
                return {"protocol": protocol, "available": True, "reason_unavailable": None,
                        "splits": splits, "n_folds": folds, "coverage": float(covered)}
        return _unavailable(protocol, "validation folds would be too small")
    if protocol in {"date_group_kfold", "state_group_kfold", "date_state_group_kfold"}:
        if protocol == "date_group_kfold":
            if "sampling_date_dt" not in frame: return _unavailable(protocol, "date metadata unavailable")
            groups = frame["sampling_date_dt"].dt.strftime("%Y-%m-%d")
        elif protocol == "state_group_kfold":
            if "state_norm" not in frame: return _unavailable(protocol, "state metadata unavailable")
            groups = frame["state_norm"].astype("string")
        else:
            if "state_norm" not in frame: return _unavailable(protocol, "state metadata unavailable")
            groups = frame["state_norm"].astype(str) + "__" + frame["sampling_date_dt"].dt.strftime("%Y-%m-%d")
        if groups.isna().any(): return _unavailable(protocol, "group metadata contain missing values")
        n_groups = int(groups.nunique())
        if n_groups < 2: return _unavailable(protocol, "insufficient groups")
        for folds in range(min(n_folds, n_groups), 1, -1):
            splits = list(GroupKFold(folds).split(np.arange(n_samples), groups=groups))
            splits = _validate_split_pairs(frame, protocol, splits, min_validation_samples)
            if len(splits) == folds:
                covered = len(np.unique(np.concatenate([val for _, val in splits]))) / n_samples
                return {"protocol": protocol, "available": True, "reason_unavailable": None,
                        "splits": splits, "n_folds": folds, "coverage": float(covered), "n_groups": n_groups}
        return _unavailable(protocol, "grouped validation folds would be too small")
    # Rolling-origin validation over contiguous blocks of exact acquisition dates.
    dates = np.array(sorted(frame["sampling_date_dt"].dropna().unique()))
    if len(dates) < 3: return _unavailable(protocol, "too few temporal groups")
    for block_count in range(min(N_TEMPORAL_BLOCKS, len(dates)), 2, -1):
        blocks = [np.asarray(block) for block in np.array_split(dates, block_count) if len(block)]
        splits = []
        for block_index in range(1, len(blocks)):
            train_dates = np.concatenate(blocks[:block_index])
            validation_dates = blocks[block_index]
            fit_idx = np.flatnonzero(frame["sampling_date_dt"].isin(train_dates).to_numpy())
            val_idx = np.flatnonzero(frame["sampling_date_dt"].isin(validation_dates).to_numpy())
            if len(fit_idx) >= min_validation_samples and len(val_idx) >= min_validation_samples:
                assert frame.iloc[fit_idx]["sampling_date_dt"].max() < frame.iloc[val_idx]["sampling_date_dt"].min()
                splits.append((fit_idx, val_idx))
        if splits:
            covered = len(np.unique(np.concatenate([val for _, val in splits]))) / n_samples
            return {"protocol": protocol, "available": True, "reason_unavailable": None,
                    "splits": splits, "n_folds": len(splits), "coverage": float(covered),
                    "temporal_direction": "past_to_future"}
    return _unavailable(protocol, "no viable chronological train-to-future block")

## 9. Episode construction

Episode proposals are deterministic and outcome-blind. They include rolling future-time shifts, leave-one-state-out spatial shifts, conservative future-state intersections, and random in-distribution controls. Acceptance depends only on sample counts, metadata validity, disjointness, and the declared episode cap; accepted and rejected candidates are both recorded.


In [ ]:
@dataclass(frozen=True)
class DeploymentEpisode:
    episode_id: str
    episode_type: str
    train_ids: Tuple[str, ...]
    deployment_ids: Tuple[str, ...]
    random_seed: int
    metadata: Dict[str, Any] = field(default_factory=dict)


def _date_iso(value):
    return None if pd.isna(value) else pd.Timestamp(value).date().isoformat()


def _episode_summary(candidate_id, episode_type, train: pd.DataFrame, deploy: pd.DataFrame,
                     accepted, reason, seed):
    train_states = sorted(train["state_norm"].dropna().astype(str).unique()) if "state_norm" in train else []
    deploy_states = sorted(deploy["state_norm"].dropna().astype(str).unique()) if "state_norm" in deploy else []
    train_date_max = train["sampling_date_dt"].max() if len(train) else pd.NaT
    temporal_novelty = (float((deploy["sampling_date_dt"] > train_date_max).mean())
                        if len(deploy) and pd.notna(train_date_max) else None)
    return {
        "episode_candidate_id": candidate_id, "episode_type": episode_type,
        "accepted": bool(accepted), "rejection_reason": reason,
        "train_size": int(len(train)), "deployment_size": int(len(deploy)),
        "train_date_min": _date_iso(train["sampling_date_dt"].min()) if len(train) else None,
        "train_date_max": _date_iso(train["sampling_date_dt"].max()) if len(train) else None,
        "deploy_date_min": _date_iso(deploy["sampling_date_dt"].min()) if len(deploy) else None,
        "deploy_date_max": _date_iso(deploy["sampling_date_dt"].max()) if len(deploy) else None,
        "n_train_states": len(train_states), "n_deploy_states": len(deploy_states),
        "train_states": "|".join(train_states), "deployment_states": "|".join(deploy_states),
        "unseen_state": bool(set(deploy_states) - set(train_states)), "random_seed": int(seed),
        "deployment_later_than_train_max_fraction": temporal_novelty,
    }


def generate_deployment_episodes(df: pd.DataFrame, min_train_samples=MIN_TRAIN_SAMPLES,
                                 min_deploy_samples=MIN_DEPLOY_SAMPLES,
                                 max_episodes=MAX_EPISODES, master_seed=MASTER_SEED,
                                 n_id_controls=N_ID_CONTROLS):
    if not df["image_id"].is_unique: raise ValueError("Episodes require one row per image ID.")
    if "sampling_date_dt" not in df or df["sampling_date_dt"].isna().all():
        raise ValueError("Temporal episode generation requires parsed dates.")
    proposals = []
    unique_dates = np.array(sorted(df["sampling_date_dt"].dropna().unique()))
    temporal_blocks = [np.asarray(block) for block in np.array_split(unique_dates, min(N_TEMPORAL_BLOCKS, len(unique_dates))) if len(block)]
    for index in range(1, len(temporal_blocks)):
        train_dates = np.concatenate(temporal_blocks[:index])
        deploy_dates = temporal_blocks[index]
        proposals.append((f"temporal_{index:02d}", "temporal_future_shift",
                          df[df["sampling_date_dt"].isin(train_dates)],
                          df[df["sampling_date_dt"].isin(deploy_dates)], stable_seed("temporal", index, base=master_seed)))
    if "state_norm" in df:
        for state in sorted(df["state_norm"].dropna().astype(str).unique()):
            proposals.append((f"spatial_{state}", "spatial_state_holdout",
                              df[df["state_norm"].astype(str) != state],
                              df[df["state_norm"].astype(str) == state], stable_seed("spatial", state, base=master_seed)))
        if len(temporal_blocks) >= 2:
            late_dates = set(temporal_blocks[-1])
            earlier_dates = set(np.concatenate(temporal_blocks[:-1]))
            for state in sorted(df["state_norm"].dropna().astype(str).unique()):
                deploy_mask = (df["state_norm"].astype(str) == state) & df["sampling_date_dt"].isin(late_dates)
                train_mask = (df["state_norm"].astype(str) != state) & df["sampling_date_dt"].isin(earlier_dates)
                proposals.append((f"spatiotemporal_{state}", "spatiotemporal_shift",
                                  df[train_mask], df[deploy_mask], stable_seed("spatiotemporal", state, base=master_seed)))
    for replicate in range(n_id_controls):
        rng = np.random.default_rng(stable_seed("id_control", replicate, base=master_seed))
        deploy_n = max(min_deploy_samples, int(round(len(df) * ID_DEPLOY_FRACTION)))
        deploy_n = min(deploy_n, max(1, len(df) - min_train_samples))
        deploy_positions = np.sort(rng.choice(len(df), size=deploy_n, replace=False)) if deploy_n > 0 else np.array([], dtype=int)
        deploy_mask = np.zeros(len(df), dtype=bool); deploy_mask[deploy_positions] = True
        proposals.append((f"id_control_{replicate:02d}", "in_distribution_control",
                          df[~deploy_mask], df[deploy_mask], stable_seed("id_control", replicate, base=master_seed)))
    episodes, registry = [], []
    for candidate_id, episode_type, train, deploy, seed in proposals:
        train, deploy = train.copy(), deploy.copy()
        overlap = set(train["image_id"].astype(str)) & set(deploy["image_id"].astype(str))
        reason = None
        if overlap: reason = "train/deployment image overlap"
        elif len(train) < min_train_samples: reason = f"training size below {min_train_samples}"
        elif len(deploy) < min_deploy_samples: reason = f"deployment size below {min_deploy_samples}"
        elif len(episodes) >= max_episodes: reason = "predeclared MAX_EPISODES reached"
        accepted = reason is None
        summary = _episode_summary(candidate_id, episode_type, train, deploy, accepted, reason, seed)
        registry.append(summary)
        if accepted:
            episode_id = f"E{len(episodes) + 1:03d}_{candidate_id}"
            metadata_fields = {**summary, "episode_id": episode_id}
            episode = DeploymentEpisode(
                episode_id=episode_id, episode_type=episode_type,
                train_ids=tuple(train["image_id"].astype(str)),
                deployment_ids=tuple(deploy["image_id"].astype(str)),
                random_seed=seed, metadata=metadata_fields,
            )
            assert set(episode.train_ids).isdisjoint(episode.deployment_ids)
            episodes.append(episode)
    if not episodes:
        raise RuntimeError("No valid deployment episodes were generated; inspect episode_registry rejection reasons.")
    return episodes, pd.DataFrame(registry)


def save_episode_registry(episodes, registry: pd.DataFrame):
    ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
    registry.to_csv(ARTIFACT_DIR / "episode_registry.csv", index=False)
    accepted = pd.DataFrame([episode.metadata for episode in episodes])
    accepted.to_csv(ARTIFACT_DIR / "episodes.csv", index=False)
    membership = [{"episode_id": episode.episode_id, "train_ids": episode.train_ids,
                   "deployment_ids": episode.deployment_ids, "random_seed": episode.random_seed}
                  for episode in episodes]
    with gzip.open(ARTIFACT_DIR / "episode_membership.json.gz", "wt", encoding="utf-8") as handle:
        json.dump(membership, handle)


def _jensen_shannon_from_counts(train_values: pd.Series, deploy_values: pd.Series) -> Optional[float]:
    keys = sorted(set(train_values.dropna().astype(str)) | set(deploy_values.dropna().astype(str)))
    if not keys: return None
    p = train_values.astype(str).value_counts(normalize=True).reindex(keys, fill_value=0).to_numpy(float)
    q = deploy_values.astype(str).value_counts(normalize=True).reindex(keys, fill_value=0).to_numpy(float)
    midpoint = 0.5 * (p + q)
    def kl(a, b):
        mask = a > 0
        return float(np.sum(a[mask] * np.log2(a[mask] / b[mask])))
    return float(0.5 * kl(p, midpoint) + 0.5 * kl(q, midpoint))


def compute_shift_summary(train: pd.DataFrame, deploy: pd.DataFrame,
                          X_train: np.ndarray, X_deploy: np.ndarray) -> Dict[str, Any]:
    if len(train) != len(X_train) or len(deploy) != len(X_deploy): raise ValueError("Shift arrays are misaligned.")
    train_max = train["sampling_date_dt"].max()
    deploy_dates = deploy["sampling_date_dt"]
    future_fraction = float((deploy_dates > train_max).mean()) if pd.notna(train_max) else None
    train_states = set(train["state_norm"].dropna().astype(str)) if "state_norm" in train else set()
    deploy_states = set(deploy["state_norm"].dropna().astype(str)) if "state_norm" in deploy else set()
    unseen_fraction = (float(deploy["state_norm"].astype(str).isin(deploy_states - train_states).mean())
                       if "state_norm" in deploy and len(deploy_states) else None)
    train_centroid, deploy_centroid = X_train.mean(0), X_deploy.mean(0)
    cosine = 1.0 - float(np.dot(train_centroid, deploy_centroid) /
                         ((np.linalg.norm(train_centroid) * np.linalg.norm(deploy_centroid)) + 1e-12))
    scale = X_train.std(0); scale[scale < 1e-8] = 1.0
    standardized_distance = float(np.linalg.norm((deploy_centroid - train_centroid) / scale) / math.sqrt(X_train.shape[1]))
    neighbors = NearestNeighbors(n_neighbors=1).fit(X_train)
    nn_distance = float(neighbors.kneighbors(X_deploy, return_distance=True)[0].mean())
    date_counts = train["sampling_date_dt"].value_counts(normalize=True)
    summary = {
        "train_size": int(len(train)), "deployment_size": int(len(deploy)),
        "train_date_min": _date_iso(train["sampling_date_dt"].min()),
        "train_date_max": _date_iso(train_max),
        "deployment_date_min": _date_iso(deploy_dates.min()),
        "deployment_date_max": _date_iso(deploy_dates.max()),
        "future_date_fraction": future_fraction,
        "n_train_dates": int(train["sampling_date_dt"].nunique()),
        "n_deployment_dates": int(deploy_dates.nunique()),
        "train_date_max_group_fraction": float(date_counts.max()) if len(date_counts) else None,
        "n_train_states": len(train_states), "n_deployment_states": len(deploy_states),
        "train_states": sorted(train_states), "deployment_states": sorted(deploy_states),
        "train_state_distribution": (train["state_norm"].astype(str).value_counts(normalize=True).to_dict()
                                      if "state_norm" in train else None),
        "deployment_state_distribution": (deploy["state_norm"].astype(str).value_counts(normalize=True).to_dict()
                                           if "state_norm" in deploy else None),
        "unseen_states": sorted(deploy_states - train_states), "unseen_state_fraction": unseen_fraction,
        "state_distribution_js_divergence": (_jensen_shannon_from_counts(train["state_norm"], deploy["state_norm"])
                                                if "state_norm" in train else None),
        "embedding_centroid_cosine_distance": cosine,
        "embedding_standardized_centroid_distance": standardized_distance,
        "embedding_mean_nearest_train_distance": nn_distance,
        "small_sample_warning": bool(len(train) < 2 * MIN_TRAIN_SAMPLES or len(deploy) < 2 * MIN_DEPLOY_SAMPLES),
    }
    return summary

## 10. Validation evidence and deployment ground truth

Candidate validation results are precomputed from each episode's training pool and cached by episode, protocol, and specialist configuration hash. The primary score uses concatenated out-of-fold predictions over covered samples. Deployment performance is computed later in a separate `EpisodeGroundTruth` object and is never inserted into `EpisodeEvidence`, prompts, or tools.


In [ ]:
def _jsonable_protocol_result(result):
    clean = dict(result)
    clean.pop("splits", None)
    return clean


def evaluate_validation_protocol(X: np.ndarray, y: np.ndarray, frame: pd.DataFrame,
                                 protocol: str, mode=SPECIALIST_MODE, seed=MASTER_SEED,
                                 n_folds=N_OUTER_FOLDS, min_validation_samples=MIN_VALIDATION_SAMPLES):
    started = time.perf_counter()
    split_result = make_validation_splits(frame, protocol, n_folds, min_validation_samples, seed)
    config_hash = object_hash({"specialist": specialist_configuration(mode), "protocol": protocol,
                               "n_folds": n_folds, "min_validation_samples": min_validation_samples})
    if not split_result["available"]:
        return {**_jsonable_protocol_result(split_result), "n_validation_samples": 0,
                "weighted_r2": None, "weighted_mae": None, "weighted_rmse": None,
                "fold_scores": [], "fold_sizes": [], "fold_score_variance": None,
                "per_target_metrics": [],
                "runtime_s": time.perf_counter() - started, "configuration_hash": config_hash,
                "preprocessing_train_only": True}
    oof = np.full_like(y, np.nan, dtype=float)
    fold_scores, fold_sizes, audit_verified = [], [], []
    ids = frame["image_id"].astype(str).to_numpy()
    for fold_index, (fit_idx, val_idx) in enumerate(split_result["splits"]):
        model = SpecialistBiomassRegressor(mode=mode, seed=stable_seed(seed, protocol, fold_index),
                                           n_inner_folds=N_INNER_FOLDS, use_reconciliation=USE_RECONCILIATION)
        prediction = model.fit_predict(X[fit_idx], y[fit_idx], X[val_idx],
                                       sample_ids=ids[fit_idx], forbidden_ids=ids[val_idx])
        oof[val_idx] = prediction
        fold_scores.append(weighted_r2_metric(y[val_idx], prediction))
        fold_sizes.append(int(len(val_idx)))
        audit_verified.append(model.preprocessing_train_only_verified)
    covered = ~np.isnan(oof).any(axis=1)
    if not covered.any(): raise AssertionError("Available protocol produced no OOF predictions.")
    summary, per_target = calculate_regression_metrics(y[covered], oof[covered], protocol)
    return {
        "protocol": protocol, "available": True, "reason_unavailable": None,
        "n_folds": int(len(fold_scores)), "n_validation_samples": int(covered.sum()),
        "coverage": float(covered.mean()), "weighted_r2": summary["Weighted_R2"],
        "weighted_mae": summary["Weighted_MAE_g"], "weighted_rmse": summary["Weighted_RMSE_g"],
        "fold_scores": list(map(float, fold_scores)), "fold_sizes": fold_sizes,
        "fold_score_variance": float(np.var(fold_scores, ddof=1)) if len(fold_scores) > 1 else 0.0,
        "per_target_metrics": per_target.drop(columns="Model").to_dict(orient="records"),
        "signed_bias_diagnostic": None, "runtime_s": float(time.perf_counter() - started),
        "configuration_hash": config_hash, "preprocessing_train_only": bool(all(audit_verified)),
    }


@dataclass
class EpisodeEvidence:
    episode_id: str
    initial_observation: Dict[str, Any]
    shift_summary: Dict[str, Any]
    protocol_results: Dict[str, Dict[str, Any]] = field(repr=False)
    def agent_visible_snapshot(self):
        return {"episode_id": self.episode_id, "initial_observation": self.initial_observation,
                "unlabeled_shift_diagnostics": self.shift_summary}


@dataclass
class EpisodeGroundTruth:
    episode_id: str
    deployment_metrics: Dict[str, Any]
    raw_predictions: np.ndarray = field(repr=False)
    selected_predictions: np.ndarray = field(repr=False)
    deployment_targets: np.ndarray = field(repr=False)
    preprocessing_train_only: bool = True


def build_initial_observation(train: pd.DataFrame, deploy: pd.DataFrame,
                              shift_summary: Mapping[str, Any], protocol_results):
    # The same concise, label-free observation is used for LLM-only and tool-agent methods.
    return {
        "biological_task": "five-output pasture biomass prediction from field images",
        "target_names": TARGET_COLS,
        "n_training_samples": int(len(train)), "n_unlabeled_deployment_samples": int(len(deploy)),
        "metadata_fields_available": ["sampling_date_dt"] + (["state_norm"] if "state_norm" in train else []),
        "candidate_protocols": PROTOCOLS,
        "protocol_availability": {name: bool(result["available"]) for name, result in protocol_results.items()},
        "concise_unlabeled_shift_signals": {
            key: shift_summary.get(key) for key in (
                "future_date_fraction", "unseen_state_fraction",
                "embedding_centroid_cosine_distance", "embedding_standardized_centroid_distance",
                "small_sample_warning")},
        "deployment_labels_available": False,
    }

# v2 cache-safe overrides. Every result-affecting seed/configuration participates in cache identity.
def specialist_configuration(mode=PRIMARY_SPECIALIST_MODE):
    label = "DINOv2 + Ridge specialist" if mode == "fast" else "DINOv2 + LGBM/XGB/Ridge + meta-Ridge"
    return {"mode": mode, "paper_label": label, "n_inner_folds": N_INNER_FOLDS,
            "use_reconciliation": USE_RECONCILIATION, "targets": TARGET_COLS, "weights": TARGET_WEIGHTS,
            "lgbm": {"n_estimators": 300, "learning_rate": 0.05, "num_leaves": 31},
            "xgb": {"n_estimators": 300, "learning_rate": 0.05, "max_depth": 6,
                    "subsample": 0.8, "colsample_bytree": 0.8},
            "ridge_alpha": 10.0 if mode == "fast" else 1.0, "meta_ridge_alpha": 10.0,
            "use_gpu_trees": USE_GPU_TREES, "code_config_version": CODE_CONFIG_VERSION}


def protocol_cache_configuration(episode, protocol, mode, train_frame, X_train, y_train):
    return {"episode_id": episode.episode_id, "protocol": protocol,
            "ordered_train_ids": train_frame["image_id"].astype(str).tolist(),
            "train_feature_hash": array_sha256(np.asarray(X_train)),
            "train_target_hash": array_sha256(np.asarray(y_train)),
            "specialist": specialist_configuration(mode), "reconciliation": USE_RECONCILIATION,
            "episode_random_seed": episode.random_seed, "master_seed": MASTER_SEED,
            "outer_folds": N_OUTER_FOLDS, "inner_folds": N_INNER_FOLDS,
            "minimum_validation_samples": MIN_VALIDATION_SAMPLES,
            "notebook_version": NOTEBOOK_VERSION, "code_config_version": CODE_CONFIG_VERSION,
            "device": str(CFG.device), "tree_gpu": USE_GPU_TREES}


def evaluate_protocols_for_episode(episode, train_frame, X_train, y_train, mode=PRIMARY_SPECIALIST_MODE):
    cache_dir = ARTIFACT_DIR / "protocol_cache"; cache_dir.mkdir(parents=True, exist_ok=True)
    results = {}
    for protocol in PROTOCOLS:
        cache_configuration = protocol_cache_configuration(
            episode, protocol, mode, train_frame, X_train, y_train)
        cache_hash = object_hash(cache_configuration)
        path = cache_dir / f"{episode.episode_id}__{mode}__{protocol}__{cache_hash[:16]}.json"
        if path.exists() and not FORCE_RECOMPUTE_PROTOCOLS:
            payload = json.loads(path.read_text(encoding="utf-8"))
            if payload.get("cache_configuration_hash") == cache_hash and payload.get("cache_configuration") == cache_configuration:
                results[protocol] = payload["result"]
                continue
        result = evaluate_validation_protocol(
            X_train, y_train, train_frame, protocol, mode=mode,
            seed=stable_seed(episode.random_seed, protocol, mode),
            n_folds=N_OUTER_FOLDS, min_validation_samples=MIN_VALIDATION_SAMPLES)
        result["core_configuration_hash"] = result.get("configuration_hash")
        result["configuration_hash"] = cache_hash
        result["cache_configuration_hash"] = cache_hash
        result["nested_preprocessing_fold_local"] = bool(
            mode != "nested_stacking" or result.get("preprocessing_train_only", False))
        atomic_json_dump({"cache_configuration_hash": cache_hash,
                          "cache_configuration": cache_configuration, "result": result}, path)
        results[protocol] = result
    if not any(result["available"] for result in results.values()):
        raise RuntimeError(f"All validation protocols are unavailable for {episode.episode_id}.")
    return results


def deployment_cache_configuration(episode, mode, X_train, X_deploy, y_train, y_deploy):
    return {"episode_id": episode.episode_id, "ordered_train_ids": list(episode.train_ids),
            "ordered_deployment_ids": list(episode.deployment_ids),
            "train_target_hash": array_sha256(np.asarray(y_train)),
            "deployment_target_hash": array_sha256(np.asarray(y_deploy)),
            "train_feature_hash": array_sha256(np.asarray(X_train)),
            "deployment_feature_hash": array_sha256(np.asarray(X_deploy)),
            "specialist": specialist_configuration(mode), "reconciliation": USE_RECONCILIATION,
            "episode_random_seed": episode.random_seed, "master_seed": MASTER_SEED,
            "notebook_version": NOTEBOOK_VERSION, "code_config_version": CODE_CONFIG_VERSION,
            "device": str(CFG.device), "tree_gpu": USE_GPU_TREES}


def compute_deployment_ground_truth(episode, train_frame, deploy_frame, X_train, X_deploy,
                                    y_train, y_deploy, mode=PRIMARY_SPECIALIST_MODE):
    cache_dir = ARTIFACT_DIR / "deployment_predictions"; cache_dir.mkdir(parents=True, exist_ok=True)
    configuration = deployment_cache_configuration(episode, mode, X_train, X_deploy, y_train, y_deploy)
    config_hash = object_hash(configuration)
    metrics_path = cache_dir / f"{episode.episode_id}__{mode}__{config_hash[:16]}.json"
    pred_path = cache_dir / f"{episode.episode_id}__{mode}__{config_hash[:16]}.npz"
    if metrics_path.exists() and pred_path.exists() and not FORCE_RECOMPUTE_PROTOCOLS:
        payload = json.loads(metrics_path.read_text(encoding="utf-8"))
        if payload.get("cache_configuration_hash") == config_hash and payload.get("cache_configuration") == configuration:
            with np.load(pred_path, allow_pickle=False) as saved:
                raw, selected = saved["raw"], saved["selected"]
            return EpisodeGroundTruth(episode.episode_id, payload["metrics"], raw, selected,
                                      np.asarray(y_deploy), payload["preprocessing_train_only"])
    model = SpecialistBiomassRegressor(mode=mode, seed=episode.random_seed,
                                       n_inner_folds=N_INNER_FOLDS, use_reconciliation=USE_RECONCILIATION)
    prediction = model.fit_predict(X_train, y_train, X_deploy,
                                   sample_ids=train_frame["image_id"].astype(str),
                                   forbidden_ids=deploy_frame["image_id"].astype(str), return_both=True)
    summary, per_target = calculate_regression_metrics(y_deploy, prediction["selected"], "deployment")
    metrics = {**summary, "configuration_hash": config_hash,
               "per_target": per_target.drop(columns="Model").to_dict(orient="records")}
    np.savez_compressed(pred_path, image_ids=deploy_frame["image_id"].astype(str).to_numpy(),
                        raw=prediction["raw"], reconciled=prediction["reconciled"], selected=prediction["selected"])
    atomic_json_dump({"cache_configuration_hash": config_hash, "cache_configuration": configuration,
                      "metrics": metrics, "preprocessing_train_only": model.preprocessing_train_only_verified},
                     metrics_path)
    return EpisodeGroundTruth(episode.episode_id, metrics, prediction["raw"], prediction["selected"],
                              np.asarray(y_deploy), model.preprocessing_train_only_verified)

## 11. Local Qwen selectors and baselines

Qwen3-4B is loaded with Transformers from local files only. The primary configuration is deterministic FP16 with thinking disabled; a 4-bit fallback is attempted only after CUDA out-of-memory and only when a compatible `bitsandbytes` installation is already present.

LLM-only and tool-agent conditions share the checkpoint, generation settings, initial evidence, protocol vocabulary, and task framing. The only difference is access to allowlisted local tools that return sanitized, precomputed validation evidence. Voluntary abstention is disabled, and invalid decisions or runtime failures are recorded without fallback.


In [ ]:
# Versioned local-Qwen prompts. The shared framing and initial observation are identical.
COMMON_VALIDATION_SYSTEM_PROMPT = """
You select a validation strategy for a five-output biological image-prediction model.
Deployment labels are unavailable. Select a strategy that estimates deployment performance honestly;
do not choose merely because its local validation score is high. Use only observable metadata,
unlabeled representation diagnostics, and explicitly returned local tool evidence. Never fabricate evidence.
Return exactly one JSON object and no hidden reasoning or surrounding prose.
The final schema is:
{"action":"final","selected_protocol":"<protocol>","confidence":0.0,
 "risk_level":"low|medium|high","reason_codes":["<allowed code>"],
 "evidence_summary":"brief evidence-based explanation"}.
Voluntary abstention is not allowed. Do not infer a scenario label; reason only from measured evidence.
""".strip()

SCIENTIFIC_AGENT_SYSTEM_PROMPT = (COMMON_VALIDATION_SYSTEM_PROMPT + """

You may instead request one allowlisted local tool with:
{"action":"tool","tool_name":"<allowed tool>","arguments":{}}.
Python—not you—executes the predefined tool. You cannot execute code, files, shell commands, or network calls.
Use tools selectively and return a final JSON action within the fixed budget.
""").strip()

LLM_ONLY_SYSTEM_PROMPT = (COMMON_VALIDATION_SYSTEM_PROMPT + """

You have no tools in this condition and must make a direct final choice from the supplied protocol vocabulary.
""").strip()


def save_prompts():
    prompt_dir = ARTIFACT_DIR / "prompts"; prompt_dir.mkdir(parents=True, exist_ok=True)
    (prompt_dir / f"common_{AGENT_PROMPT_VERSION}.txt").write_text(COMMON_VALIDATION_SYSTEM_PROMPT, encoding="utf-8")
    (prompt_dir / f"scientific_agent_{AGENT_PROMPT_VERSION}.txt").write_text(SCIENTIFIC_AGENT_SYSTEM_PROMPT, encoding="utf-8")
    (prompt_dir / f"llm_only_{AGENT_PROMPT_VERSION}.txt").write_text(LLM_ONLY_SYSTEM_PROMPT, encoding="utf-8")

In [ ]:
PROTOCOL_TOOL_NAMES = {
    "run_random_cv": "random_kfold", "run_date_cv": "date_group_kfold",
    "run_state_cv": "state_group_kfold", "run_date_state_cv": "date_state_group_kfold",
    "run_temporal_holdout": "temporal_block_holdout",
}
METADATA_TOOL_NAMES = {"inspect_dataset_summary", "inspect_temporal_structure", "inspect_spatial_structure"}
SHIFT_TOOL_NAMES = {"inspect_embedding_shift"}
ALL_TOOL_NAMES = METADATA_TOOL_NAMES | SHIFT_TOOL_NAMES | set(PROTOCOL_TOOL_NAMES)
PROTOCOL_RESULT_ALLOWLIST = {
    "protocol", "available", "reason_unavailable", "n_folds", "n_validation_samples", "coverage",
    "weighted_r2", "weighted_mae", "weighted_rmse", "fold_scores", "fold_sizes",
    "fold_score_variance", "runtime_s",
}
FORBIDDEN_AGENT_KEYS = {
    "deployment_metrics", "deployment_targets", "deployment_r2", "oracle_protocol", "oracle_gap",
    "absolute_gap", "signed_gap", "regret_vs_oracle", "episode_type", "episode_category",
    "temporal_future_shift", "spatial_state_holdout", "spatiotemporal_shift", "in_distribution_control",
}


def _recursive_keys(value):
    if isinstance(value, Mapping):
        for key, child in value.items():
            yield str(key)
            yield from _recursive_keys(child)
    elif isinstance(value, (list, tuple)):
        for child in value: yield from _recursive_keys(child)


def assert_agent_payload_safe(payload):
    present = set(_recursive_keys(payload)) & FORBIDDEN_AGENT_KEYS
    if present: raise AssertionError(f"Evaluator-only fields reached agent payload: {sorted(present)}")
    serialized = canonical_json(payload).casefold()
    for forbidden in ("oracle_protocol", "deployment_r2", "regret_vs_oracle",
                      "temporal_future_shift", "spatial_state_holdout", "spatiotemporal_shift",
                      "in_distribution_control"):
        if forbidden in serialized: raise AssertionError(f"Forbidden evaluator token reached agent payload: {forbidden}")
    return True


def sanitize_protocol_tool_result(result):
    sanitized = {key: result.get(key) for key in PROTOCOL_RESULT_ALLOWLIST if key in result}
    assert_agent_payload_safe(sanitized)
    return sanitized


class ScientificValidationTools:
    """Strict local allowlist. No arbitrary Python, shell, filesystem, or network execution exists."""
    def __init__(self, evidence, allowed_tools=None):
        self.evidence = evidence
        self.allowed_tools = set(ALL_TOOL_NAMES if allowed_tools is None else allowed_tools)
        if not self.allowed_tools.issubset(ALL_TOOL_NAMES): raise ValueError("Unknown tool in allowlist.")
    def available_tool_names(self): return sorted(self.allowed_tools)
    def call(self, name, arguments=None):
        if name not in self.allowed_tools: raise ValueError(f"Tool {name!r} is not available.")
        shift, initial = self.evidence.shift_summary, self.evidence.initial_observation
        if name == "inspect_dataset_summary":
            result = {key: initial[key] for key in (
                "biological_task", "target_names", "n_training_samples", "n_unlabeled_deployment_samples",
                "metadata_fields_available", "candidate_protocols", "protocol_availability",
                "deployment_labels_available")}
            result.update({key: shift.get(key) for key in (
                "n_train_dates", "n_deployment_dates", "n_train_states", "n_deployment_states")})
        elif name == "inspect_temporal_structure":
            result = {key: shift.get(key) for key in (
                "train_date_min", "train_date_max", "deployment_date_min", "deployment_date_max",
                "future_date_fraction", "n_train_dates", "n_deployment_dates", "train_date_max_group_fraction")}
        elif name == "inspect_spatial_structure":
            result = {key: shift.get(key) for key in (
                "n_train_states", "n_deployment_states", "train_states", "deployment_states",
                "train_state_distribution", "deployment_state_distribution", "unseen_states",
                "unseen_state_fraction", "state_distribution_js_divergence")}
        elif name == "inspect_embedding_shift":
            result = {key: shift.get(key) for key in (
                "embedding_centroid_cosine_distance", "embedding_standardized_centroid_distance",
                "embedding_mean_nearest_train_distance", "small_sample_warning")}
        elif name in PROTOCOL_TOOL_NAMES:
            result = sanitize_protocol_tool_result(self.evidence.protocol_results[PROTOCOL_TOOL_NAMES[name]])
        else:
            raise ValueError(f"Unknown tool: {name}")
        assert_agent_payload_safe(result)
        return result


class BaseLLMClient:
    backend = "base"
    model = "base"
    def complete(self, system_prompt, user_payload, seed): raise NotImplementedError
    def cache_identity(self): return {"backend": self.backend, "model": self.model}
    def unload(self): return release_cuda_memory("base_client_unload")


class MockAgentClient(BaseLLMClient):
    """Deterministic smoke plumbing only; never a scientific result."""
    backend, model = "mock", "mock-local-agent-v2"
    def __init__(self, fail=False, malformed=False):
        self.calls, self.fail, self.malformed = 0, fail, malformed
    def complete(self, system_prompt, user_payload, seed):
        self.calls += 1
        if self.fail: raise RuntimeError("synthetic generation failure")
        if self.malformed: return {"text": "not-json", "usage": None}
        history, tools = user_payload.get("tool_history", []), user_payload.get("available_tools", [])
        if tools and not history and "inspect_temporal_structure" in tools:
            action = {"action": "tool", "tool_name": "inspect_temporal_structure", "arguments": {}}
        else:
            observation = user_payload.get("initial_observation", {})
            available = {name for name, flag in observation.get("protocol_availability", {}).items() if flag}
            signals = observation.get("concise_unlabeled_shift_signals", {})
            if (signals.get("future_date_fraction") or 0) >= 0.5 and "temporal_block_holdout" in available:
                protocol, codes = "temporal_block_holdout", ["future_temporal_shift", "temporal_holdout_supported"]
            else:
                protocol = "random_kfold" if "random_kfold" in available else sorted(available)[0]
                codes = ["embedding_shift_low"]
            action = {"action": "final", "selected_protocol": protocol, "confidence": 0.6,
                      "risk_level": "medium", "reason_codes": codes,
                      "evidence_summary": "Deterministic mock output for infrastructure testing."}
        return {"text": canonical_json(action), "usage": {"input_tokens": None, "output_tokens": None,
                                                            "generation_latency_s": 0.0}}
    def unload(self): return {"mock": True, "substantial_release": True, "allocated_bytes": 0}


def validate_local_llm_path(path=None):
    path = resolve_local_model_path(path if path is not None else LOCAL_LLM_PATH,
                                    ["qwen3-4b", "qwen3", "qwen"], ["config.json"], "Qwen3-4B model")
    tokenizer_candidates = ["tokenizer.json", "tokenizer.model", "vocab.json"]
    if not any((path / filename).exists() for filename in tokenizer_candidates):
        raise FileNotFoundError(f"Qwen directory {path} lacks tokenizer files {tokenizer_candidates}.")
    return path


class LocalTransformersClient(BaseLLMClient):
    backend = "transformers"
    def __init__(self, model_path=None):
        if not TORCH_STACK_AVAILABLE or AutoModelForCausalLM is None or AutoTokenizer is None:
            raise ImportError(f"Local Transformers causal-LM stack is unavailable: {TORCH_IMPORT_ERROR}")
        require_transformers_version()
        if not torch.cuda.is_available(): raise RuntimeError("CUDA is required for the primary local Qwen3-4B experiment.")
        self.path = validate_local_llm_path(model_path)
        self.model = LOCAL_LLM_MODEL_NAME
        self.quantization_mode = "fp16"
        self.dtype = "float16"
        self.before_load = cuda_memory_snapshot("before_qwen_load")
        self.tokenizer = AutoTokenizer.from_pretrained(
            str(self.path), local_files_only=True, trust_remote_code=False)
        load_kwargs = {"local_files_only": True, "trust_remote_code": False,
                       "torch_dtype": torch.float16, "device_map": "auto", "low_cpu_mem_usage": True}
        try:
            self.model_object = AutoModelForCausalLM.from_pretrained(str(self.path), **load_kwargs).eval()
        except RuntimeError as error:
            if "out of memory" not in str(error).casefold() or not LOCAL_LLM_ALLOW_4BIT_FALLBACK:
                raise
            release_cuda_memory("after_qwen_fp16_oom")
            try:
                from transformers import BitsAndBytesConfig
                if get_package_version("bitsandbytes") is None:
                    raise ImportError("bitsandbytes is not installed")
                quantization = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16)
                quantized_kwargs = {**load_kwargs, "quantization_config": quantization}
                quantized_kwargs.pop("torch_dtype", None)
                self.model_object = AutoModelForCausalLM.from_pretrained(
                    str(self.path), **quantized_kwargs).eval()
                self.quantization_mode, self.dtype = "4bit", "float16_compute"
            except Exception as fallback_error:
                raise RuntimeError(
                    "Qwen3-4B FP16 loading exhausted CUDA memory and the optional preinstalled 4-bit fallback failed: "
                    f"{fallback_error}") from fallback_error
        self.after_load = cuda_memory_snapshot("after_qwen_load")
        self.parameter_count = int(sum(parameter.numel() for parameter in self.model_object.parameters()))
        self.model_config_hash = file_sha256(self.path / "config.json")
        self.thinking_flag_supported = None
        print(f"Loaded {LOCAL_LLM_MODEL_NAME}: {self.parameter_count:,} parameters, "
              f"mode={self.quantization_mode}, device={self.primary_device}")
    @property
    def primary_device(self):
        try: return next(self.model_object.parameters()).device
        except StopIteration: return torch.device("cuda:0")
    def cache_identity(self):
        return {"backend": self.backend, "model_name": LOCAL_LLM_MODEL_NAME,
                "model_path": str(self.path.resolve()), "model_config_hash": self.model_config_hash,
                "dtype": self.dtype, "quantization_mode": self.quantization_mode,
                "enable_thinking": LOCAL_LLM_ENABLE_THINKING, "do_sample": LOCAL_LLM_DO_SAMPLE,
                "max_new_tokens": LOCAL_LLM_MAX_NEW_TOKENS, "temperature": LOCAL_LLM_TEMPERATURE,
                "transformers_version": TRANSFORMERS_VERSION}
    def _render_chat(self, messages):
        try:
            rendered = self.tokenizer.apply_chat_template(
                messages, tokenize=False, add_generation_prompt=True,
                enable_thinking=LOCAL_LLM_ENABLE_THINKING)
            self.thinking_flag_supported = True
            return rendered
        except TypeError:
            if LOCAL_LLM_ENABLE_THINKING:
                raise RuntimeError("Tokenizer chat template does not support the required thinking-mode control.")
            self.thinking_flag_supported = False
            return self.tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    def complete(self, system_prompt, user_payload, seed):
        messages = [{"role": "system", "content": system_prompt},
                    {"role": "user", "content": canonical_json(user_payload)}]
        rendered = self._render_chat(messages)
        inputs = self.tokenizer(rendered, return_tensors="pt")
        inputs = {key: value.to(self.primary_device) for key, value in inputs.items()}
        input_tokens = int(inputs["input_ids"].shape[-1])
        pad_id = self.tokenizer.pad_token_id
        eos_id = self.tokenizer.eos_token_id
        if pad_id is None: pad_id = eos_id[0] if isinstance(eos_id, list) else eos_id
        generation_kwargs = {
            "max_new_tokens": LOCAL_LLM_MAX_NEW_TOKENS, "do_sample": LOCAL_LLM_DO_SAMPLE,
            "pad_token_id": pad_id, "eos_token_id": eos_id,
        }
        if LOCAL_LLM_DO_SAMPLE: generation_kwargs["temperature"] = LOCAL_LLM_TEMPERATURE
        started = time.perf_counter()
        with torch.inference_mode():
            generated = self.model_object.generate(**inputs, **generation_kwargs)
        latency = time.perf_counter() - started
        new_tokens = generated[0, input_tokens:]
        text = self.tokenizer.decode(new_tokens, skip_special_tokens=True).strip()
        return {"text": text, "usage": {"input_tokens": input_tokens,
                                         "output_tokens": int(new_tokens.numel()),
                                         "generation_latency_s": float(latency)}}
    def unload(self):
        before = cuda_memory_snapshot("before_qwen_unload")
        if hasattr(self, "model_object"): del self.model_object
        if hasattr(self, "tokenizer"): del self.tokenizer
        return release_cuda_memory("after_qwen_unload", before)


class OpenAIResponsesClient(BaseLLMClient):
    """Optional legacy adapter. It is never imported or used by the primary offline experiment."""
    backend = "openai-legacy"
    def __init__(self, model, temperature=0.0):
        try: from openai import OpenAI
        except ImportError as error: raise ImportError("Optional legacy OpenAI adapter requires openai.") from error
        self.client, self.model, self.temperature = OpenAI(), model, temperature
    def complete(self, system_prompt, user_payload, seed):
        response = self.client.responses.create(model=self.model, instructions=system_prompt,
                                                input=canonical_json(user_payload), temperature=self.temperature)
        return {"text": response.output_text, "usage": None}


def create_llm_client(mock=False):
    if mock: return MockAgentClient()
    if not EXECUTE_LLM_CALLS: raise RuntimeError("Set EXECUTE_LLM_CALLS=True only for the explicit final run.")
    if LOCAL_LLM_BACKEND != "transformers":
        raise ValueError("Primary FINAL execution supports only LOCAL_LLM_BACKEND='transformers'.")
    return LocalTransformersClient(LOCAL_LLM_PATH)


class InvalidModelDecisionError(ValueError):
    """A model-produced scientific/action error, distinct from infrastructure failure."""
    def __init__(self, invalid_reason, message):
        super().__init__(message)
        self.invalid_reason = invalid_reason


def parse_json_action(text):
    """Accept one JSON object; classify malformed text separately from invalid action semantics."""
    stripped = str(text).strip()
    fenced = re.fullmatch(r"```(?:json)?\s*(\{.*\})\s*```", stripped, flags=re.DOTALL | re.IGNORECASE)
    candidate = fenced.group(1) if fenced else stripped
    if not (candidate.startswith("{") and candidate.endswith("}")):
        raise InvalidModelDecisionError(
            "invalid_json", "Response must be raw JSON or a single fenced JSON object without surrounding prose.")
    try:
        action = json.loads(candidate)
    except (json.JSONDecodeError, TypeError) as error:
        raise InvalidModelDecisionError("invalid_json", f"Malformed JSON: {error}") from error
    if not isinstance(action, dict):
        raise InvalidModelDecisionError("invalid_model_action", "Response must be a JSON object.")
    if action.get("action") not in {"tool", "final"}:
        raise InvalidModelDecisionError("invalid_model_action", "JSON action must be 'tool' or 'final'.")
    return action


def validate_final_action(action, available_protocols):
    if action.get("action") != "final":
        raise InvalidModelDecisionError("invalid_model_action", "Expected a final action.")
    required = {"selected_protocol", "confidence", "risk_level", "reason_codes", "evidence_summary"}
    missing = required - set(action)
    if missing:
        raise InvalidModelDecisionError(
            "invalid_model_action", f"Final action is missing fields: {sorted(missing)}")
    protocol = action["selected_protocol"]
    if protocol not in set(PROTOCOLS) | {"ABSTAIN"}:
        raise InvalidModelDecisionError("invalid_model_action", f"Invalid protocol name: {protocol}")
    try:
        confidence = float(action["confidence"])
    except (TypeError, ValueError) as error:
        raise InvalidModelDecisionError("invalid_model_action", "confidence must be numeric.") from error
    if not 0 <= confidence <= 1:
        raise InvalidModelDecisionError("invalid_model_action", "confidence must lie in [0,1].")
    if action["risk_level"] not in {"low", "medium", "high"}:
        raise InvalidModelDecisionError("invalid_model_action", "Invalid risk level.")
    if not isinstance(action["reason_codes"], list) or not set(action["reason_codes"]).issubset(REASON_CODES):
        raise InvalidModelDecisionError("invalid_model_action", "Invalid reason-code vocabulary.")
    if not isinstance(action["evidence_summary"], str) or len(action["evidence_summary"]) > 600:
        raise InvalidModelDecisionError(
            "invalid_model_action", "evidence_summary must be a concise string (<=600 characters).")
    result = dict(action)
    result.update({"invalid_action": False, "decision_status": "valid_decision", "invalid_reason": None})
    if protocol == "ABSTAIN" and not ALLOW_VOLUNTARY_ABSTAIN:
        result.update({"invalid_action": True, "decision_status": "invalid_decision",
                       "invalid_reason": "voluntary_abstain_not_allowed"})
    elif protocol not in set(available_protocols):
        result.update({"invalid_action": True, "decision_status": "invalid_decision",
                       "invalid_reason": "selected_protocol_unavailable"})
    return result


def _available_protocols(evidence):
    return sorted(name for name, result in evidence.protocol_results.items() if result.get("available"))


def _llm_cache_key(evidence, client, agent_type, replicate, allowed_tools):
    visible = evidence.agent_visible_snapshot(); assert_agent_payload_safe(visible)
    tool_evidence = None
    if agent_type == "tool_agent":
        allowed = set(allowed_tools or [])
        tool_evidence = {tool: sanitize_protocol_tool_result(evidence.protocol_results[protocol])
                         for tool, protocol in PROTOCOL_TOOL_NAMES.items() if tool in allowed}
    prompt = LLM_ONLY_SYSTEM_PROMPT if agent_type == "llm_only" else SCIENTIFIC_AGENT_SYSTEM_PROMPT
    return object_hash({"episode_id": evidence.episode_id, "client": client.cache_identity(),
                        "agent_type": agent_type, "replicate": replicate,
                        "prompt_version": AGENT_PROMPT_VERSION, "prompt_hash": object_hash(prompt),
                        "visible_evidence_hash": object_hash(visible),
                        "allowed_tools": sorted(allowed_tools or []),
                        "tool_visible_protocol_evidence": tool_evidence,
                        "allow_voluntary_abstain": ALLOW_VOLUNTARY_ABSTAIN,
                        "code_config_version": CODE_CONFIG_VERSION})


def _require_valid_final_action(action, available_protocols):
    """Return a valid final decision or raise so bounded repair can correct it."""
    decision = validate_final_action(action, available_protocols)
    if decision.get("decision_status") != "valid_decision":
        reason = decision.get("invalid_reason") or "invalid_model_action"
        raise InvalidModelDecisionError(
            reason,
            f"Final choice is not valid: {reason}. "
            f"Choose one of available_protocols={sorted(available_protocols)} and do not ABSTAIN."
        )
    return decision


def _validate_llm_only_action_semantics(action, available_protocols):
    if action.get("action") != "final":
        raise InvalidModelDecisionError(
            "invalid_tool_request",
            "LLM-only condition must return a final action and cannot request tools."
        )
    _require_valid_final_action(action, available_protocols)
    return True


def _validate_tool_agent_action_semantics(action, available_protocols, allowed_tools):
    action_type = action.get("action")
    if action_type == "final":
        _require_valid_final_action(action, available_protocols)
        return True
    if action_type != "tool":
        raise InvalidModelDecisionError("invalid_model_action", f"Unsupported action type: {action_type!r}")
    tool_name = action.get("tool_name")
    arguments = action.get("arguments", {})
    if tool_name not in allowed_tools or tool_name not in ALL_TOOL_NAMES:
        raise InvalidModelDecisionError(
            "invalid_tool_request",
            f"Tool {tool_name!r} is not available. Choose one of {sorted(allowed_tools)}."
        )
    if not isinstance(arguments, Mapping):
        raise InvalidModelDecisionError("invalid_tool_request", "Tool arguments must be a JSON object.")
    return True


def _call_with_bounded_repair(client, system_prompt, payload, seed,
                              semantic_validator=None, audit_context=None):
    """Bounded parse + semantic repair; infrastructure errors still propagate."""
    errors, reasons = [], []
    for attempt in range(MAX_LLM_RETRIES + 1):
        request = dict(payload)
        if errors:
            request["repair_instruction"] = (
                "Your previous response was invalid. "
                f"Validation error: {errors[-1]}. "
                "Return ONLY one corrected JSON object and no prose. "
                "For a tool action, tool_name must be exactly one of available_tools and arguments must be a JSON object. "
                "For a final action, selected_protocol must be exactly one of available_protocols; do not ABSTAIN; "
                "confidence must be in [0,1], risk_level must be low|medium|high, and reason_codes must use only allowed_reason_codes."
            )
        response = client.complete(system_prompt, request, seed + attempt)
        raw_text = response.get("text", "")
        try:
            action = parse_json_action(raw_text)
            if semantic_validator is not None:
                semantic_validator(action)
            return action, response.get("usage"), attempt
        except InvalidModelDecisionError as error:
            errors.append(str(error)); reasons.append(error.invalid_reason)
            if audit_context is not None:
                append_jsonl(
                    ARTIFACT_DIR / "invalid_llm_attempts.jsonl",
                    [{**dict(audit_context), "attempt": int(attempt),
                      "error_type": type(error).__name__,
                      "invalid_reason": error.invalid_reason,
                      "validation_error": str(error),
                      "raw_response": raw_text}],
                )
    reason = "invalid_json" if reasons and all(item == "invalid_json" for item in reasons) else "invalid_model_action"
    raise InvalidModelDecisionError(reason, "exhausted_action_repair_attempts: " + " | ".join(errors))


def run_llm_only_selector(evidence, client, replicate=0):
    cache_dir = ARTIFACT_DIR / "llm_cache"; cache_dir.mkdir(parents=True, exist_ok=True)
    key = _llm_cache_key(evidence, client, "llm_only", replicate, [])
    path = cache_dir / f"llm_only__{evidence.episode_id}__r{replicate}__{key[:16]}.json"
    if path.exists() and not FORCE_RECOMPUTE_LLM:
        return json.loads(path.read_text(encoding="utf-8"))
    available = _available_protocols(evidence)
    payload = {"initial_observation": evidence.initial_observation, "protocol_vocabulary": PROTOCOLS,
               "available_protocols": available, "allowed_reason_codes": sorted(REASON_CODES)}
    assert_agent_payload_safe(payload)
    started = time.perf_counter()
    action, usage, retries = _call_with_bounded_repair(
        client, LLM_ONLY_SYSTEM_PROMPT, payload,
        stable_seed(evidence.episode_id, "llm_only", replicate),
        semantic_validator=lambda candidate: _validate_llm_only_action_semantics(candidate, available),
        audit_context={"episode_id": evidence.episode_id, "agent_type": "llm_only", "replicate": replicate},
    )
    decision = _require_valid_final_action(action, available)
    record = {**decision, "episode_id": evidence.episode_id, "replicate": replicate,
              "agent_type": "llm_only", "tool_calls": 0, "tool_types": [],
              "latency_s": time.perf_counter() - started, "retry_count": retries,
              "token_usage": usage, "prompt_hash": object_hash(LLM_ONLY_SYSTEM_PROMPT),
              "visible_evidence_hash": object_hash(evidence.agent_visible_snapshot()),
              "model_hash": object_hash(client.cache_identity()), "local_model_identifier": client.model,
              "tool_trace_hash": object_hash([]), "cache_key": key}
    atomic_json_dump(record, path)
    return record


def run_tool_agent(evidence, client, replicate=0, allowed_tools=None, trace_path=None):
    allowed_tools = set(ALL_TOOL_NAMES if allowed_tools is None else allowed_tools)
    cache_dir = ARTIFACT_DIR / "llm_cache"; cache_dir.mkdir(parents=True, exist_ok=True)
    key = _llm_cache_key(evidence, client, "tool_agent", replicate, allowed_tools)
    path = cache_dir / f"tool_agent__{evidence.episode_id}__r{replicate}__{key[:16]}.json"
    if path.exists() and not FORCE_RECOMPUTE_LLM:
        cached = json.loads(path.read_text(encoding="utf-8"))
        if trace_path is not None and cached.get("trace"): append_jsonl(trace_path, cached["trace"])
        return cached["decision"], cached.get("trace", [])
    tools = ScientificValidationTools(evidence, allowed_tools)
    available = _available_protocols(evidence)
    history, trace, usage_records, retries_total = [], [], [], 0
    started, final_action = time.perf_counter(), None
    for step in range(MAX_TOOL_CALLS + 1):
        payload = {"initial_observation": evidence.initial_observation,
                   "protocol_vocabulary": PROTOCOLS, "available_protocols": available,
                   "available_tools": tools.available_tool_names(), "allowed_reason_codes": sorted(REASON_CODES),
                   "tool_history": history, "remaining_tool_calls": max(0, MAX_TOOL_CALLS - len(history))}
        if step == MAX_TOOL_CALLS:
            payload["forced_decision"] = "Tool budget exhausted; return a valid final JSON now using one of available_protocols."
        assert_agent_payload_safe(payload)
        action, usage, retries = _call_with_bounded_repair(
            client, SCIENTIFIC_AGENT_SYSTEM_PROMPT, payload,
            stable_seed(evidence.episode_id, "tool_agent", replicate, step),
            semantic_validator=lambda candidate: _validate_tool_agent_action_semantics(
                candidate, available, allowed_tools),
            audit_context={"episode_id": evidence.episode_id, "agent_type": "tool_agent",
                           "replicate": replicate, "step": step},
        )
        retries_total += retries
        if usage is not None: usage_records.append(usage)
        if action["action"] == "final":
            final_action = _require_valid_final_action(action, available)
            trace.append({"episode_id": evidence.episode_id, "replicate": replicate, "step": step,
                          "action": "final", **final_action,
                          "timestamp": datetime.now(timezone.utc).isoformat()})
            break
        if len(history) >= MAX_TOOL_CALLS:
            continue
        name, arguments = action.get("tool_name"), action.get("arguments", {})
        # Already semantically validated above; keep defensive checks.
        if name not in allowed_tools or name not in ALL_TOOL_NAMES or not isinstance(arguments, Mapping):
            raise InvalidModelDecisionError(
                "invalid_tool_request", f"Agent requested invalid or non-allowlisted tool {name!r}.")
        try:
            result = tools.call(name, arguments)
        except Exception as error:
            append_jsonl(
                ARTIFACT_DIR / "invalid_llm_attempts.jsonl",
                [{"episode_id": evidence.episode_id, "agent_type": "tool_agent",
                  "replicate": replicate, "step": step, "stage": "tool_execution",
                  "parsed_action": action, "tool_name": name,
                  "error_type": type(error).__name__, "validation_error": str(error)}],
            )
            raise
        event = {"episode_id": evidence.episode_id, "replicate": replicate, "step": step,
                 "action": "tool", "tool_name": name, "tool_result": result,
                 "timestamp": datetime.now(timezone.utc).isoformat()}
        trace.append(event); history.append({"tool_name": name, "tool_result": result})
    if final_action is None:
        raise InvalidModelDecisionError(
            "invalid_model_action", "Tool agent exhausted its budget without a valid final action.")
    decision = {**final_action, "episode_id": evidence.episode_id, "replicate": replicate,
                "agent_type": "tool_agent", "tool_calls": len(history),
                "tool_types": [item["tool_name"] for item in history],
                "latency_s": time.perf_counter() - started, "retry_count": retries_total,
                "token_usage": usage_records or None, "prompt_hash": object_hash(SCIENTIFIC_AGENT_SYSTEM_PROMPT),
                "visible_evidence_hash": object_hash(evidence.agent_visible_snapshot()),
                "model_hash": object_hash(client.cache_identity()), "local_model_identifier": client.model,
                "tool_trace_hash": object_hash(trace), "cache_key": key}
    atomic_json_dump({"decision": decision, "trace": trace}, path)
    if trace_path is not None: append_jsonl(trace_path, trace)
    return decision, trace


def run_local_llm_preflight(model_path=None, client=None, use_mock=False):
    """Load, generate one synthetic JSON decision, validate it, and release the local model."""
    ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

    report = {
        "model_name": LOCAL_LLM_MODEL_NAME,
        "model_path": None,
        "transformers_version": TRANSFORMERS_VERSION,
        "torch_version": getattr(torch, "__version__", None),
        "gpu": (
            torch.cuda.get_device_name(0)
            if TORCH_STACK_AVAILABLE and torch.cuda.is_available()
            else None
        ),
        "load_success": False,
        "generation_success": False,
        "parsing_success": False,
        "validator_success": False,
        "unload_memory_status": False,
        "mock": bool(use_mock),
    }

    owned_client = client is None

    try:
        if use_mock:
            client = client or MockAgentClient()
            report["model_path"] = "mock"
        else:
            require_transformers_version()

            resolved = validate_local_llm_path(model_path)
            report["model_path"] = str(resolved)

            if not torch.cuda.is_available():
                raise RuntimeError("CUDA is unavailable.")

            client = client or LocalTransformersClient(resolved)

        report.update({
            "load_success": True,
            "dtype": getattr(client, "dtype", "mock"),
            "quantization_mode": getattr(client, "quantization_mode", "none"),
            "parameter_count": getattr(client, "parameter_count", None),
        })

        synthetic = {
            "initial_observation": {
                "biological_task": "synthetic biomass validation preflight",
                "n_training_samples": 20,
                "n_unlabeled_deployment_samples": 5,
                "metadata_fields_available": ["sampling_date_dt"],
                "candidate_protocols": PROTOCOLS,
                "protocol_availability": {
                    p: p == "random_kfold"
                    for p in PROTOCOLS
                },
                "concise_unlabeled_shift_signals": {},
                "deployment_labels_available": False,
            },

            "protocol_vocabulary": PROTOCOLS,

            "available_protocols": [
                "random_kfold"
            ],

            "allowed_reason_codes": sorted(REASON_CODES),

            "instruction": (
                "Return exactly one final JSON object selecting "
                "'random_kfold'. "
                "Use confidence between 0 and 1, "
                "risk_level='low', "
                "reason_codes=[], "
                "and a brief evidence_summary. "
                "Do not invent any reason codes. "
                "Do not return prose outside the JSON object."
            ),
        }

        response = client.complete(
            LLM_ONLY_SYSTEM_PROMPT,
            synthetic,
            MASTER_SEED,
        )

        report["generation_success"] = True

        action = parse_json_action(response["text"])
        report["parsing_success"] = True

        validated = validate_final_action(
            action,
            ["random_kfold"],
        )

        report["validator_success"] = (
            validated["decision_status"] == "valid_decision"
        )

        report["usage"] = response.get("usage")

        report["peak_vram_bytes"] = cuda_memory_snapshot(
            "preflight_peak"
        ).get("peak_allocated_bytes", 0)

    except Exception as error:
        report["error"] = f"{type(error).__name__}: {error}"

        atomic_json_dump(
            report,
            ARTIFACT_DIR / "local_llm_preflight_report.json",
        )

        raise

    finally:
        if client is not None and (owned_client or use_mock):
            unload = client.unload()

            report["unload_report"] = unload

            report["unload_memory_status"] = bool(
                unload.get("substantial_release", True)
            )

        atomic_json_dump(
            report,
            ARTIFACT_DIR / "local_llm_preflight_report.json",
        )

    return report

In [ ]:
def fixed_baseline_decisions(evidence):
    temporal_available = bool(evidence.protocol_results.get("temporal_block_holdout", {}).get("available", False))
    return [
        {"method": "Fixed Random CV", "selected_protocol": "random_kfold", "replicate": 0,
         "confidence": None, "tool_calls": 0, "decision_status": "valid_decision", "invalid_action": False},
        {"method": "Fixed Date CV", "selected_protocol": "date_group_kfold", "replicate": 0,
         "confidence": None, "tool_calls": 0, "decision_status": "valid_decision", "invalid_action": False},
        {"method": "Fixed State CV", "selected_protocol": "state_group_kfold", "replicate": 0,
         "confidence": None, "tool_calls": 0, "decision_status": "valid_decision", "invalid_action": False},
        {"method": "Fixed Date-State CV", "selected_protocol": "date_state_group_kfold", "replicate": 0,
         "confidence": None, "tool_calls": 0, "decision_status": "valid_decision", "invalid_action": False},
        {"method": "Fixed Temporal Holdout", "selected_protocol": "temporal_block_holdout", "replicate": 0,
         "confidence": None, "tool_calls": 0,
         "decision_status": "valid_decision" if temporal_available else "unavailable_baseline",
         "invalid_action": False,
         "invalid_reason": None if temporal_available else "selected_protocol_unavailable"},
    ]


def conservative_heuristic(evidence):
    """Outcome-blind, predeclared rules using only visible shift evidence and protocol feasibility."""
    shift = evidence.shift_summary
    available = {name for name, result in evidence.protocol_results.items() if result["available"]}
    future, unseen = shift.get("future_date_fraction") or 0.0, shift.get("unseen_state_fraction") or 0.0
    embedding = shift.get("embedding_standardized_centroid_distance") or 0.0
    if future >= 0.5 and unseen > 0:
        ordered = ["date_state_group_kfold", "temporal_block_holdout", "state_group_kfold",
                   "date_group_kfold", "random_kfold"]
        codes = ["future_temporal_shift", "unseen_state", "random_cv_optimism_risk"]
    elif future >= 0.5:
        ordered, codes = ["temporal_block_holdout", "date_group_kfold", "random_kfold"], [
            "future_temporal_shift", "temporal_holdout_supported"]
    elif unseen > 0:
        ordered, codes = ["state_group_kfold", "date_state_group_kfold", "date_group_kfold", "random_kfold"], [
            "unseen_state", "state_distribution_shift"]
    elif embedding >= 0.5:
        ordered, codes = ["date_group_kfold", "date_state_group_kfold", "random_kfold"], [
            "embedding_shift_high", "grouped_cv_supported"]
    else:
        ordered, codes = ["random_kfold", "date_group_kfold"], ["embedding_shift_low"]
    selected = next((protocol for protocol in ordered if protocol in available), None)
    status = "valid_decision" if selected is not None else "system_failure"
    return {"method": "Conservative Heuristic", "selected_protocol": selected, "replicate": 0,
            "confidence": None, "tool_calls": 0, "reason_codes": codes,
            "decision_status": status, "invalid_action": selected is None,
            "invalid_reason": None if selected is not None else "no_available_protocol"}


def compute_oracle(protocol_results, deployment_r2, tolerance=ORACLE_TOLERANCE):
    candidates = [(abs(float(result["weighted_r2"]) - float(deployment_r2)), protocol)
                  for protocol, result in protocol_results.items()
                  if result.get("available") and result.get("weighted_r2") is not None]
    if not candidates: raise RuntimeError("No available protocol exists for retrospective oracle evaluation.")
    candidates.sort(key=lambda item: (item[0], item[1]))
    best_gap, best_protocol = candidates[0]
    return {"oracle_protocol": best_protocol, "oracle_gap": float(best_gap),
            "acceptable_protocols": [protocol for gap, protocol in candidates if gap <= best_gap + tolerance],
            "second_best_protocol": candidates[1][1] if len(candidates) > 1 else None,
            "second_best_gap": float(candidates[1][0]) if len(candidates) > 1 else None,
            "best_second_difference": float(candidates[1][0] - best_gap) if len(candidates) > 1 else None}


def decision_to_result_row(episode, decision, evidence, ground_truth, oracle,
                           specialist_mode=PRIMARY_SPECIALIST_MODE, analysis_scope="primary"):
    protocol = decision.get("selected_protocol")
    protocol_result = evidence.protocol_results.get(protocol, {}) if protocol in PROTOCOLS else {}
    selected_available = bool(protocol_result.get("available", False))
    status = decision.get("decision_status", "valid_decision")
    invalid_action = bool(decision.get("invalid_action", False))
    if status == "valid_decision" and protocol in PROTOCOLS and not selected_available:
        if str(decision.get("agent_type", "")).startswith(("llm", "tool")):
            status, invalid_action = "invalid_decision", True
            invalid_reason = "selected_protocol_unavailable"
        else:
            status, invalid_reason = "unavailable_baseline", "selected_protocol_unavailable"
    else:
        invalid_reason = decision.get("invalid_reason")
    coverage = bool(status == "valid_decision" and not invalid_action and selected_available)
    validation_r2 = float(protocol_result["weighted_r2"]) if coverage else np.nan
    deployment_r2 = float(ground_truth.deployment_metrics["Weighted_R2"])
    absolute_gap = abs(validation_r2 - deployment_r2) if coverage else np.nan
    signed_gap = validation_r2 - deployment_r2 if coverage else np.nan
    return {
        "episode_id": episode.episode_id, "episode_type": episode.episode_type,
        "episode_category": canonical_episode_category(episode.episode_type),
        "analysis_scope": analysis_scope, "method": decision["method"],
        "replicate": int(decision.get("replicate", 0)), "selected_protocol": protocol,
        "selected_protocol_available": selected_available, "validation_r2": validation_r2,
        "deployment_r2": deployment_r2, "absolute_gap": absolute_gap, "signed_gap": signed_gap,
        "oracle_protocol": oracle["oracle_protocol"], "oracle_gap": oracle["oracle_gap"],
        "regret_vs_oracle": absolute_gap - oracle["oracle_gap"] if coverage else np.nan,
        "protocol_correct": bool(coverage and protocol in oracle["acceptable_protocols"]),
        "catastrophic_optimism": bool(coverage and signed_gap > CATASTROPHIC_THRESHOLD),
        "confidence": decision.get("confidence"), "tool_calls": int(decision.get("tool_calls", 0)),
        "tool_types": "|".join(decision.get("tool_types", [])), "coverage": coverage,
        "invalid_action": invalid_action, "decision_status": status, "invalid_reason": invalid_reason,
        "abstained": protocol == "ABSTAIN", "retry_count": int(decision.get("retry_count", 0)),
        "decision_latency_s": decision.get("latency_s"), "specialist_mode": specialist_mode,
    }


def canonical_episode_category(episode_type):
    mapping = {"in_distribution_control": "in_distribution", "temporal_future_shift": "temporal",
               "spatial_state_holdout": "spatial", "spatiotemporal_shift": "spatiotemporal"}
    if episode_type not in mapping: raise ValueError(f"Unknown evaluator episode type: {episode_type}")
    return mapping[episode_type]


def summarize_episode_results(results):
    rows = []
    total_episodes = results["episode_id"].nunique()
    for method, group in results.groupby("method", sort=False):
        valid = group[group["coverage"]].copy()
        correct_all = group["protocol_correct"].fillna(False).astype(bool)
        def corr(function):
            if len(valid) < 2 or valid["validation_r2"].nunique() < 2 or valid["deployment_r2"].nunique() < 2:
                return np.nan
            return float(function(valid["validation_r2"], valid["deployment_r2"])[0])
        rows.append({
            "method": method, "n_episodes": total_episodes, "n_decisions": len(group),
            "coverage": float(group["coverage"].mean()),
            "mean_abs_gap": float(valid["absolute_gap"].mean()) if len(valid) else np.nan,
            "std_abs_gap": float(valid["absolute_gap"].std(ddof=1)) if len(valid) > 1 else np.nan,
            "median_abs_gap": float(valid["absolute_gap"].median()) if len(valid) else np.nan,
            "mean_signed_gap": float(valid["signed_gap"].mean()) if len(valid) else np.nan,
            "median_signed_gap": float(valid["signed_gap"].median()) if len(valid) else np.nan,
            "mean_regret": float(valid["regret_vs_oracle"].mean()) if len(valid) else np.nan,
            "protocol_accuracy_all_attempts": float(correct_all.mean()) if len(group) else np.nan,
            "protocol_accuracy_valid_decisions": float(valid["protocol_correct"].mean()) if len(valid) else np.nan,
            "catastrophic_optimism_rate": float(valid["catastrophic_optimism"].mean()) if len(valid) else np.nan,
            "pearson_validation_deployment": corr(pearsonr),
            "spearman_validation_deployment": corr(spearmanr),
            "mean_selected_validation_r2": float(valid["validation_r2"].mean()) if len(valid) else np.nan,
            "mean_deployment_r2": float(valid["deployment_r2"].mean()) if len(valid) else np.nan,
            "mean_confidence": float(pd.to_numeric(group["confidence"], errors="coerce").mean()),
            "mean_tool_calls": float(group["tool_calls"].mean()),
            "invalid_decision_rate": float((group["decision_status"] == "invalid_decision").mean()),
            "system_failure_rate": float((group["decision_status"] == "system_failure").mean()),
            "model_generation_failure_rate": float((group["decision_status"] == "system_failure").mean()),
        })
    return pd.DataFrame(rows)


def summarize_by_method_and_episode_type(results):
    rows = []
    for (method, category), group in results.groupby(["method", "episode_category"], sort=False):
        valid = group[group["coverage"]]
        correct_all = group["protocol_correct"].fillna(False).astype(bool)
        rows.append({"method": method, "episode_category": category,
                     "n": int(group["episode_id"].nunique()), "coverage": float(group["coverage"].mean()),
                     "mean_abs_gap": float(valid["absolute_gap"].mean()) if len(valid) else np.nan,
                     "median_abs_gap": float(valid["absolute_gap"].median()) if len(valid) else np.nan,
                     "signed_bias": float(valid["signed_gap"].mean()) if len(valid) else np.nan,
                     "regret": float(valid["regret_vs_oracle"].mean()) if len(valid) else np.nan,
                     "protocol_accuracy_all_attempts": float(correct_all.mean()) if len(group) else np.nan,
                     "protocol_accuracy_valid_decisions": float(valid["protocol_correct"].mean()) if len(valid) else np.nan,
                     "invalid_decision_rate": float((group["decision_status"] == "invalid_decision").mean()),
                     "system_failure_rate": float((group["decision_status"] == "system_failure").mean()),
                     "catastrophic_optimism": float(valid["catastrophic_optimism"].mean()) if len(valid) else np.nan})
    return pd.DataFrame(rows)


def effective_method_episode_votes(results):
    """One deterministic effective vote per episode/method, preventing replicate overweighting."""
    rows = []
    for (_, _), group in results.groupby(["episode_id", "method"], sort=False):
        counts = Counter(group["selected_protocol"].fillna("SYSTEM_FAILURE"))
        choice = sorted(counts.items(), key=lambda item: (-item[1], item[0]))[0][0]
        selected = group[group["selected_protocol"].fillna("SYSTEM_FAILURE") == choice]
        row = selected.sort_values("replicate").iloc[0].copy()
        numeric_confidence = pd.to_numeric(selected["confidence"], errors="coerce")
        row["confidence"] = float(numeric_confidence.mean()) if numeric_confidence.notna().any() else np.nan
        row["effective_vote_replicates"] = len(group)
        rows.append(row)
    return pd.DataFrame(rows).reset_index(drop=True)


@dataclass(frozen=True)
class BlindDecisionBundle:
    records: Tuple[Dict[str, Any], ...]
    canonical_hash: str
    csv_sha256: str
    frozen_at: str


def blind_audit_record(episode, evidence, decision, selected_available, analysis_scope="primary"):
    record = {
        "episode_id": episode.episode_id, "analysis_scope": analysis_scope,
        "method": decision["method"], "replicate": int(decision.get("replicate", 0)),
        "selected_protocol": decision.get("selected_protocol"), "confidence": decision.get("confidence"),
        "decision_status": decision.get("decision_status", "valid_decision"),
        "invalid_action": bool(decision.get("invalid_action", False)),
        "invalid_reason": decision.get("invalid_reason"),
        "visible_evidence_hash": decision.get("visible_evidence_hash") or object_hash(evidence.agent_visible_snapshot()),
        "prompt_hash": decision.get("prompt_hash"),
        "local_model_identifier": decision.get("local_model_identifier", "deterministic"),
        "tool_trace_hash": decision.get("tool_trace_hash", object_hash([])),
        "selected_protocol_available": bool(selected_available),
        "specialist_mode": decision.get("specialist_mode", PRIMARY_SPECIALIST_MODE),
    }
    assert_agent_payload_safe(record)
    return record


def freeze_blind_decisions(records):
    safe_records = tuple(sorted((dict(record) for record in records),
                                key=lambda row: (row["analysis_scope"], row["episode_id"], row["method"], row["replicate"])))
    for record in safe_records: assert_agent_payload_safe(record)
    canonical_hash = object_hash(safe_records)
    path = ARTIFACT_DIR / "blind_decisions.csv"
    temporary = path.with_suffix(".tmp.csv")
    pd.DataFrame(safe_records).to_csv(temporary, index=False)
    temporary.replace(path)
    frozen_at = datetime.now(timezone.utc).isoformat()
    audit = {"phase": "blind_scientific_decision", "deployment_labels_accessed": False,
             "deployment_score_exists": False, "oracle_exists": False,
             "decision_hash": canonical_hash, "records": safe_records}
    atomic_json_dump(audit, ARTIFACT_DIR / "blind_decision_audit.json")
    manifest = {"decision_hash": canonical_hash, "blind_decisions_csv_sha256": file_sha256(path),
                "record_count": len(safe_records), "frozen_at": frozen_at,
                "notebook_version": NOTEBOOK_VERSION, "code_config_version": CODE_CONFIG_VERSION}
    atomic_json_dump(manifest, ARTIFACT_DIR / "blind_decisions_manifest.json")
    return BlindDecisionBundle(safe_records, canonical_hash, manifest["blind_decisions_csv_sha256"], frozen_at)


def assert_blind_decisions_unchanged(bundle):
    if object_hash(bundle.records) != bundle.canonical_hash:
        raise AssertionError("Frozen blind decision records changed after deployment-label reveal.")
    if file_sha256(ARTIFACT_DIR / "blind_decisions.csv") != bundle.csv_sha256:
        raise AssertionError("blind_decisions.csv changed after deployment-label reveal.")
    return True


def invalid_model_decision(method, agent_type, episode_id, replicate, client, error):
    reason = getattr(error, "invalid_reason", "invalid_model_action")
    return {"action": "final", "selected_protocol": None, "confidence": None,
            "risk_level": None, "reason_codes": [],
            "evidence_summary": f"Invalid model decision: {type(error).__name__}",
            "episode_id": episode_id, "replicate": replicate, "agent_type": agent_type,
            "method": method, "tool_calls": 0, "tool_types": [], "latency_s": None,
            "retry_count": MAX_LLM_RETRIES + 1, "invalid_action": True,
            "decision_status": "invalid_decision", "invalid_reason": reason,
            "token_usage": None, "prompt_hash": object_hash(
                LLM_ONLY_SYSTEM_PROMPT if agent_type == "llm_only" else SCIENTIFIC_AGENT_SYSTEM_PROMPT),
            "visible_evidence_hash": None, "model_hash": object_hash(client.cache_identity()),
            "local_model_identifier": client.model, "tool_trace_hash": object_hash([]),
            "cache_key": None, "error_type": type(error).__name__}


def system_failure_decision(method, agent_type, episode_id, replicate, client, error):
    return {"action": "final", "selected_protocol": None, "confidence": None,
            "risk_level": None, "reason_codes": [],
            "evidence_summary": f"Local model infrastructure failed: {type(error).__name__}",
            "episode_id": episode_id, "replicate": replicate, "agent_type": agent_type,
            "method": method, "tool_calls": 0, "tool_types": [], "latency_s": None,
            "retry_count": 0, "invalid_action": False,
            "decision_status": "system_failure", "invalid_reason": None,
            "system_failure_reason": "model_runtime_failure",
            "token_usage": None, "prompt_hash": object_hash(
                LLM_ONLY_SYSTEM_PROMPT if agent_type == "llm_only" else SCIENTIFIC_AGENT_SYSTEM_PROMPT),
            "visible_evidence_hash": None, "model_hash": object_hash(client.cache_identity()),
            "local_model_identifier": client.model, "tool_trace_hash": object_hash([]),
            "cache_key": None, "error_type": type(error).__name__}


def llm_systemic_failure_report(decisions):
    rows = []
    for agent_type in ("llm_only", "tool_agent"):
        group = [decision for decision in decisions if decision.get("agent_type") == agent_type]
        attempts = len(group)
        valid = sum(decision.get("decision_status") == "valid_decision" for decision in group)
        invalid = sum(decision.get("decision_status") == "invalid_decision" for decision in group)
        failures = sum(decision.get("decision_status") == "system_failure" for decision in group)
        malformed = sum(decision.get("invalid_reason") == "invalid_json" for decision in group)
        unavailable = sum(decision.get("invalid_reason") == "selected_protocol_unavailable" for decision in group)
        rate = valid / attempts if attempts else 0.0
        rows.append({"agent_type": agent_type, "attempts": attempts,
                     "successful_valid_decisions": valid, "invalid_decisions": invalid,
                     "system_failures": failures, "generation_failures": failures,
                     "malformed_or_exhausted_repairs": malformed,
                     "unavailable_protocol_choices": unavailable,
                     "invalid_decision_rate": invalid / attempts if attempts else 0.0,
                     "system_failure_rate": failures / attempts if attempts else 0.0,
                     "success_rate": rate, "minimum_required": MIN_LLM_SUCCESS_RATE,
                     "passes_gate": rate >= MIN_LLM_SUCCESS_RATE})
    return pd.DataFrame(rows)


## 12. Metrics, calibration, and paired statistics

Calibration includes only valid, covered scientific decisions. Invalid JSON, unavailable selections, prohibited abstentions, and system failures are reported separately.

Method comparisons use one effective vote per episode, paired episode-level bootstrap differences, Wilcoxon tests when valid, and Holm correction. Episodes reuse one biological dataset, so interpretation emphasizes effect sizes and consistency across evaluator shift categories rather than treating scenarios as independent datasets.


In [ ]:
def paired_method_table(results, method):
    effective = effective_method_episode_votes(results)
    subset = effective[(effective["method"] == method) & effective["coverage"]]
    return subset[["episode_id", "absolute_gap", "signed_gap", "validation_r2", "deployment_r2"]].copy()


def paired_comparison(results, method_a, method_b, resamples=BOOTSTRAP_RESAMPLES, seed=MASTER_SEED):
    a, b = paired_method_table(results, method_a), paired_method_table(results, method_b)
    paired = a.merge(b, on="episode_id", suffixes=("_a", "_b"), validate="one_to_one")
    differences = paired["absolute_gap_a"].to_numpy() - paired["absolute_gap_b"].to_numpy()
    if not len(differences):
        return {"method_a": method_a, "method_b": method_b, "n_common_episodes": 0,
                "mean_paired_difference": np.nan, "median_paired_difference": np.nan,
                "ci_low": np.nan, "ci_high": np.nan, "wilcoxon_p": np.nan, "effect_size_dz": np.nan}
    rng = np.random.default_rng(seed)
    bootstrap = np.array([differences[rng.integers(0, len(differences), len(differences))].mean()
                          for _ in range(int(resamples))])
    if np.allclose(differences, 0): p_value = 1.0
    elif wilcoxon is None: p_value = np.nan
    else:
        try: p_value = float(wilcoxon(differences, zero_method="wilcox", alternative="two-sided").pvalue)
        except ValueError: p_value = np.nan
    sd = differences.std(ddof=1) if len(differences) > 1 else np.nan
    return {"method_a": method_a, "method_b": method_b, "n_common_episodes": int(len(differences)),
            "mean_paired_difference": float(differences.mean()),
            "median_paired_difference": float(np.median(differences)),
            "ci_low": float(np.quantile(bootstrap, 0.025)), "ci_high": float(np.quantile(bootstrap, 0.975)),
            "wilcoxon_p": p_value,
            "effect_size_dz": float(differences.mean() / sd) if np.isfinite(sd) and sd > 0 else np.nan}


def holm_adjust(p_values):
    p = np.asarray(p_values, float); adjusted = np.full(len(p), np.nan)
    valid = np.flatnonzero(np.isfinite(p))
    order = valid[np.argsort(p[valid])]
    running, m = 0.0, len(order)
    for rank, index in enumerate(order):
        running = max(running, (m - rank) * p[index]); adjusted[index] = min(1.0, running)
    return adjusted


def run_pairwise_statistics(results, resamples=BOOTSTRAP_RESAMPLES):
    planned = [
        ("Tool-Using Local Agent", "Fixed Random CV"),
        ("Tool-Using Local Agent", "Conservative Heuristic"),
        ("Tool-Using Local Agent", "LLM Only"),
        ("Fixed Date CV", "Fixed Random CV"),
        ("Fixed State CV", "Fixed Random CV"),
        ("Fixed Date-State CV", "Fixed Random CV"),
        ("Fixed Temporal Holdout", "Fixed Random CV"),
    ]
    rows = [paired_comparison(results, a, b, resamples, stable_seed("paired", a, b)) for a, b in planned]
    table = pd.DataFrame(rows); table["holm_adjusted_p"] = holm_adjust(table["wilcoxon_p"])
    table["independence_note"] = "Episodes reuse one underlying biological dataset; inference is paired by episode."
    return table


def confidence_calibration(results, n_bins=5):
    rows, summaries = [], []
    for method in ("LLM Only", "Tool-Using Local Agent"):
        all_rows = results[results["method"] == method].copy()
        eligible = all_rows[(all_rows["decision_status"] == "valid_decision") &
                            (~all_rows["invalid_action"]) & all_rows["coverage"] &
                            all_rows["confidence"].notna()].copy()
        total = len(all_rows)
        summaries.append({
            "method": method, "total_decisions": total, "valid_calibration_decisions": len(eligible),
            "valid_decision_calibration_coverage": len(eligible) / total if total else 0.0,
            "invalid_decision_rate": float((all_rows["decision_status"] == "invalid_decision").mean()) if total else np.nan,
            "system_failure_rate": float((all_rows["decision_status"] == "system_failure").mean()) if total else np.nan,
            "model_generation_failure_rate": float((all_rows["decision_status"] == "system_failure").mean()) if total else np.nan,
            "brier_score": np.nan, "ece": np.nan, "ece_small_sample_warning": len(eligible) < 50,
        })
        if not len(eligible): continue
        confidence = eligible["confidence"].astype(float).clip(0, 1)
        correctness = eligible["protocol_correct"].astype(float)
        eligible["bin"] = pd.cut(confidence, bins=np.linspace(0, 1, n_bins + 1), include_lowest=True)
        ece = 0.0
        for bin_name, bin_group in eligible.groupby("bin", observed=False):
            if not len(bin_group): continue
            mean_conf = float(bin_group["confidence"].astype(float).mean())
            accuracy = float(bin_group["protocol_correct"].mean())
            ece += len(bin_group) / len(eligible) * abs(mean_conf - accuracy)
            rows.append({"method": method, "confidence_bin": str(bin_name), "n": len(bin_group),
                         "mean_confidence": mean_conf, "protocol_accuracy": accuracy})
        summaries[-1]["brier_score"] = float(np.mean((confidence - correctness) ** 2))
        summaries[-1]["ece"] = float(ece)
    return pd.DataFrame(rows), pd.DataFrame(summaries)


def catastrophic_threshold_table(results, thresholds=(0.10, 0.15, 0.20)):
    rows = []
    for method, group in results[results["coverage"]].groupby("method"):
        for threshold in thresholds:
            rows.append({"method": method, "threshold": threshold, "n": len(group),
                         "catastrophic_optimism_rate": float((group["signed_gap"] > threshold).mean())})
    return pd.DataFrame(rows)


def risk_coverage_table(results):
    rows = []
    for method in ("LLM Only", "Tool-Using Local Agent"):
        group = results[(results["method"] == method) & (results["decision_status"] == "valid_decision") &
                        (~results["invalid_action"]) & results["coverage"] & results["confidence"].notna()].copy()
        if not len(group): continue
        group = group.sort_values("confidence", ascending=False)
        for retained in range(1, len(group) + 1):
            subset = group.iloc[:retained]
            rows.append({"method": method, "coverage": retained / len(group),
                         "selective_mean_abs_gap": float(subset["absolute_gap"].mean()),
                         "selective_protocol_accuracy": float(subset["protocol_correct"].mean())})
    return pd.DataFrame(rows)

### Saved outputs and optional analyses

These helpers serialize compact research tables and figures, estimate the declared compute budget, and retain optional robustness, ablation, and competition-submission paths without changing the primary workflow.


In [ ]:
def save_paper_outputs(results, summary, calibration_table, risk_coverage, ablation_results=None):
    paper_dir = ARTIFACT_DIR / "paper"; paper_dir.mkdir(parents=True, exist_ok=True)
    desired_order = ["Fixed Random CV", "Fixed Date CV", "Fixed State CV", "Fixed Date-State CV",
                     "Fixed Temporal Holdout", "Conservative Heuristic", "LLM Only",
                     "Tool-Using Local Agent", "Oracle"]
    main = summary[summary["method"].isin(desired_order)].copy()
    main["_order"] = main["method"].map({name: i for i, name in enumerate(desired_order)})
    main = main.sort_values("_order").drop(columns="_order").rename(columns={
        "method": "Method", "mean_abs_gap": "Mean Abs. Gen. Gap",
        "median_abs_gap": "Median Abs. Gen. Gap", "mean_signed_gap": "Signed Bias",
        "mean_regret": "Regret vs Oracle",
        "protocol_accuracy_all_attempts": "Protocol Accuracy (All Attempts)",
        "protocol_accuracy_valid_decisions": "Protocol Accuracy (Valid Decisions)",
        "invalid_decision_rate": "Invalid Decision Rate", "system_failure_rate": "System Failure Rate",
        "catastrophic_optimism_rate": "Catastrophic Optimism", "mean_tool_calls": "Mean Tool Calls",
        "coverage": "Coverage"})
    columns = ["Method", "Coverage", "Mean Abs. Gen. Gap", "Median Abs. Gen. Gap", "Signed Bias",
               "Regret vs Oracle", "Protocol Accuracy (All Attempts)",
               "Protocol Accuracy (Valid Decisions)", "Invalid Decision Rate", "System Failure Rate",
               "Catastrophic Optimism", "Mean Tool Calls"]
    main[columns].to_csv(paper_dir / "table_main.csv", index=False)
    effective = effective_method_episode_votes(results)
    selection = (effective[effective["method"].isin([
        "Fixed Temporal Holdout", "Conservative Heuristic", "LLM Only",
        "Tool-Using Local Agent", "Oracle"])]
        .groupby(["method", "episode_category", "selected_protocol"], as_index=False).size())
    selection.to_csv(paper_dir / "table_protocol_selection.csv", index=False)
    calibration_table.to_csv(paper_dir / "table_calibration.csv", index=False)
    if ablation_results is not None: ablation_results.to_csv(paper_dir / "table_ablation.csv", index=False)
    if plt is None or sns is None:
        warnings.warn("matplotlib/seaborn unavailable; tables were saved and figures skipped.")
        return
    sns.set_theme(style="whitegrid", context="paper")
    valid = effective[effective["coverage"]].copy()
    fig, ax = plt.subplots(figsize=(11, 5))
    sns.boxplot(data=valid, x="method", y="absolute_gap", order=desired_order, ax=ax,
                color="#d8e5f2", showfliers=False)
    sns.stripplot(data=valid, x="method", y="absolute_gap", order=desired_order, ax=ax,
                  color="#244a73", alpha=0.5, size=3)
    ax.set(xlabel="", ylabel="Absolute generalization gap"); ax.tick_params(axis="x", rotation=30)
    fig.tight_layout(); fig.savefig(paper_dir / "figure_gap_by_method.png", dpi=300); plt.close(fig)
    important = ["Fixed Random CV", "Conservative Heuristic", "LLM Only", "Tool-Using Local Agent", "Oracle"]
    scatter = valid[valid["method"].isin(important)]
    if len(scatter):
        grid = sns.FacetGrid(scatter, col="method", col_wrap=3, sharex=True, sharey=True, height=3)
        grid.map_dataframe(sns.scatterplot, x="validation_r2", y="deployment_r2", alpha=0.7)
        values = pd.concat([scatter["validation_r2"], scatter["deployment_r2"]]).dropna()
        low, high = float(values.min()), float(values.max())
        for axis in grid.axes.flat: axis.plot([low, high], [low, high], "--", color="black", linewidth=1)
        grid.set_axis_labels("Selected validation weighted R2", "Deployment weighted R2")
        grid.tight_layout(); grid.savefig(paper_dir / "figure_validation_vs_deployment.png", dpi=300); plt.close(grid.fig)
    if len(selection):
        matrix = selection.pivot_table(index=["method", "episode_category"], columns="selected_protocol",
                                       values="size", fill_value=0)
        fig, ax = plt.subplots(figsize=(10, max(4, 0.35 * len(matrix))))
        sns.heatmap(matrix, annot=True, fmt="g", cmap="Blues", ax=ax)
        ax.set(xlabel="Selected protocol", ylabel="Method / evaluator category")
        fig.tight_layout(); fig.savefig(paper_dir / "figure_protocol_selection.png", dpi=300); plt.close(fig)
    if len(risk_coverage):
        fig, ax = plt.subplots(figsize=(6, 4))
        sns.lineplot(data=risk_coverage, x="coverage", y="selective_mean_abs_gap", hue="method", ax=ax)
        fig.tight_layout(); fig.savefig(paper_dir / "figure_risk_coverage.png", dpi=300); plt.close(fig)


def common_support_results(results, methods=None):
    methods = methods or ["Fixed Random CV", "Fixed Date CV", "Fixed State CV", "Fixed Date-State CV",
                          "Fixed Temporal Holdout", "Conservative Heuristic", "LLM Only",
                          "Tool-Using Local Agent"]
    effective = effective_method_episode_votes(results)
    flags = effective[effective["method"].isin(methods)].pivot_table(
        index="episode_id", columns="method", values="coverage", aggfunc="first", fill_value=False)
    for method in methods:
        if method not in flags: flags[method] = False
    ids = flags.index[flags[methods].all(axis=1)]
    return effective[effective["episode_id"].isin(ids)].copy()


EMBEDDING_EVIDENCE_KEYS = {
    "embedding_centroid_cosine_distance", "embedding_standardized_centroid_distance",
    "embedding_mean_nearest_train_distance", "mmd", "embedding_shift_high", "embedding_shift_low",
}


def make_ablation_evidence(evidence, condition):
    initial = json.loads(canonical_json(evidence.initial_observation))
    shift = json.loads(canonical_json(evidence.shift_summary))
    if condition in {"no_embedding", "metadata_only"}:
        shift = {key: value for key, value in shift.items()
                 if key not in EMBEDDING_EVIDENCE_KEYS and not key.startswith("embedding_") and "mmd" not in key.casefold()}
        signals = initial.get("concise_unlabeled_shift_signals", {})
        initial["concise_unlabeled_shift_signals"] = {
            key: value for key, value in signals.items()
            if key not in EMBEDDING_EVIDENCE_KEYS and not key.startswith("embedding_") and "mmd" not in key.casefold()}
    if condition == "validation_only":
        shift = {}
        initial["concise_unlabeled_shift_signals"] = {}
        initial["metadata_fields_available"] = []
    result = EpisodeEvidence(evidence.episode_id, initial, shift, evidence.protocol_results)
    assert_ablation_evidence_isolation(result, condition)
    return result


def ablation_allowed_tools(condition):
    if condition == "no_embedding": return ALL_TOOL_NAMES - SHIFT_TOOL_NAMES
    if condition == "metadata_only": return METADATA_TOOL_NAMES
    if condition == "validation_only": return set(PROTOCOL_TOOL_NAMES)
    raise ValueError(condition)


def assert_ablation_evidence_isolation(evidence, condition):
    serialized = canonical_json(evidence.agent_visible_snapshot()).casefold()
    if condition in {"no_embedding", "metadata_only"}:
        forbidden = ["embedding_centroid", "embedding_standardized", "embedding_mean_nearest", "mmd"]
        if any(token in serialized for token in forbidden):
            raise AssertionError(f"{condition} still exposes embedding-derived evidence.")
    if condition == "validation_only" and evidence.shift_summary:
        raise AssertionError("validation_only still exposes shift diagnostics.")
    return True


def select_robustness_episodes(episodes, count=ROBUSTNESS_EPISODE_COUNT):
    """Pre-performance deterministic selection: at most one episode from each evaluator category."""
    selected = []
    for category in ("in_distribution", "temporal", "spatial", "spatiotemporal"):
        candidates = sorted((episode for episode in episodes
                             if canonical_episode_category(episode.episode_type) == category),
                            key=lambda episode: episode.episode_id)
        if candidates: selected.append(candidates[0])
        if len(selected) >= count: break
    if len(selected) < count:
        remaining = [episode for episode in sorted(episodes, key=lambda episode: episode.episode_id)
                     if episode.episode_id not in {item.episode_id for item in selected}]
        selected.extend(remaining[:count - len(selected)])
    return selected[:count]


def estimate_compute_budget(episodes, metadata_df, id_to_position, robustness_episodes):
    primary_folds, robust_folds = 0, 0
    for episode in episodes:
        fit_idx = np.array([id_to_position[image_id] for image_id in episode.train_ids])
        fit_metadata = metadata_df.iloc[fit_idx].reset_index(drop=True)
        primary_folds += sum(make_validation_splits(fit_metadata, protocol)["n_folds"] for protocol in PROTOCOLS)
    for episode in robustness_episodes:
        fit_idx = np.array([id_to_position[image_id] for image_id in episode.train_ids])
        fit_metadata = metadata_df.iloc[fit_idx].reset_index(drop=True)
        robust_folds += sum(make_validation_splits(fit_metadata, protocol)["n_folds"] for protocol in PROTOCOLS)
    robust_specialist_fits = robust_folds + len(robustness_episodes)
    report = {
        "episodes": len(episodes), "candidate_protocols": len(PROTOCOLS),
        "primary_protocol_evaluations": len(episodes) * len(PROTOCOLS),
        "primary_validation_folds": primary_folds,
        "approximate_primary_ridge_fits": primary_folds + len(episodes),
        "robustness_episodes": len(robustness_episodes),
        "robustness_protocol_evaluations": len(robustness_episodes) * len(PROTOCOLS),
        "robustness_specialist_fits": robust_specialist_fits,
        "approximate_nested_tree_fits": robust_specialist_fits * 10 * (N_INNER_FOLDS + 1),
        "maximum_primary_llm_generations": len(episodes) * AGENT_REPLICATES * (1 + MAX_TOOL_CALLS + 1),
        "maximum_robustness_llm_generations": len(robustness_episodes) * AGENT_REPLICATES * (1 + MAX_TOOL_CALLS + 1),
        "wall_clock_estimate": None,
        "note": "Counts are deterministic upper bounds/approximations; no wall-clock time is guessed.",
    }
    atomic_json_dump(report, ARTIFACT_DIR / "compute_budget_report.json")
    print("Compute-budget counts (no wall-clock estimate):\n" + json.dumps(report, indent=2))
    return report


def create_competition_submission(train_df, test_df, X_train, data_path=DATA_PATH):
    if not RUN_COMPETITION_SUBMISSION: return None
    if test_df is None: raise ValueError("Competition submission requested but test.csv was not loaded.")
    X_test, _, _ = load_or_extract_dinov2_features(test_df, data_path, "dinov2_test_features")
    model = SpecialistBiomassRegressor(mode=PRIMARY_SPECIALIST_MODE, seed=MASTER_SEED)
    predictions = model.fit_predict(X_train, train_df[TARGET_COLS].to_numpy(float), X_test,
                                    sample_ids=train_df["image_id"], forbidden_ids=test_df["image_id"])
    rows = [{"sample_id": f"{image_id}__{target}", "target": float(value)}
            for image_id, prediction in zip(test_df["image_id"].astype(str), predictions)
            for target, value in zip(TARGET_COLS, prediction)]
    submission = pd.DataFrame(rows); submission.to_csv(ARTIFACT_DIR / "submission.csv", index=False)
    return submission

## 13. Sanity and integrity checks

The final success gate requires all leakage, phase-order, decision-validity, cache, GPU-lifecycle, local-model, ablation-isolation, common-support, and offline-safety checks to pass. LLM-only and tool-agent valid-decision rates must each reach `MIN_LLM_SUCCESS_RATE`; otherwise the run is explicitly marked invalid.


In [ ]:
def _cache_records(directory):
    directory = Path(directory)
    records = []
    if directory.exists():
        for path in directory.glob("*.json"):
            try: records.append(json.loads(path.read_text(encoding="utf-8")))
            except Exception: return []
    return records


def run_integrity_checks(episodes, evidences, grounds, protocol_table, results, llm_decisions,
                         blind_bundle, phase_audit, llm_failure_table, feature_ids_aligned,
                         lifecycle_audit, ablation_evidences=()):
    checks = {}
    checks["train_deployment_ids_disjoint"] = all(
        set(episode.train_ids).isdisjoint(episode.deployment_ids) for episode in episodes)
    checks["deployment_labels_absent_from_episode_evidence"] = all(
        not any(key in set(_recursive_keys(evidence.agent_visible_snapshot()))
                for key in ("deployment_targets", "deployment_metrics", *TARGET_COLS))
        and assert_agent_payload_safe(evidence.agent_visible_snapshot()) for evidence in evidences)
    prompt_text = (COMMON_VALIDATION_SYSTEM_PROMPT + SCIENTIFIC_AGENT_SYSTEM_PROMPT +
                   LLM_ONLY_SYSTEM_PROMPT).casefold()
    checks["deployment_scores_absent_from_prompts"] = all(
        token not in prompt_text for token in ("deployment_r2", "oracle_protocol", "generalization_gap"))
    blind_audit = json.loads((ARTIFACT_DIR / "blind_decision_audit.json").read_text(encoding="utf-8"))
    checks["oracle_absent_during_blind_phase"] = (
        blind_audit.get("oracle_exists") is False and
        all("oracle" not in canonical_json(record).casefold() for record in blind_bundle.records))
    checks["episode_type_absent_from_prompts"] = (
        all(token not in prompt_text for token in (
            "in_distribution", "temporal_future_shift", "spatial_state_holdout", "spatiotemporal_shift"))
        and all("episode_type" not in set(_recursive_keys(evidence.agent_visible_snapshot())) for evidence in evidences))
    checks["decisions_frozen_before_deployment_reveal"] = bool(
        phase_audit.get("blind_frozen_before_reveal") and
        phase_audit.get("blind_frozen_at") <= phase_audit.get("deployment_reveal_started_at"))
    checks["blind_decision_hash_unchanged_after_reveal"] = assert_blind_decisions_unchanged(blind_bundle)
    checks["preprocessing_fit_training_only"] = bool(
        protocol_table["preprocessing_train_only"].fillna(False).all()
        and all(ground.preprocessing_train_only for ground in grounds.values()))
    checks["nested_preprocessing_fold_local"] = bool(
        protocol_table.loc[protocol_table["specialist_mode"] == "nested_stacking",
                           "nested_preprocessing_fold_local"].fillna(False).all())
    agent_results = results[results["method"].isin(["LLM Only", "Tool-Using Local Agent"])]
    unavailable = agent_results[~agent_results["selected_protocol_available"] &
                                agent_results["selected_protocol"].isin(PROTOCOLS)]
    checks["unavailable_protocol_is_invalid"] = bool(
        unavailable.empty or ((unavailable["decision_status"] == "invalid_decision") &
                              (~unavailable["coverage"]) & (~unavailable["protocol_correct"])).all())
    blind_choice = {(record["analysis_scope"], record["episode_id"], record["method"], record["replicate"]):
                    record["selected_protocol"] for record in blind_bundle.records}
    evaluated_choice = {(row.analysis_scope, row.episode_id, row.method, int(row.replicate)): row.selected_protocol
                        for row in results.itertuples() if row.method != "Oracle"}
    checks["no_silent_protocol_fallback"] = all(
        evaluated_choice.get(key) == value for key, value in blind_choice.items() if key in evaluated_choice)
    abstentions = agent_results[agent_results["selected_protocol"] == "ABSTAIN"]
    checks["voluntary_abstention_disabled"] = (not ALLOW_VOLUNTARY_ABSTAIN and
        (abstentions.empty or ((abstentions["decision_status"] == "invalid_decision") &
                               (abstentions["invalid_reason"] == "voluntary_abstain_not_allowed")).all()))
    failures = agent_results[agent_results["decision_status"] == "system_failure"]
    checks["model_failure_separate_from_abstention"] = bool(
        failures.empty or (failures["selected_protocol"].isna() & (~failures["abstained"])).all())
    checks["local_llm_success_rate_gate"] = bool(len(llm_failure_table) == 2 and llm_failure_table["passes_gate"].all())
    checks["dino_feature_ids_aligned"] = bool(feature_ids_aligned)
    deployment_caches = _cache_records(ARTIFACT_DIR / "deployment_predictions")
    deployment_required = {"episode_id", "ordered_train_ids", "ordered_deployment_ids", "train_target_hash",
                           "deployment_target_hash", "train_feature_hash", "deployment_feature_hash",
                           "specialist", "reconciliation", "episode_random_seed", "master_seed",
                           "notebook_version", "device", "code_config_version"}
    checks["deployment_cache_seed_and_config_safe"] = bool(deployment_caches and all(
        deployment_required.issubset(record.get("cache_configuration", {})) and
        object_hash(record["cache_configuration"]) == record.get("cache_configuration_hash")
        for record in deployment_caches))
    protocol_caches = _cache_records(ARTIFACT_DIR / "protocol_cache")
    protocol_required = {"episode_id", "protocol", "ordered_train_ids", "train_feature_hash",
                         "train_target_hash", "specialist", "episode_random_seed", "master_seed",
                         "notebook_version", "device", "code_config_version"}
    checks["protocol_cache_config_safe"] = bool(protocol_caches and all(
        protocol_required.issubset(record.get("cache_configuration", {})) and
        object_hash(record["cache_configuration"]) == record.get("cache_configuration_hash")
        for record in protocol_caches))
    model_hashes = {decision.get("model_hash") for decision in llm_decisions if decision.get("model_hash")}
    checks["identical_qwen_config_for_llm_methods"] = len(model_hashes) == 1
    checks["no_duplicate_episode_ids"] = len({episode.episode_id for episode in episodes}) == len(episodes)
    covered = results[results["coverage"]]
    checks["no_nan_for_valid_covered_metrics"] = bool(covered[
        ["validation_r2", "deployment_r2", "absolute_gap", "signed_gap", "regret_vs_oracle"]].notna().all().all())
    common = common_support_results(results[results["analysis_scope"] == "primary"])
    checks["common_support_reporting_correct"] = bool(
        common.empty or common.groupby("method")["episode_id"].nunique().nunique() == 1)
    checks["ablation_evidence_isolation"] = all(
        assert_ablation_evidence_isolation(evidence, condition) for evidence, condition in ablation_evidences)
    checks["oracle_generated_only_after_label_reveal"] = bool(
        phase_audit.get("oracle_created_after_reveal") and
        phase_audit.get("deployment_reveal_started_at") <= phase_audit.get("oracle_created_at"))
    checks["qwen_loaded_after_dino_unloaded"] = bool(
        lifecycle_audit.get("dino_unloaded_before_preflight") and
        lifecycle_audit.get("dino_unloaded_before_primary_qwen") and
        lifecycle_audit.get("primary_qwen_unloaded_before_reveal"))
    checks["no_required_external_network_or_api"] = bool(
        LOCAL_LLM_BACKEND == "transformers" and
        lifecycle_audit.get("local_files_only") and
        lifecycle_audit.get("external_api_calls", 0) == 0)
    passed = bool(all(checks.values()))
    return {"checks": checks, "ALL_INTEGRITY_CHECKS_PASSED": passed,
            "FINAL_EXPERIMENT_INVALID_DUE_TO_LLM_FAILURE": not checks["local_llm_success_rate_gate"],
            "blind_decision_hash": blind_bundle.canonical_hash}

## 14. Two-phase experiment orchestration

`run_final_experiment()` resolves local inputs, obtains and releases DINO features, preflights and releases Qwen, proposes episodes, reports the compute budget, and computes training-only protocol evidence. It then uses one Qwen instance for all blind decisions, freezes the complete decision artifact, unloads Qwen, and only then accesses deployment targets to produce scores, oracles, statistics, figures, and the run manifest.


In [ ]:
def validate_configuration(expect_final=False):
    if RUN_MODE not in {"smoke", "final"}: raise ValueError("RUN_MODE must be 'smoke' or 'final'.")
    if PRIMARY_SPECIALIST_MODE != "fast": raise ValueError("PRIMARY_SPECIALIST_MODE must be 'fast'.")
    if ROBUSTNESS_SPECIALIST_MODE != "nested_stacking": raise ValueError("Invalid robustness specialist mode.")
    if AGENT_REPLICATES != 1 and not RUN_AGENT_STABILITY_ANALYSIS:
        raise ValueError("Primary deterministic analysis uses AGENT_REPLICATES=1.")
    if ALLOW_VOLUNTARY_ABSTAIN: raise ValueError("Primary experiment requires ALLOW_VOLUNTARY_ABSTAIN=False.")
    if expect_final:
        problems = []
        if RUN_MODE != "final": problems.append("set RUN_MODE='final'")
        if not EXECUTE_LLM_CALLS: problems.append("set EXECUTE_LLM_CALLS=True")
        if LOCAL_LLM_BACKEND != "transformers": problems.append("set LOCAL_LLM_BACKEND='transformers'")
        if BOOTSTRAP_RESAMPLES < 10_000: problems.append("set BOOTSTRAP_RESAMPLES>=10000")
        if problems: raise ValueError("Final-run configuration is incomplete: " + "; ".join(problems))
        if not SKLEARN_AVAILABLE: raise ImportError("Final mode requires scikit-learn.")
        if not SCIPY_AVAILABLE: raise ImportError("Final paired inference requires scipy.")
        if RUN_SPECIALIST_ROBUSTNESS and (lgb is None or xgb is None):
            raise ImportError("Robustness nested stacking requires lightgbm and xgboost.")
        if not KAGGLE_RUNTIME:
            warnings.warn("Final mode requires local data and model resources configured below.")
        qwen_path = validate_local_llm_path(LOCAL_LLM_PATH)
        require_transformers_version()
        # A compatible read-only/writable feature cache may remove the need to attach DINO itself.
        possible_cache_dirs = [Path(FEATURE_CACHE_READ_DIR)] if FEATURE_CACHE_READ_DIR is not None else []
        possible_cache_dirs.append(ARTIFACT_DIR / "cache")
        has_cache_files = any((directory / FEATURE_CACHE_FILENAME).exists() and
                              (directory / "feature_cache_manifest.json").exists()
                              for directory in possible_cache_dirs)
        dino_path = None if has_cache_files else resolved_dino_model_path()
        assert_final_offline_safety(qwen_path, dino_path)
    return True


def collect_environment_info(feature_manifest=None):
    gpu = torch.cuda.get_device_name(0) if TORCH_STACK_AVAILABLE and torch.cuda.is_available() else None
    packages = {name: get_package_version(name) for name in (
        "numpy", "pandas", "scipy", "scikit-learn", "torch", "torchvision", "transformers",
        "lightgbm", "xgboost", "Pillow", "psutil", "accelerate", "bitsandbytes")}
    return {"python_version": platform.python_version(), "operating_system": platform.platform(),
            "cpu_model": get_cpu_model(), "gpu": gpu, "cuda_version": getattr(torch.version, "cuda", None)
            if TORCH_STACK_AVAILABLE else None, "packages": packages, "feature_manifest": feature_manifest}


def protocol_table_rows(evidence, analysis_scope, specialist_mode):
    rows = []
    for protocol, result in evidence.protocol_results.items():
        row = {"episode_id": evidence.episode_id, "analysis_scope": analysis_scope,
               "specialist_mode": specialist_mode, **result}
        for key in ("fold_scores", "fold_sizes", "per_target_metrics"):
            row[key] = canonical_json(row.get(key, []))
        rows.append(row)
    return rows


def run_one_blind_method_set(episode, evidence, client, analysis_scope, specialist_mode,
                             trace_path, include_fixed=True):
    decisions, traces = [], []
    deterministic = fixed_baseline_decisions(evidence) + [conservative_heuristic(evidence)] if include_fixed else []
    for decision in deterministic:
        decision = {**decision, "episode_id": episode.episode_id, "analysis_scope": analysis_scope,
                    "specialist_mode": specialist_mode,
                    "visible_evidence_hash": object_hash(evidence.agent_visible_snapshot()),
                    "prompt_hash": None, "local_model_identifier": "deterministic", "tool_trace_hash": object_hash([])}
        decisions.append(decision)
    for replicate in range(AGENT_REPLICATES):
        try:
            llm = run_llm_only_selector(evidence, client, replicate)
            llm["method"] = "LLM Only"
        except InvalidModelDecisionError as error:
            llm = invalid_model_decision("LLM Only", "llm_only", episode.episode_id, replicate, client, error)
            llm["visible_evidence_hash"] = object_hash(evidence.agent_visible_snapshot())
        except Exception as error:
            llm = system_failure_decision("LLM Only", "llm_only", episode.episode_id, replicate, client, error)
            llm["visible_evidence_hash"] = object_hash(evidence.agent_visible_snapshot())
        try:
            tool, trace = run_tool_agent(evidence, client, replicate, trace_path=trace_path)
            tool["method"] = "Tool-Using Local Agent"; traces.extend(trace)
        except InvalidModelDecisionError as error:
            tool = invalid_model_decision(
                "Tool-Using Local Agent", "tool_agent", episode.episode_id, replicate, client, error)
            tool["visible_evidence_hash"] = object_hash(evidence.agent_visible_snapshot())
            failure_trace = [{"episode_id": episode.episode_id, "replicate": replicate, "step": 0,
                              "action": "invalid_decision", "invalid_reason": error.invalid_reason,
                              "timestamp": datetime.now(timezone.utc).isoformat()}]
            tool["tool_trace_hash"] = object_hash(failure_trace); append_jsonl(trace_path, failure_trace)
        except Exception as error:
            tool = system_failure_decision(
                "Tool-Using Local Agent", "tool_agent", episode.episode_id, replicate, client, error)
            tool["visible_evidence_hash"] = object_hash(evidence.agent_visible_snapshot())
            failure_trace = [{"episode_id": episode.episode_id, "replicate": replicate, "step": 0,
                              "action": "system_failure", "error_type": type(error).__name__,
                              "timestamp": datetime.now(timezone.utc).isoformat()}]
            tool["tool_trace_hash"] = object_hash(failure_trace); append_jsonl(trace_path, failure_trace)
        for decision in (llm, tool):
            decision["analysis_scope"] = analysis_scope
            decision["specialist_mode"] = specialist_mode
            decisions.append(decision)
    return decisions, traces


def run_final_experiment():
    """Offline two-phase benchmark invoked by the final notebook cell."""
    validate_configuration(expect_final=True)
    started = time.perf_counter(); runtime = RuntimeTracker(); seed_everything(MASTER_SEED)
    ARTIFACT_DIR.mkdir(parents=True, exist_ok=True); save_prompts()
    for name in ("cache", "protocol_cache", "llm_cache", "deployment_predictions", "paper"):
        (ARTIFACT_DIR / name).mkdir(parents=True, exist_ok=True)
    lifecycle_audit = {"local_files_only": True, "external_api_calls": 0,
                       "dino_unloaded_before_preflight": False,
                       "dino_unloaded_before_primary_qwen": False,
                       "primary_qwen_unloaded_before_reveal": False}
    phase_audit = {"phase": "setup", "oracle_created_after_reveal": False}

    # Data are loaded once, but Phase I receives deployment metadata only; targets are indexed only in Phase II.
    resolved_data = resolve_data_path(DATA_PATH)
    (loaded, elapsed) = timed_call(load_data, resolved_data, RUN_COMPETITION_SUBMISSION)
    train_df, test_df, discovery = loaded
    metadata_df = train_df.drop(columns=TARGET_COLS).copy()
    runtime.add("Data loading and metadata validation", elapsed, len(train_df), "Data")

    # Frozen DINO representation. load_or_extract always releases DINO before returning.
    (feature_output, elapsed) = timed_call(load_or_extract_dinov2_features, metadata_df, resolved_data)
    X, feature_manifest, feature_cache_hit = feature_output
    feature_ids_aligned = (X.shape[0] == len(metadata_df) and metadata_df["image_id"].is_unique and
                           feature_manifest.get("image_ids_hash") == object_hash(
                               sorted(metadata_df["image_id"].astype(str).tolist())))
    if not feature_ids_aligned: raise AssertionError("DINO feature cache is not ID-aligned.")
    lifecycle_audit["dino_unloaded_before_preflight"] = bool(
        feature_cache_hit or feature_manifest.get("gpu_lifecycle", {}).get("after_unload", {}).get("substantial_release"))
    runtime.add("DINOv2 feature cache/extraction and unload", elapsed, len(metadata_df), "Feature extraction",
                cache_hit=feature_cache_hit)

    # Required real local-Qwen preflight occurs before expensive protocol training, then unloads Qwen.
    preflight, elapsed = timed_call(run_local_llm_preflight, LOCAL_LLM_PATH)
    if not preflight.get("unload_memory_status"): raise RuntimeError("Local Qwen preflight did not release GPU memory.")
    lifecycle_audit["dino_unloaded_before_primary_qwen"] = lifecycle_audit["dino_unloaded_before_preflight"]
    runtime.add("Local Qwen preflight and unload", elapsed, category="Preflight")

    # Outcome-blind episode proposal and pre-performance robustness selection.
    (episode_output, elapsed) = timed_call(generate_deployment_episodes, metadata_df)
    episodes, registry = episode_output; save_episode_registry(episodes, registry)
    id_to_position = {str(image_id): index for index, image_id in enumerate(metadata_df["image_id"].astype(str))}
    robustness_episodes = select_robustness_episodes(episodes) if RUN_SPECIALIST_ROBUSTNESS else []
    compute_budget = estimate_compute_budget(episodes, metadata_df, id_to_position, robustness_episodes)
    runtime.add("Deployment episode generation", elapsed, len(episodes), "Episodes")

    # ------------------------------------------------------------------
    # PHASE I: BLIND SCIENTIFIC DECISIONS (NO deployment target access)
    # ------------------------------------------------------------------
    phase_audit["phase"] = "blind_scientific_decision"
    primary_evidences, robustness_evidences, protocol_rows = {}, {}, []
    frame_registry = {}
    phase_one_started = time.perf_counter()
    for episode in episodes:
        fit_idx = np.array([id_to_position[image_id] for image_id in episode.train_ids], int)
        deploy_idx = np.array([id_to_position[image_id] for image_id in episode.deployment_ids], int)
        fit_metadata = metadata_df.iloc[fit_idx].reset_index(drop=True)
        deploy_metadata = metadata_df.iloc[deploy_idx].reset_index(drop=True)
        # Training labels are permitted; the deployment target columns are never indexed here.
        y_fit = train_df.iloc[fit_idx][TARGET_COLS].to_numpy(float)
        protocols = evaluate_protocols_for_episode(
            episode, fit_metadata, X[fit_idx], y_fit, PRIMARY_SPECIALIST_MODE)
        shift = compute_shift_summary(fit_metadata, deploy_metadata, X[fit_idx], X[deploy_idx])
        evidence = EpisodeEvidence(episode.episode_id,
            build_initial_observation(fit_metadata, deploy_metadata, shift, protocols), shift, protocols)
        assert_agent_payload_safe(evidence.agent_visible_snapshot())
        primary_evidences[episode.episode_id] = evidence
        frame_registry[episode.episode_id] = {"fit_idx": fit_idx, "deploy_idx": deploy_idx,
                                               "fit_metadata": fit_metadata, "deploy_metadata": deploy_metadata,
                                               "y_fit": y_fit}
        protocol_rows.extend(protocol_table_rows(evidence, "primary", PRIMARY_SPECIALIST_MODE))
    for episode in robustness_episodes:
        frame = frame_registry[episode.episode_id]
        protocols = evaluate_protocols_for_episode(
            episode, frame["fit_metadata"], X[frame["fit_idx"]], frame["y_fit"], ROBUSTNESS_SPECIALIST_MODE)
        shift = primary_evidences[episode.episode_id].shift_summary
        evidence = EpisodeEvidence(episode.episode_id,
            build_initial_observation(frame["fit_metadata"], frame["deploy_metadata"], shift, protocols), shift, protocols)
        robustness_evidences[episode.episode_id] = evidence
        protocol_rows.extend(protocol_table_rows(evidence, "robustness", ROBUSTNESS_SPECIALIST_MODE))
    protocol_table = pd.DataFrame(protocol_rows)
    protocol_table.to_csv(ARTIFACT_DIR / "protocol_evaluations.csv", index=False)
    runtime.add("Blind protocol evidence", time.perf_counter() - phase_one_started, len(protocol_rows), "Validation")

    # DINO has no live reference. Load one local Qwen instance for all blind decisions.
    client = create_llm_client(); client_identity = client.cache_identity()
    blind_decisions, blind_records, ablation_evidence_pairs = [], [], []
    trace_path = ARTIFACT_DIR / "agent_tool_traces.jsonl"
    for path in (trace_path, ARTIFACT_DIR / "llm_only_decisions.jsonl", ARTIFACT_DIR / "agent_decisions.jsonl",
                 ARTIFACT_DIR / "invalid_llm_attempts.jsonl"):
        path.write_text("", encoding="utf-8")
    episode_lookup = {episode.episode_id: episode for episode in episodes}
    for episode in episodes:
        evidence = primary_evidences[episode.episode_id]
        decisions, _ = run_one_blind_method_set(
            episode, evidence, client, "primary", PRIMARY_SPECIALIST_MODE, trace_path)
        for decision in decisions:
            selected_available = bool(evidence.protocol_results.get(decision.get("selected_protocol"), {}).get("available", False))
            blind_records.append(blind_audit_record(episode, evidence, decision, selected_available, "primary"))
            blind_decisions.append(decision)
    for episode in robustness_episodes:
        evidence = robustness_evidences[episode.episode_id]
        decisions, _ = run_one_blind_method_set(
            episode, evidence, client, "robustness", ROBUSTNESS_SPECIALIST_MODE, trace_path)
        for decision in decisions:
            selected_available = bool(evidence.protocol_results.get(decision.get("selected_protocol"), {}).get("available", False))
            blind_records.append(blind_audit_record(episode, evidence, decision, selected_available, "robustness"))
            blind_decisions.append(decision)
    if RUN_ABLATIONS:
        for condition in ("no_embedding", "metadata_only", "validation_only"):
            scope = f"ablation_{condition}"
            for episode in episodes:
                evidence = make_ablation_evidence(primary_evidences[episode.episode_id], condition)
                ablation_evidence_pairs.append((evidence, condition))
                try:
                    decision, _ = run_tool_agent(
                        evidence, client, 0, allowed_tools=ablation_allowed_tools(condition),
                        trace_path=ARTIFACT_DIR / f"{scope}_traces.jsonl")
                    decision["method"] = f"Tool Agent: {condition}"
                except InvalidModelDecisionError as error:
                    decision = invalid_model_decision(
                        f"Tool Agent: {condition}", "tool_agent_ablation", episode.episode_id, 0, client, error)
                    decision["visible_evidence_hash"] = object_hash(evidence.agent_visible_snapshot())
                except Exception as error:
                    decision = system_failure_decision(
                        f"Tool Agent: {condition}", "tool_agent_ablation", episode.episode_id, 0, client, error)
                    decision["visible_evidence_hash"] = object_hash(evidence.agent_visible_snapshot())
                decision.update({"analysis_scope": scope, "specialist_mode": PRIMARY_SPECIALIST_MODE})
                selected_available = bool(evidence.protocol_results.get(decision.get("selected_protocol"), {}).get("available", False))
                blind_records.append(blind_audit_record(episode, evidence, decision, selected_available, scope))
                blind_decisions.append(decision)

    # Serialize/freeze all blind choices before any deployment target or oracle is created.
    blind_bundle = freeze_blind_decisions(blind_records)
    phase_audit.update({"blind_frozen_before_reveal": True, "blind_frozen_at": blind_bundle.frozen_at,
                        "deployment_labels_accessed_in_phase_i": False, "oracle_exists_in_phase_i": False})
    llm_primary = [decision for decision in blind_decisions
                   if decision.get("analysis_scope") == "primary" and
                   decision.get("agent_type") in {"llm_only", "tool_agent"}]
    llm_failure_table = llm_systemic_failure_report(llm_primary)
    llm_failure_table.to_csv(ARTIFACT_DIR / "local_llm_failure_report.csv", index=False)
    llm_only_records = [decision for decision in blind_decisions if decision.get("agent_type") == "llm_only"]
    tool_records = [decision for decision in blind_decisions if decision.get("agent_type") == "tool_agent"]
    append_jsonl(ARTIFACT_DIR / "llm_only_decisions.jsonl", llm_only_records)
    append_jsonl(ARTIFACT_DIR / "agent_decisions.jsonl", tool_records)
    pd.DataFrame([decision for decision in blind_decisions if decision.get("local_model_identifier") == "deterministic"]).to_csv(
        ARTIFACT_DIR / "baseline_decisions.csv", index=False)
    qwen_unload = client.unload(); del client
    lifecycle_audit["primary_qwen_unloaded_before_reveal"] = bool(qwen_unload.get("substantial_release"))
    if not lifecycle_audit["primary_qwen_unloaded_before_reveal"]:
        raise RuntimeError("Primary Qwen GPU memory was not released before deployment-label reveal.")

    # ------------------------------------------------------------------
    # PHASE II: DEPLOYMENT LABEL REVEAL AND RETROSPECTIVE EVALUATION
    # ------------------------------------------------------------------
    phase_audit["phase"] = "deployment_label_reveal"
    phase_audit["deployment_reveal_started_at"] = datetime.now(timezone.utc).isoformat()
    primary_grounds, robustness_grounds, ground_rows = {}, {}, []
    primary_rows, robustness_rows, ablation_rows = [], [], []
    decision_lookup = {(decision["analysis_scope"], decision["episode_id"], decision["method"],
                        int(decision.get("replicate", 0))): decision for decision in blind_decisions}
    oracle_created_at = None
    for episode in episodes:
        frame = frame_registry[episode.episode_id]
        # First and only access to this episode's deployment biomass labels.
        y_deploy = train_df.iloc[frame["deploy_idx"]][TARGET_COLS].to_numpy(float)
        ground = compute_deployment_ground_truth(
            episode, frame["fit_metadata"], frame["deploy_metadata"],
            X[frame["fit_idx"]], X[frame["deploy_idx"]], frame["y_fit"], y_deploy,
            PRIMARY_SPECIALIST_MODE)
        primary_grounds[episode.episode_id] = ground
        ground_rows.append({"episode_id": episode.episode_id, "analysis_scope": "primary",
                            **ground.deployment_metrics, "preprocessing_train_only": ground.preprocessing_train_only})
        evidence = primary_evidences[episode.episode_id]
        oracle = compute_oracle(evidence.protocol_results, ground.deployment_metrics["Weighted_R2"])
        oracle_created_at = oracle_created_at or datetime.now(timezone.utc).isoformat()
        episode_decisions = [decision for decision in blind_decisions
                             if decision["analysis_scope"] == "primary" and decision["episode_id"] == episode.episode_id]
        for decision in episode_decisions:
            primary_rows.append(decision_to_result_row(
                episode, decision, evidence, ground, oracle, PRIMARY_SPECIALIST_MODE, "primary"))
        oracle_decision = {"method": "Oracle", "selected_protocol": oracle["oracle_protocol"], "replicate": 0,
                           "confidence": None, "tool_calls": 0, "decision_status": "valid_decision",
                           "invalid_action": False}
        primary_rows.append(decision_to_result_row(
            episode, oracle_decision, evidence, ground, oracle, PRIMARY_SPECIALIST_MODE, "primary"))
    for episode in robustness_episodes:
        frame = frame_registry[episode.episode_id]
        y_deploy = train_df.iloc[frame["deploy_idx"]][TARGET_COLS].to_numpy(float)
        ground = compute_deployment_ground_truth(
            episode, frame["fit_metadata"], frame["deploy_metadata"],
            X[frame["fit_idx"]], X[frame["deploy_idx"]], frame["y_fit"], y_deploy,
            ROBUSTNESS_SPECIALIST_MODE)
        robustness_grounds[episode.episode_id] = ground
        ground_rows.append({"episode_id": episode.episode_id, "analysis_scope": "robustness",
                            **ground.deployment_metrics, "preprocessing_train_only": ground.preprocessing_train_only})
        evidence = robustness_evidences[episode.episode_id]
        oracle = compute_oracle(evidence.protocol_results, ground.deployment_metrics["Weighted_R2"])
        for decision in [item for item in blind_decisions
                         if item["analysis_scope"] == "robustness" and item["episode_id"] == episode.episode_id]:
            robustness_rows.append(decision_to_result_row(
                episode, decision, evidence, ground, oracle, ROBUSTNESS_SPECIALIST_MODE, "robustness"))
        oracle_decision = {"method": "Oracle", "selected_protocol": oracle["oracle_protocol"], "replicate": 0,
                           "confidence": None, "tool_calls": 0, "decision_status": "valid_decision",
                           "invalid_action": False}
        robustness_rows.append(decision_to_result_row(
            episode, oracle_decision, evidence, ground, oracle, ROBUSTNESS_SPECIALIST_MODE, "robustness"))
    if RUN_ABLATIONS:
        evidence_lookup = {(evidence.episode_id, condition): evidence for evidence, condition in ablation_evidence_pairs}
        for decision in [item for item in blind_decisions if item["analysis_scope"].startswith("ablation_")]:
            condition = decision["analysis_scope"].replace("ablation_", "", 1)
            episode = episode_lookup[decision["episode_id"]]
            evidence = evidence_lookup[(episode.episode_id, condition)]
            ground = primary_grounds[episode.episode_id]
            oracle = compute_oracle(evidence.protocol_results, ground.deployment_metrics["Weighted_R2"])
            ablation_rows.append(decision_to_result_row(
                episode, decision, evidence, ground, oracle, PRIMARY_SPECIALIST_MODE, decision["analysis_scope"]))

    phase_audit.update({"oracle_created_after_reveal": True,
                        "oracle_created_at": oracle_created_at or datetime.now(timezone.utc).isoformat()})
    assert_blind_decisions_unchanged(blind_bundle)
    atomic_json_dump(phase_audit, ARTIFACT_DIR / "phase_order_audit.json")
    pd.DataFrame(ground_rows).to_csv(ARTIFACT_DIR / "deployment_ground_truth_metrics.csv", index=False)

    results = pd.DataFrame(primary_rows)
    robustness_results = pd.DataFrame(robustness_rows)
    ablation_results = pd.DataFrame(ablation_rows)
    results.to_csv(ARTIFACT_DIR / "episode_results.csv", index=False)
    robustness_results.to_csv(ARTIFACT_DIR / "specialist_robustness_results.csv", index=False)
    ablation_results.to_csv(ARTIFACT_DIR / "ablation_results.csv", index=False)
    common = common_support_results(results); common.to_csv(ARTIFACT_DIR / "common_support_episode_results.csv", index=False)
    summary = summarize_episode_results(results); summary.to_csv(ARTIFACT_DIR / "summary_by_method.csv", index=False)
    category_summary = summarize_by_method_and_episode_type(results)
    category_summary["independence_note"] = "Scenarios reuse a shared underlying biological dataset and are not independent datasets."
    category_summary.to_csv(ARTIFACT_DIR / "summary_by_method_and_episode_type.csv", index=False)
    pairwise = run_pairwise_statistics(results); pairwise.to_csv(ARTIFACT_DIR / "pairwise_statistics.csv", index=False)
    calibration, calibration_summary = confidence_calibration(results)
    calibration.to_csv(ARTIFACT_DIR / "confidence_calibration.csv", index=False)
    calibration_summary.to_csv(ARTIFACT_DIR / "confidence_calibration_summary.csv", index=False)
    catastrophic_threshold_table(results).to_csv(ARTIFACT_DIR / "catastrophic_optimism_thresholds.csv", index=False)
    risk_coverage = risk_coverage_table(results); risk_coverage.to_csv(ARTIFACT_DIR / "risk_coverage.csv", index=False)
    effective_method_episode_votes(results).to_csv(ARTIFACT_DIR / "effective_method_episode_votes.csv", index=False)
    submission = create_competition_submission(train_df, test_df, X, resolved_data)
    save_paper_outputs(results, summary, calibration, risk_coverage,
                       ablation_results if len(ablation_results) else None)

    all_evidences = list(primary_evidences.values()) + list(robustness_evidences.values())
    all_grounds = {**{f"primary::{key}": value for key, value in primary_grounds.items()},
                   **{f"robustness::{key}": value for key, value in robustness_grounds.items()}}
    all_results = pd.concat([results, robustness_results, ablation_results], ignore_index=True)
    integrity = run_integrity_checks(
        episodes, all_evidences, all_grounds, protocol_table, all_results, llm_primary,
        blind_bundle, phase_audit, llm_failure_table, feature_ids_aligned,
        lifecycle_audit, ablation_evidence_pairs)
    atomic_json_dump(integrity, ARTIFACT_DIR / "integrity_report.json")
    total_runtime = time.perf_counter() - started
    runtime.add("Complete offline agentic validation experiment", total_runtime, len(episodes), "Overall")
    runtime_table = runtime.to_dataframe(); runtime_table.to_csv(ARTIFACT_DIR / "runtime_summary.csv", index=False)
    configuration = {
        "RUN_MODE": RUN_MODE, "MASTER_SEED": MASTER_SEED, "DATA_PATH": str(DATA_PATH),
        "ARTIFACT_DIR": str(ARTIFACT_DIR), "PRIMARY_SPECIALIST_MODE": PRIMARY_SPECIALIST_MODE,
        "ROBUSTNESS_SPECIALIST_MODE": ROBUSTNESS_SPECIALIST_MODE,
        "ROBUSTNESS_EPISODE_COUNT": ROBUSTNESS_EPISODE_COUNT, "N_OUTER_FOLDS": N_OUTER_FOLDS,
        "N_INNER_FOLDS": N_INNER_FOLDS, "USE_RECONCILIATION": USE_RECONCILIATION,
        "AGENT_REPLICATES": AGENT_REPLICATES, "ALLOW_VOLUNTARY_ABSTAIN": ALLOW_VOLUNTARY_ABSTAIN,
        "MAX_TOOL_CALLS": MAX_TOOL_CALLS, "MIN_LLM_SUCCESS_RATE": MIN_LLM_SUCCESS_RATE,
        "BOOTSTRAP_RESAMPLES": BOOTSTRAP_RESAMPLES, "ORACLE_TOLERANCE": ORACLE_TOLERANCE,
        "CATASTROPHIC_THRESHOLD": CATASTROPHIC_THRESHOLD, "code_config_version": CODE_CONFIG_VERSION}
    manifest = {
        "timestamp": datetime.now(timezone.utc).isoformat(), "notebook_version": NOTEBOOK_VERSION,
        "environment": collect_environment_info(feature_manifest), "offline_kaggle_execution": KAGGLE_RUNTIME,
        "dino_feature_config": feature_configuration(), "feature_cache_hash": feature_manifest.get("cache_sha256"),
        "primary_specialist": specialist_configuration(PRIMARY_SPECIALIST_MODE),
        "robustness_specialist": specialist_configuration(ROBUSTNESS_SPECIALIST_MODE),
        "robustness_episode_ids_selected_before_performance": [e.episode_id for e in robustness_episodes],
        "episode_ids": [e.episode_id for e in episodes], "episode_seeds": {e.episode_id: e.random_seed for e in episodes},
        "local_llm": client_identity, "local_llm_preflight": preflight,
        "agent_prompt_version": AGENT_PROMPT_VERSION,
        "prompt_fairness": {"same_checkpoint": True, "same_model_configuration": True,
                            "same_initial_visible_evidence": True, "same_protocol_vocabulary": True,
                            "shared_system_task_hash": object_hash(COMMON_VALIDATION_SYSTEM_PROMPT),
                            "only_difference": "tool-agent has allowlisted local tools"},
        "blind_decision_hash": blind_bundle.canonical_hash, "blind_decisions_csv_sha256": blind_bundle.csv_sha256,
        "compute_budget": compute_budget, "lifecycle_audit": lifecycle_audit,
        "configuration": configuration, "configuration_hash": object_hash(configuration),
        "total_runtime_s": total_runtime, "integrity_passed": integrity["ALL_INTEGRITY_CHECKS_PASSED"]}
    atomic_json_dump(manifest, ARTIFACT_DIR / "run_manifest.json")

    if not llm_failure_table["passes_gate"].all():
        print("FINAL_EXPERIMENT_INVALID_DUE_TO_LLM_FAILURE")
        raise RuntimeError("Local-Qwen valid-decision rate fell below MIN_LLM_SUCCESS_RATE.")
    if not integrity["ALL_INTEGRITY_CHECKS_PASSED"]:
        failed = [name for name, passed in integrity["checks"].items() if not passed]
        raise AssertionError("Mandatory integrity checks failed: " + ", ".join(failed))
    print("=" * 64)
    print("OFFLINE AGENTIC VALIDATION EXPERIMENT COMPLETE")
    print("=" * 64)
    print(f"Episodes accepted: {len(episodes)}; rejected: {int((~registry['accepted']).sum())}")
    print(summary[["method", "coverage", "mean_abs_gap", "mean_signed_gap", "mean_regret"]]
          .round(4).to_string(index=False))
    print("\nALL_INTEGRITY_CHECKS_PASSED = True")
    print(f"Artifacts: {ARTIFACT_DIR}")
    return {"episode_results": results, "robustness_results": robustness_results,
            "summary_by_method": summary, "summary_by_category": category_summary,
            "pairwise_statistics": pairwise, "calibration": calibration,
            "blind_decisions": blind_bundle, "integrity": integrity, "manifest": manifest,
            "runtime": runtime_table, "competition_submission": submission}

## 15. Synthetic smoke validation

The smoke suite requires neither CSIRO files nor model weights. It exercises data/protocol/specialist plumbing, invalid-decision handling, calibration filtering, blind-decision hashing, phase ordering, evaluator-label isolation, ablation isolation, cache identity, path/version failures, category summaries, fold-local preprocessing, strict JSON parsing, and model-unload lifecycle logic.


In [ ]:
def run_smoke_test(raise_on_failure=True):
    global ARTIFACT_DIR
    original_artifact_dir = ARTIFACT_DIR
    reports = []
    def record(name, function):
        try:
            value = function()
            passed = bool(value) if isinstance(value, (bool, np.bool_)) else True
            reports.append({"check": name, "passed": passed, "detail": "ok" if passed else "returned False"})
            return value
        except Exception as error:
            reports.append({"check": name, "passed": False,
                            "detail": f"{type(error).__name__}: {str(error)[:240]}"})
            return None
    def raises(expected, function):
        try: function()
        except expected: return True
        return False
    with tempfile.TemporaryDirectory(prefix="agenticls_v2_smoke_") as temporary:
        ARTIFACT_DIR = Path(temporary) / "artifacts"; ARTIFACT_DIR.mkdir(parents=True)
        try:
            rng = np.random.default_rng(MASTER_SEED)
            n_samples, n_features = 200, 18
            dates = pd.date_range("2024-01-01", periods=40, freq="7D")
            X = rng.normal(size=(n_samples, n_features))
            frame = pd.DataFrame({
                "image_id": [f"SYN_{i:04d}" for i in range(n_samples)],
                "image_path": [f"synthetic/{i}.jpg" for i in range(n_samples)],
                "Sampling_Date": [dates[i % 40].date().isoformat() for i in range(n_samples)],
                "State": [["north", "south", "east", "west"][i % 4] for i in range(n_samples)],
                "Species": [["ryegrass", "clover"][i % 2] for i in range(n_samples)],
            })
            frame["sampling_date_dt"] = pd.to_datetime(frame["Sampling_Date"])
            frame["state_norm"] = frame["State"].str.casefold()
            green = np.clip(18 + 4 * X[:, 0] + rng.normal(0, 1, n_samples), 0, None)
            clover = np.clip(6 + 2 * X[:, 1] + rng.normal(0, .7, n_samples), 0, None)
            dead = np.clip(9 + 3 * X[:, 2] + rng.normal(0, 1, n_samples), 0, None)
            frame["Dry_Green_g"], frame["Dry_Dead_g"], frame["Dry_Clover_g"] = green, dead, clover
            frame["GDM_g"], frame["Dry_Total_g"] = green + clover, green + clover + dead
            metadata_frame, y = frame.drop(columns=TARGET_COLS), frame[TARGET_COLS].to_numpy(float)
            record("synthetic_metadata", lambda: frame["image_id"].is_unique and frame["sampling_date_dt"].notna().all())
            episode_output = record("episode_generation", lambda: generate_deployment_episodes(
                metadata_frame, min_train_samples=60, min_deploy_samples=12, max_episodes=10,
                master_seed=MASTER_SEED, n_id_controls=3))
            if episode_output is None: raise RuntimeError("Episode generation failed.")
            episodes, registry = episode_output
            record("accepted_and_rejected_registry", lambda: len(registry) >= len(episodes) and
                   {True, False}.issubset(set(registry["accepted"])))
            episode = episodes[0]
            lookup = {image_id: index for index, image_id in enumerate(frame["image_id"])}
            fit_idx = np.array([lookup[image_id] for image_id in episode.train_ids])
            deploy_idx = np.array([lookup[image_id] for image_id in episode.deployment_ids])
            fit_meta = metadata_frame.iloc[fit_idx].reset_index(drop=True)
            deploy_meta = metadata_frame.iloc[deploy_idx].reset_index(drop=True)
            record("episode_disjointness", lambda: set(episode.train_ids).isdisjoint(episode.deployment_ids))
            splits = {protocol: make_validation_splits(fit_meta, protocol, 3, 5, MASTER_SEED)
                      for protocol in PROTOCOLS}
            record("validation_split_generation", lambda: splits["random_kfold"]["available"] and
                   splits["date_group_kfold"]["available"] and splits["temporal_block_holdout"]["available"])
            protocol_results = {protocol: evaluate_validation_protocol(
                X[fit_idx], y[fit_idx], fit_meta, protocol, "fast", stable_seed("smoke", protocol), 3, 5)
                for protocol in PROTOCOLS}
            record("leakage_safe_protocol_evaluation", lambda: all(
                result["preprocessing_train_only"] for result in protocol_results.values()))
            shift = compute_shift_summary(fit_meta, deploy_meta, X[fit_idx], X[deploy_idx])
            evidence = EpisodeEvidence(episode.episode_id,
                build_initial_observation(fit_meta, deploy_meta, shift, protocol_results), shift, protocol_results)
            record("evaluator_label_absence", lambda: assert_agent_payload_safe(evidence.agent_visible_snapshot()))
            record("oracle_absence_before_reveal", lambda: "oracle" not in canonical_json(
                evidence.agent_visible_snapshot()).casefold())

            # Unavailable selection, no fallback, and abstention semantics.
            unavailable = validate_final_action({"action": "final", "selected_protocol": "state_group_kfold",
                "confidence": .4, "risk_level": "high", "reason_codes": ["insufficient_evidence"],
                "evidence_summary": "synthetic"}, ["random_kfold"])
            record("unavailable_protocol_invalid", lambda: unavailable["invalid_action"] and
                   unavailable["invalid_reason"] == "selected_protocol_unavailable")
            record("no_fallback_after_unavailable_selection", lambda: unavailable["selected_protocol"] == "state_group_kfold")
            abstain = validate_final_action({"action": "final", "selected_protocol": "ABSTAIN",
                "confidence": .1, "risk_level": "high", "reason_codes": ["insufficient_evidence"],
                "evidence_summary": "synthetic"}, ["random_kfold"])
            record("voluntary_abstain_rejected", lambda: abstain["decision_status"] == "invalid_decision" and
                   abstain["invalid_reason"] == "voluntary_abstain_not_allowed")

            # Specialist deployment score is created only after evidence and blind choices in this fixture.
            specialist = SpecialistBiomassRegressor("fast", MASTER_SEED)
            prediction = specialist.fit_predict(X[fit_idx], y[fit_idx], X[deploy_idx],
                                                fit_meta["image_id"], deploy_meta["image_id"])
            deployment_metrics, _ = calculate_regression_metrics(y[deploy_idx], prediction, "smoke")
            ground = EpisodeGroundTruth(episode.episode_id, deployment_metrics, prediction, prediction,
                                        y[deploy_idx], specialist.preprocessing_train_only_verified)
            oracle = record("oracle_computation_after_reveal", lambda: compute_oracle(
                protocol_results, deployment_metrics["Weighted_R2"]))
            heuristic = record("heuristic_selection", lambda: conservative_heuristic(evidence))

            mock_tool = MockAgentClient()
            tool_decision, trace = record("mock_agent_tool_loop", lambda: run_tool_agent(
                evidence, mock_tool, 0, trace_path=ARTIFACT_DIR / "smoke_trace.jsonl"))
            mock_llm = MockAgentClient()
            llm_decision = record("mock_llm_only_loop", lambda: run_llm_only_selector(evidence, mock_llm, 0))
            record("structured_trace_serialization", lambda: len(trace) >= 2 and all(
                json.loads(line)["episode_id"] == episode.episode_id
                for line in (ARTIFACT_DIR / "smoke_trace.jsonl").read_text(encoding="utf-8").splitlines()))
            record("mock_local_preflight", lambda: run_local_llm_preflight(use_mock=True)["validator_success"])
            record("missing_local_qwen_path_graceful_failure", lambda: raises(
                FileNotFoundError, lambda: validate_local_llm_path(Path(temporary) / "missing-qwen")))
            record("unsupported_transformers_version_graceful_failure", lambda: raises(
                RuntimeError, lambda: require_transformers_version("4.50.0")))

            # Strict parser accepts only raw/fenced JSON and rejects prose.
            raw_json = canonical_json({"action": "final", "selected_protocol": "random_kfold",
                "confidence": .5, "risk_level": "low", "reason_codes": [], "evidence_summary": "ok"})
            record("local_transformers_raw_json_parser", lambda: parse_json_action(raw_json)["action"] == "final")
            record("local_transformers_fenced_json_parser", lambda: parse_json_action(
                "```json\n" + raw_json + "\n```")["selected_protocol"] == "random_kfold")
            record("local_transformers_parser_rejects_prose", lambda: raises(
                ValueError, lambda: parse_json_action("Here is the answer: " + raw_json)))

            # Blind artifact freeze/hash and phase ordering.
            fixed = fixed_baseline_decisions(evidence)
            record("fixed_state_cv_present", lambda: any(item["method"] == "Fixed State CV" for item in fixed))
            base_decision = {**fixed[0], "episode_id": episode.episode_id, "analysis_scope": "primary",
                             "specialist_mode": "fast", "visible_evidence_hash": object_hash(
                                 evidence.agent_visible_snapshot()), "prompt_hash": None,
                             "local_model_identifier": "deterministic", "tool_trace_hash": object_hash([])}
            blind_record = blind_audit_record(episode, evidence, base_decision, True, "primary")
            bundle = record("blind_decision_freeze", lambda: freeze_blind_decisions([blind_record]))
            record("decision_hash_immutability", lambda: assert_blind_decisions_unchanged(bundle))
            audit = json.loads((ARTIFACT_DIR / "blind_decision_audit.json").read_text(encoding="utf-8"))
            record("blind_before_reveal_ordering", lambda: audit["deployment_labels_accessed"] is False and
                   audit["oracle_exists"] is False)

            # Ablation isolation from both initial evidence and tools.
            no_embedding = make_ablation_evidence(evidence, "no_embedding")
            metadata_only = make_ablation_evidence(evidence, "metadata_only")
            validation_only = make_ablation_evidence(evidence, "validation_only")
            record("no_embedding_evidence_removal", lambda: assert_ablation_evidence_isolation(
                no_embedding, "no_embedding") and "inspect_embedding_shift" not in ablation_allowed_tools("no_embedding"))
            record("metadata_only_evidence_removal", lambda: assert_ablation_evidence_isolation(
                metadata_only, "metadata_only") and ablation_allowed_tools("metadata_only") == METADATA_TOOL_NAMES)
            record("validation_only_evidence_removal", lambda: assert_ablation_evidence_isolation(
                validation_only, "validation_only") and not validation_only.shift_summary)

            # Cache identity changes with episode seed and includes complete safeguards.
            episode_seed_2 = DeploymentEpisode(episode.episode_id, episode.episode_type, episode.train_ids,
                                               episode.deployment_ids, episode.random_seed + 1, episode.metadata)
            config_1 = protocol_cache_configuration(episode, "random_kfold", "fast", fit_meta, X[fit_idx], y[fit_idx])
            config_2 = protocol_cache_configuration(episode_seed_2, "random_kfold", "fast", fit_meta, X[fit_idx], y[fit_idx])
            record("cache_seed_invalidation", lambda: object_hash(config_1) != object_hash(config_2))

            # Nested fold-local preprocessing audit without requiring tree packages.
            def nested_audit_check():
                trainer = StackingTrainer(seed=MASTER_SEED)
                trainer.get_base_models = lambda: {
                    name: [Ridge(alpha=1.0) for _ in TARGET_COLS] for name in ("lgbm", "xgb", "ridge")}
                small_X, small_y = X[fit_idx][:40], np.log1p(y[fit_idx][:40])
                ids = fit_meta["image_id"].astype(str).to_numpy()[:40]
                oof, audits = trainer.generate_inner_oof(
                    small_X, small_y, ids, list(KFold(2, shuffle=True, random_state=MASTER_SEED).split(small_X)))
                return all(row["scope"] == "inner" and row["forbidden_overlap"] == 0 for row in audits)
            record("nested_specialist_fold_local_leakage", nested_audit_check)

            # Result rows, calibration exclusion, category summary, and replicate-safe aggregation.
            llm_decision["method"] = "LLM Only"; tool_decision["method"] = "Tool-Using Local Agent"
            decisions = [*fixed, heuristic, llm_decision, tool_decision,
                {"method": "Oracle", "selected_protocol": oracle["oracle_protocol"], "replicate": 0,
                 "confidence": None, "tool_calls": 0, "decision_status": "valid_decision", "invalid_action": False}]
            rows = [decision_to_result_row(episode, decision, evidence, ground, oracle, "fast")
                    for decision in decisions]
            smoke_results = pd.DataFrame(rows)
            record("summary_calculation", lambda: summarize_episode_results(smoke_results)["mean_abs_gap"].notna().any())
            record("category_summary", lambda: len(summarize_by_method_and_episode_type(smoke_results)) > 0)
            replicated = pd.concat([smoke_results, smoke_results[smoke_results["method"] == "LLM Only"].assign(replicate=1),
                                    smoke_results[smoke_results["method"] == "LLM Only"].assign(replicate=2)])
            effective = effective_method_episode_votes(replicated)
            record("replicate_safe_effective_votes", lambda: len(effective[effective["method"] == "LLM Only"]) == 1)
            failure = system_failure_decision("Tool-Using Local Agent", "tool_agent", episode.episode_id, 9,
                                              MockAgentClient(), RuntimeError("synthetic"))
            failure_row = decision_to_result_row(episode, failure, evidence, ground, oracle, "fast")
            calibration_input = pd.concat([smoke_results, pd.DataFrame([failure_row])], ignore_index=True)
            _, calibration_summary = confidence_calibration(calibration_input)
            tool_cal = calibration_summary[calibration_summary["method"] == "Tool-Using Local Agent"].iloc[0]
            record("calibration_excludes_system_failures", lambda: tool_cal["valid_calibration_decisions"] == 1 and
                   tool_cal["model_generation_failure_rate"] > 0)
            record("generation_failure_separate_state", lambda: failure["decision_status"] == "system_failure" and
                   failure["selected_protocol"] is None)
            clean = sanitize_protocol_tool_result(protocol_results["random_kfold"])
            record("tool_result_sanitization", lambda: set(clean).issubset(PROTOCOL_RESULT_ALLOWLIST) and
                   not (set(clean) & FORBIDDEN_AGENT_KEYS))
            record("mock_model_unload_lifecycle", lambda: MockAgentClient().unload()["substantial_release"])
            record("offline_backend_default", lambda: LOCAL_LLM_BACKEND == "transformers")
            def complete_integrity_gate_check():
                cached_protocols = evaluate_protocols_for_episode(
                    episode, fit_meta, X[fit_idx], y[fit_idx], "fast")
                cached_evidence = EpisodeEvidence(episode.episode_id,
                    build_initial_observation(fit_meta, deploy_meta, shift, cached_protocols),
                    shift, cached_protocols)
                cached_ground = compute_deployment_ground_truth(
                    episode, fit_meta, deploy_meta, X[fit_idx], X[deploy_idx],
                    y[fit_idx], y[deploy_idx], "fast")
                cached_oracle = compute_oracle(cached_protocols, cached_ground.deployment_metrics["Weighted_R2"])
                smoke_decisions = [*fixed_baseline_decisions(cached_evidence),
                                   conservative_heuristic(cached_evidence),
                                   {**llm_decision, "method": "LLM Only"},
                                   {**tool_decision, "method": "Tool-Using Local Agent"}]
                audit_records, evaluated = [], []
                for decision in smoke_decisions:
                    selected_available = bool(cached_protocols.get(decision.get("selected_protocol"), {}).get("available", False))
                    audit_records.append(blind_audit_record(
                        episode, cached_evidence, decision, selected_available, "primary"))
                    evaluated.append(decision_to_result_row(
                        episode, decision, cached_evidence, cached_ground, cached_oracle, "fast", "primary"))
                oracle_decision = {"method": "Oracle", "selected_protocol": cached_oracle["oracle_protocol"],
                    "replicate": 0, "confidence": None, "tool_calls": 0,
                    "decision_status": "valid_decision", "invalid_action": False}
                evaluated.append(decision_to_result_row(
                    episode, oracle_decision, cached_evidence, cached_ground, cached_oracle, "fast", "primary"))
                full_bundle = freeze_blind_decisions(audit_records)
                reveal_time = datetime.now(timezone.utc).isoformat()
                phase = {"blind_frozen_before_reveal": True, "blind_frozen_at": full_bundle.frozen_at,
                         "deployment_reveal_started_at": reveal_time, "oracle_created_after_reveal": True,
                         "oracle_created_at": datetime.now(timezone.utc).isoformat()}
                llm_items = [decision for decision in smoke_decisions
                             if decision.get("agent_type") in {"llm_only", "tool_agent"}]
                failure_table = llm_systemic_failure_report(llm_items)
                protocol_frame = pd.DataFrame(protocol_table_rows(cached_evidence, "primary", "fast"))
                integrity = run_integrity_checks(
                    [episode], [cached_evidence], {"primary": cached_ground}, protocol_frame,
                    pd.DataFrame(evaluated), llm_items, full_bundle, phase, failure_table, True,
                    {"dino_unloaded_before_preflight": True, "dino_unloaded_before_primary_qwen": True,
                     "primary_qwen_unloaded_before_reveal": True, "local_files_only": True,
                     "external_api_calls": 0},
                    [(no_embedding, "no_embedding"), (metadata_only, "metadata_only"),
                     (validation_only, "validation_only")])
                return integrity["ALL_INTEGRITY_CHECKS_PASSED"]
            record("complete_25_check_integrity_gate", complete_integrity_gate_check)
        finally:
            ARTIFACT_DIR = original_artifact_dir
    report = pd.DataFrame(reports)
    report.attrs["ALL_SMOKE_TESTS_PASSED"] = bool(report["passed"].all())
    if raise_on_failure and not report.attrs["ALL_SMOKE_TESTS_PASSED"]:
        failures = report.loc[~report["passed"], ["check", "detail"]].to_dict(orient="records")
        raise AssertionError(f"Smoke tests failed: {failures}")
    return report

## 16. Results

No historic result is stored in the supplied notebook. A completed final run writes episode-level results, method and shift-category summaries, paired common-support statistics, valid-decision calibration, the frozen blind audit, optional robustness results, and paper figures beneath `outputs/` by default.

Evaluator categories (`in_distribution`, `temporal`, `spatial`, and `spatiotemporal`) are assigned only after blind decisions and never appear in Qwen prompts or tool outputs.


In [ ]:
# ---------------------------------------------------------
# SMOKE TEST (synthetic data; no DINO extraction; no external LLM)
# ---------------------------------------------------------
smoke_report = run_smoke_test()
display(smoke_report)
print("ALL_SMOKE_TESTS_PASSED =", smoke_report.attrs["ALL_SMOKE_TESTS_PASSED"])

## 17. Full experiment workflow

1. Place the CSIRO data under `data/`, or set `CSIRO_DATA_PATH`.
2. Set `DINO_MODEL_PATH`, or set `FEATURE_CACHE_READ_DIR` to a compatible cache.
3. Set `LOCAL_LLM_PATH` to the local Qwen3-4B checkpoint.
4. Run `run_local_llm_preflight()` and inspect the generated report.
5. Run `run_smoke_test()`.
6. Confirm `RUN_MODE = "final"` and `EXECUTE_LLM_CALLS = True`.
7. Execute the final cell to call `run_final_experiment()`.

The full experiment remains offline and does not install packages. Qwen3 requires `transformers>=4.51.0`.


### Final experiment launch

The following cell is intentionally active to preserve the source notebook's execution sequence. It requires the external data and local model paths configured above and may be computationally expensive.


In [ ]:
final_results = run_final_experiment()